# nb_FactClaimPayment
## Carga de la tabla Gold.bgla_FactClaimPayment
**PatrÃƒÆ’Ã‚Â³n de carga:** Fact (Bronze ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Silver staging ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ Gold)

**Fuente:** Tablas Bronze de AmigosPlus (Claim_Run, Claim_Detail, Claim_LineDetail, etc.)

**DescripciÃƒÆ’Ã‚Â³n:** MigraciÃƒÆ’Ã‚Â³n del SP `usp_DataPrepare_FactClaimCostData` a Fabric Notebook.
Genera registros de pagos de claims desde mÃƒÆ’Ã‚Âºltiples fuentes:
- Run Member (pagos a miembros)
- Run Provider (pagos a proveedores)
- PPA Provider / PPA Member (ajustes PPA)
- Void PPA Provider / Void PPA Member (anulaciones PPA)

**Convenciones BGLA:**
- Staging: `Silver.TmpFactClaimPayment`
- Target: `Gold.bgla_FactClaimPayment`
- Defaults: Strings ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ `'_Undefined'`, Numbers ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ `0` / `-101`, Dates ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ `'1900-01-01'`

In [ ]:
# ============================================================
# SPARK PERFORMANCE CONFIGURATION
# Buenas prÃƒÆ’Ã‚Â¡cticas Fabric: AQE, Optimize Write, V-Order
# ============================================================

# ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Adaptive Query Execution (AQE) ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Delta Write Optimization ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
spark.conf.set("spark.microsoft.delta.parquet.vorder.enabled", "true")

# ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Shuffle Partitions (ajustar segÃƒÆ’Ã‚Âºn volumen de datos) ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
spark.conf.set("spark.sql.shuffle.partitions", "200")

# ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Broadcast threshold (tablas < 100MB se hacen broadcast automÃƒÆ’Ã‚Â¡ticamente) ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")  # 100MB

# ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Schema auto-merge para Delta ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print("[OK] Spark performance configuration applied")

In [ ]:
%run nb_FactClaimPayableMulticurrency

In [ ]:
%run nb_AMPview_Claim_LineDetailPayExchangeRates

In [ ]:
# ============================================================
# PARÃƒÂMETROS DE CARGA INCREMENTAL
# Equivale a @FromDate, @ToDate, @Min, @Max del SP
# ============================================================
from datetime import datetime, timedelta

# Ã¢â€â‚¬Ã¢â€â‚¬ Modo de carga Ã¢â€â‚¬Ã¢â€â‚¬
# True  = Full load del rango definido abajo (procesa TODOS los RunIds del periodo)
# False = Incremental (ÃƒÂºltimos N dÃƒÂ­as)
FULL_LOAD = False

# Ã¢â€â‚¬Ã¢â€â‚¬ Rango de fechas Ã¢â€â‚¬Ã¢â€â‚¬
# IMPORTANTE: Las vistas PPA (vw_ProviderPPA, vw_ProviderPPA_Void, vw_MemberPPA,
# vw_MemberPPA_Void) tienen el filtro `journal.PostedOn BETWEEN FROM_DATE_STR AND
# TO_DATE_STR` HARDCODED. El rango siempre se aplica a esos branches PPA.
#
# AJUSTA AQUÃƒÂ el rango que quieres procesar.
# Ejemplos:
#   - ÃƒÅ¡ltimos 60 dÃƒÂ­as        Ã¢â€ â€™ INCREMENTAL_DAYS = 60   ; FULL_LOAD = False
#   - Marzo-Abril 2026       Ã¢â€ â€™ from_date / to_date manual abajo ; FULL_LOAD = True
#   - AÃƒÂ±o en curso completo  Ã¢â€ â€™ from_date = datetime(_now.year,1,1) ; FULL_LOAD = True
INCREMENTAL_DAYS = 60

_now = datetime.now()
if FULL_LOAD:
    # Ã¢â€â‚¬Ã¢â€â‚¬ Rango fijo (manual) Ã¢â‚¬â€ solo se usa cuando FULL_LOAD = True Ã¢â€â‚¬Ã¢â€â‚¬
    from_date = datetime(2026, 3, 1, 0, 0, 0)        # Ã¢â€ Â edita inicio
    to_date   = datetime(2026, 4, 30, 23, 59, 59)    # Ã¢â€ Â edita fin
else:
    # Ã¢â€â‚¬Ã¢â€â‚¬ Incremental: ÃƒÂºltimos INCREMENTAL_DAYS Ã¢â€â‚¬Ã¢â€â‚¬
    to_date   = _now
    from_date = to_date - timedelta(days=INCREMENTAL_DAYS)

FROM_DATE_STR = from_date.strftime("%Y-%m-%d %H:%M:%S")
TO_DATE_STR   = to_date.strftime("%Y-%m-%d %H:%M:%S")

# Ã¢â€â‚¬Ã¢â€â‚¬ Outlier config Ã¢â€â‚¬Ã¢â€â‚¬
# TODO: Migrar ETLFramework.Metadata.TConfiguracion a Bronze/Silver
# Por ahora se parametriza. Actualizar cuando la tabla estÃƒÂ© disponible.
MIN_OUTLIER = 0
MAX_OUTLIER = 99999999

try:
    df_config = spark.sql("""
        SELECT Propiedad, CAST(Valor AS INT) AS Valor
        FROM Bronze.ETLFramework_Metadata_TConfiguracion
        WHERE FiltroConfiguracion = 'Claim Outlier'
          AND Propiedad IN ('Min Outlier Billed Amount', 'Max Outlier Billed Amount')
    """)
    config_map = {row["Propiedad"]: row["Valor"] for row in df_config.collect()}
    MIN_OUTLIER = config_map.get("Min Outlier Billed Amount", MIN_OUTLIER)
    MAX_OUTLIER = config_map.get("Max Outlier Billed Amount", MAX_OUTLIER)
    print(f"[OK] Outlier config from Bronze: Min={MIN_OUTLIER}, Max={MAX_OUTLIER}")
except Exception:
    print(f"[WARN] TConfiguracion not available, using defaults: Min={MIN_OUTLIER}, Max={MAX_OUTLIER}")

print(f"[OK] Load mode: {'FULL (manual range)' if FULL_LOAD else f'INCREMENTAL (last {INCREMENTAL_DAYS} days)'}")
print(f"[OK] Date range: {FROM_DATE_STR} Ã¢â€ â€™ {TO_DATE_STR}")


In [ ]:
# Cell 1: Drop staging table
spark.sql("DROP TABLE IF EXISTS Silver.TmpFactClaimPayment")
print("[OK] Silver.TmpFactClaimPayment dropped")

In [ ]:
# ============================================================
# RunBase: Identifica los RunIds a procesar
# - FULL_LOAD = True  ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ todos los RunIds (sin filtro de fecha)
# - FULL_LOAD = False ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ solo runs en el rango [FROM_DATE_STR, TO_DATE_STR] (#RunBase del SP)
# ============================================================
if FULL_LOAD:
    df_run_base = spark.sql("""
        SELECT DISTINCT run.RunId
        FROM Bronze.AmigosPlus_AMP_Claim_Run run
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun detailrun ON detailrun.RunId = run.RunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = detailrun.ClaimDetailId
    """)
else:
    df_run_base = spark.sql(f"""
        SELECT DISTINCT run.RunId
        FROM Bronze.AmigosPlus_AMP_Claim_Run run
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun detailrun ON detailrun.RunId = run.RunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = detailrun.ClaimDetailId
        LEFT JOIN Bronze.AmigosPlus_AMP_Common_Payment cmPayment ON cmPayment.PaymentId = run.PaymentId
        LEFT JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog
            ON paymentlog.PaymentId = run.PaymentId
            AND paymentlog.PaymentLogDate BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}'
        LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry journal
            ON paymentlog.JournalEntryId = journal.JournalEntryId
            AND journal.PostedOn BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}'
        WHERE
            (journal.PostedOn BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}')
            OR (run.CreatedOn BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}')
            OR (run.PrintedOn BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}')
            OR (cmPayment.PostedDate BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}')
            OR (cmPayment.PrintedDate BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}')
            OR (paymentlog.PaymentLogDate BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}')
    """)

df_run_base.cache()
df_run_base.createOrReplaceTempView("vw_RunBase")
run_count = df_run_base.count()
print(f"[OK] vw_RunBase created ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â {run_count:,} RunIds to process (FULL_LOAD={FULL_LOAD})")

# ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Pre-marcar ActualRowFlag = 0 para los runs que se van a recargar ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
if run_count > 0:
    rows_updated = spark.sql("""
        MERGE INTO Gold.bgla_FactClaimPayment F
        USING (SELECT DISTINCT RunId AS ClaimRunId FROM vw_RunBase) RB
        ON F.ClaimRunId = RB.ClaimRunId
        WHEN MATCHED THEN UPDATE SET F.ActualRowFlag = 0
    """)
    print("[OK] ActualRowFlag = 0 set for existing runs in scope")
else:
    print("[WARN] No runs in date range ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â full reload may be needed")


In [ ]:
# Cell 2: Vista temporal - ClaimLog (ÃƒÆ’Ã‚Âºltimo log activo por ClaimDetail)
df_claim_log = spark.sql("""
    SELECT cl.ClaimDetailId, cl.StatusId, cl.ReasonId, cl.FromDate,
           ROW_NUMBER() OVER (PARTITION BY cl.ClaimDetailId ORDER BY cl.LogId DESC) AS rn
    FROM Bronze.AmigosPlus_AMP_Claim_Log cl
    WHERE cl.ToDate IS NULL
""")
df_claim_log.createOrReplaceTempView("vw_ClaimLog_Payment")
print("[OK] vw_ClaimLog_Payment created")

In [ ]:
# Cell 3: Vista temporal - ProcessedDate (claims procesados con razones especÃƒÆ’Ã‚Â­ficas)
df_processed_date = spark.sql("""
    SELECT l.ClaimDetailId, l.StatusId, l.ReasonId, l.FromDate,
           ROW_NUMBER() OVER (PARTITION BY l.ClaimDetailId ORDER BY l.LogId DESC) AS Cont
    FROM Bronze.AmigosPlus_AMP_Claim_Log l
    WHERE l.StatusId = 1017
      AND l.ReasonId IN (1094, 1095, 1096, 1773, 1827, 1828, 1829, 1933)
""")
df_processed_date.createOrReplaceTempView("vw_ProcessedDate_Payment")
print("[OK] vw_ProcessedDate_Payment created")

In [ ]:
# Cell 4: Vista temporal - DiagnosticLine (diagnÃƒÆ’Ã‚Â³stico primario por lÃƒÆ’Ã‚Â­nea)
df_diagnostic_line = spark.sql("""
    SELECT diag.DiagnosticId, diag.ClaimLineDetailId,
           ROW_NUMBER() OVER (PARTITION BY diag.ClaimLineDetailId ORDER BY diag.DiagnosticLineId DESC) AS ID
    FROM Bronze.AmigosPlus_AMP_Claim_DiagnosticbyLine diag
    WHERE diag.`Primary` = 1
""")
df_diagnostic_line.createOrReplaceTempView("vw_DiagnosticLine_Payment")
print("[OK] vw_DiagnosticLine_Payment created")

In [ ]:
# Cell 5: Vista temporal - ProcedureLine (procedimiento primario por lÃƒÆ’Ã‚Â­nea)
df_procedure_line = spark.sql("""
    SELECT procl.ProcedureId, procl.ClaimLineDetailId,
           ROW_NUMBER() OVER (PARTITION BY procl.ClaimLineDetailId ORDER BY procl.ProcedureByLineId DESC) AS ID
    FROM Bronze.AmigosPlus_AMP_Claim_ProcedurebyLine procl
    WHERE procl.`Primary` = 1
""")
df_procedure_line.createOrReplaceTempView("vw_ProcedureLine_Payment")
print("[OK] vw_ProcedureLine_Payment created")

In [ ]:
# Cell 6: Vista temporal - ClaimSubmission (ÃƒÆ’Ã‚Âºltimo envÃƒÆ’Ã‚Â­o digital por claim)
df_claim_submission = spark.sql("""
    SELECT DCS.ClaimHeaderId, DCS.PolicyId, DCS.SenderTypeId, DCS.InboundChannel,
           DCS.TrackingNumber, DCS.Updatedon, DCS.ReceivedDate, DCS.IsComplement,
           DCS.SenderEmailAddress, DCS.SenderFullName,
           ROW_NUMBER() OVER (PARTITION BY DCS.ClaimHeaderId ORDER BY DCS.updatedon DESC) AS CONT
    FROM Bronze.Claim_ClaimSubmission DCS
    WHERE DCS.ClaimHeaderId <> 0
""")
df_claim_submission.createOrReplaceTempView("vw_ClaimSubmission_Payment")
print("[OK] vw_ClaimSubmission_Payment created")

In [ ]:
# Cell 7: Vista temporal - TmpReceived (primera fecha de recepciÃƒÆ’Ã‚Â³n por header)
df_tmp_received = spark.sql("""
    SELECT cd.HeaderId,
           MIN(cd.ReceivedDate) AS ReceivedDate,
           MIN(cld.FromDate) AS ServiceFromDate
    FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
    INNER JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail cld ON cd.ClaimDetailId = cld.ClaimDetailId
    GROUP BY cd.HeaderId
""")
df_tmp_received.createOrReplaceTempView("vw_TmpReceived_Payment")
print("[OK] vw_TmpReceived_Payment created")

In [ ]:
# Cell 8: Vista temporal - PolicyBaseCurrency (moneda base por pÃƒÆ’Ã‚Â³liza)
df_policy_base_currency = spark.sql("""
    SELECT DISTINCT me.PolicyId, me.MemberId, me.MemberEligibilityId, me.FromDate, me.ToDate,
           cr.CurrencyId AS BaseCurrencyId
    FROM Bronze.AmigosPlus_AMP_Policy_MemberEligibility me
    INNER JOIN Bronze.AmigosPlus_AMP_Common_DeductibleRegionPlanXRef drpx ON me.DeductibleRegionPlanXRefId = drpx.DeductibleRegionPlanXRefId
    INNER JOIN Bronze.AmigosPlus_AMP_Common_Plan pl ON drpx.PlanId = pl.PlanId
    INNER JOIN Bronze.AmigosPlus_AMP_Common_Product pr ON pl.ProductId = pr.ProductId
    LEFT JOIN Bronze.AmigosPlus_AMP_Generic_CurrencyRegionProductXRef crpx ON pr.ProductId = crpx.ProductId AND drpx.RegionId = crpx.RegionId
    INNER JOIN Bronze.AmigosPlus_AMP_Generic_Currency cr ON COALESCE(crpx.CurrencyId, pr.BaseCurrencyId) = cr.CurrencyId
    WHERE me.RelationTypeId IN (2, 5) AND me.StatusId <> 30 AND me.FromDate <= COALESCE(me.ToDate, me.FromDate)
""")
df_policy_base_currency.createOrReplaceTempView("vw_PolicyBaseCurrency_Payment")
print("[OK] vw_PolicyBaseCurrency_Payment created")

In [ ]:
# Cell 9: Vista temporal - Payee (payee preferido por contacto)
df_payee = spark.sql("""
    SELECT PayeeId, ContactBaseId,
           ROW_NUMBER() OVER (PARTITION BY ContactBaseId ORDER BY PayeeId DESC) AS RowNum
    FROM Bronze.AmigosPlus_AMP_Common_Payee
    WHERE ToDate IS NULL AND (ClaimPreferred = 1 OR ClaimPreferredOnShore = 1)
""")
df_payee.createOrReplaceTempView("vw_Payee_Payment")
print("[OK] vw_Payee_Payment created")

In [ ]:
# Cell 10: Vista temporal - ReceivedMethodDigital
df_received_method_digital = spark.sql("""
    SELECT ReceivedMethodId, ReceivedMethodName
FROM (
    SELECT rm.ReceivedMethodId, rm.Name AS ReceivedMethodName
    FROM Bronze.AmigosPlus_AMP_Common_ReceivedMethod rm
    UNION
    SELECT cc.CodeId AS ReceivedMethodId, cc.Name AS ReceivedMethodName
    FROM Bronze.AmigosPlus_AMP_Common_Code cc
    WHERE cc.CodeTypeId = 3109
    UNION
    SELECT -102 AS ReceivedMethodId, 'Web' AS ReceivedMethodName
) allMethods
WHERE ReceivedMethodName IN ('Web-Single', 'Web-Multiple', 'Web')
""")
df_received_method_digital.createOrReplaceTempView("vw_ReceivedMethodDigital_Payment")
print("[OK] vw_ReceivedMethodDigital_Payment created")

In [ ]:
# ============================================================
# Cell 11: Vista temporal - ClaimRun base (runs con sus payment logs y journal entries)
# OPTIMIZACIÃƒÆ’Ã¢â‚¬Å“N: spark.sql() + cache() ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â vista reutilizada en 6+ celdas
# ============================================================
df_claim_run = spark.sql("""
    SELECT DISTINCT
        run.RunId,
        run.RunTypeId,
        detail.HeaderId,
        detailrun.DetailRunId,
        detailrun.ClaimDetailId,
        detail.PolicyId,
        detail.MemberEligibilityId,
        detail.MemberId,
        journal.JournalEntryId,
        journal.PostedOn AS postedon,
        run.InsuranceBusinessId,
        run.IsProvider,
        run.PaymentId,
        COALESCE(paymentlog.StatusId, cmPayment.StatusId) AS statusid,
        run.PayCurrencyID,
        run.CreatedOn,
        run.PrintedOn,
        run.Status,
        run.PaymentMethodId,
        run.Processed,
        CASE WHEN paymentlog.StatusId IN (153,156,174,264,265,275) THEN -1 ELSE 1 END AS Factor,
        journal.Notes
    FROM Bronze.AmigosPlus_AMP_Claim_Run run
    INNER JOIN vw_RunBase rb ON rb.RunId = run.RunId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun detailrun ON detailrun.RunId = run.RunId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = detailrun.ClaimDetailId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_Payment cmPayment ON cmPayment.PaymentId = run.PaymentId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog ON paymentlog.PaymentId = run.PaymentId
    LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry journal ON paymentlog.JournalEntryId = journal.JournalEntryId
""")
df_claim_run.cache()
df_claim_run.createOrReplaceTempView("vw_ClaimRun")
print(f"[OK] vw_ClaimRun cached ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â {df_claim_run.count():,} rows")

In [ ]:
# Cell 12: Vista temporal - PaymentLogVoid (logs de void con razÃƒÆ’Ã‚Â³n)
df_payment_log_void = spark.sql("""
    SELECT pl.PaymentId, pl.JournalEntryId, pl.StatusId, pl.ReasonId,
           MAX(pl.PaymentLogDate) AS PaymentLogDate
    FROM Bronze.AmigosPlus_AMP_Common_PaymentLog pl
    INNER JOIN Bronze.AmigosPlus_AMP_Common_Status cs ON pl.StatusId = cs.StatusId
    WHERE cs.Description LIKE '%Void%'
    GROUP BY pl.PaymentId, pl.JournalEntryId, pl.ReasonId, pl.StatusId
""")
df_payment_log_void.createOrReplaceTempView("vw_PaymentLogVoid")
print("[OK] vw_PaymentLogVoid created")

In [ ]:
# Cell 13: Vista temporal - InsuranceBusinessRun (agrupaciÃƒÆ’Ã‚Â³n por header)
df_insurance_business_run = spark.sql("""
    SELECT HeaderId,
           MAX(InsuranceBusinessId) AS Max_InsuranceBusinessId,
           MAX(COALESCE(CAST(IsProvider AS INT), 0)) AS max_isprovider,
           MIN(COALESCE(CAST(IsProvider AS INT), 0)) AS min_isprovider
    FROM vw_ClaimRun
    GROUP BY HeaderId
""")
df_insurance_business_run.createOrReplaceTempView("vw_InsuranceBusinessRun")
print("[OK] vw_InsuranceBusinessRun created")

In [ ]:
# ============================================================
# Cell 14: Vista temporal - SubLineDetailValue (ineligibles, withholding, VAT por lÃƒÆ’Ã‚Â­nea)
# OPTIMIZACIÃƒÆ’Ã¢â‚¬Å“N: spark.sql() + cache() ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â alimenta vw_BaseClaimsLines
# ============================================================
df_subline_detail = spark.sql("""
    SELECT cld.ClaimLineDetailId, cld.ClaimDetailId, cd.HeaderId,
        SUM(CASE WHEN ic.InelCodeId IS NOT NULL THEN sl.IneligibleAmount ELSE 0 END) AS BaseHB_DiscountAmount,
        SUM(CASE WHEN ic.InelCodeId IS NULL THEN sl.IneligibleAmount ELSE 0 END) AS BaseOtherIneligibleAmount,
        SUM(sl.IneligibleAmount) AS BaseTotalIneligibleAmount,
        SUM(sl.LocalIneligibleAmount) AS LocalTotalIneligibleAmount,
        SUM(CASE WHEN cld.ServiceIndicatorId = 230 THEN cld.ProviderLiabilityAmount + (CASE WHEN sl.InelCodeId = 15114 THEN sl.IneligibleAmount ELSE 0 END) ELSE 0 END) AS BaseWithHolding,
        SUM(CASE WHEN cld.ServiceIndicatorId = 203 THEN cld.NetCoveredAmount ELSE 0 END) AS BaseVAT
    FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Log cl ON cl.ClaimDetailId = cd.ClaimDetailId AND cl.ToDate IS NULL
    LEFT JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail cld ON cld.ClaimDetailId = cd.ClaimDetailId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_SubLineDetail sl ON cld.ClaimLineDetailId = sl.ClaimLineDetailId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_IneligibleCode ic ON sl.InelCodeId = ic.InelCodeId
        AND ic.InelCodeId IN (139, 143, 145, 148, 165, 308, 315, 368, 399, 505, 568, 569, 579, 601, 604, 605, 15004, 15062, 15099, 159, 15113, 15114)
    WHERE cd.HeaderId NOT IN (1107843, 2140356)
        AND cd.ServiceIndicatorId NOT IN (432)
        AND cl.ReasonID NOT IN (1094, 1555, 1577, 1721, 1858, 1859)
    GROUP BY cld.ClaimLineDetailId, cld.ClaimDetailId, cd.HeaderId
""")
df_subline_detail.cache()
df_subline_detail.createOrReplaceTempView("vw_SubLineDetailValue")
print(f"[OK] vw_SubLineDetailValue cached ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â {df_subline_detail.count():,} rows")

In [ ]:
# Cell 15: Vista temporal - NetworkMetadataXref (service indicators de red)
df_network_metadata = spark.sql("""
    SELECT DISTINCT ServiceIndicatorId
    FROM Bronze.AmigosPlus_AMP_Claim_NetworkMetadataXref
""")
df_network_metadata.createOrReplaceTempView("vw_NetworkMetadataXref")
print("[OK] vw_NetworkMetadataXref created")

In [ ]:
# Cell 15b: Vista temporal - PolicyHistory (dimensiones de pÃƒÆ’Ã‚Â³liza para FactClaimPayment)
# Fuente: Silver.tmp_policyamigosplushistory (migrado de __TMPvwTMPBGLAEDW___PolicyAmigosPlusHistory)
df_policy_history = spark.sql("""
    SELECT DISTINCT
        pp.PolicySKey_PolicyId AS PolicyId,
        COALESCE(pp.ModeOfPaymentSKey_ModeOfPaymentId, -101) AS ModeOfPaymentId,
        COALESCE(pp.PaymentMethodSKey_PaymentMethodId, -1) AS PaymentMethodId,
        COALESCE(pp.ReceivedMethodSKey_ReceivedMethodId, -101) AS ReceivedMethodId,
        COALESCE(pp.GroupSKey_GroupId, -101) AS GroupId,
        COALESCE(pp.FamilyTypeSKey_FamilyTypeId, -101) AS FamilyTypeId,
        COALESCE(pp.ProductSKey_ProductId, -101) AS ProductId,
        COALESCE(pp.EntitySKey_EntityId, -101) AS EntityId,
        COALESCE(pp.PlanSKey_PlanId, -101) AS PlanId,
        COALESCE(pp.BusinessModeSKey_BusinessModeId, -101) AS BusinessModeId,
        COALESCE(pp.BusinessTypeSKey_BusinessTypeId, -101) AS BusinessTypeId,
        COALESCE(pp.InsuranceBusinessSKey_InsuranceBusinessId, -101) AS InsuranceBusinessId,
        COALESCE(pp.CompanySKey_CompanyId, -101) AS CompanyId,
        pp.PolicyIssueDateKey_Date AS PolicyIssueDate,
        pp.RenewalDateKey_Date AS PolicyRenewalDate,
        pp.AnniversaryDateKey_Date AS PolicyAnniversaryDate,
        pp.ApplicationReceivedDateKey_Date AS PolicyApplicationReceivedDate,
        pp.PolicyEffectiveDateKey_Date AS PolicyEffectiveDate,
        pp.PolicyCancelDateKey_Date AS PolicyCancelDate,
        pp.DeathBenefitperiodDateKey_Date AS PolicyDeathBenefitPeriodDate,
        COALESCE(pp.AgentSKey_AgentHierarchyId, -101) AS AgentHierarchyId,
        COALESCE(pp.RegionSkey_RegionId, -101) AS RegionId,
        COALESCE(pp.MemberOwnerSKey_PolicyMemberEligibilityId, -101) AS MemberOwnerId,
        pp.GeographySKey_GlobalLocationId AS PolicyGlobalLocationId,
        pp.GeographySKey_CountryId AS PolicyCountryId,
        pp.GeographySKey_StateId AS PolicyStateId,
        pp.GeographySKey_CityId AS PolicyCityId
    FROM Silver.tmp_policyamigosplushistory pp
""")
df_policy_history.createOrReplaceTempView("vw_PolicyHistory")
print("[OK] vw_PolicyHistory created from Silver.tmp_policyamigosplushistory")

In [ ]:
# Cell 15c: Vista temporal - ThirdPartyClaim (terceros asociados al claim)
# TODO: Verificar que la tabla estÃƒÆ’Ã‚Â© ingestada en Bronze
df_third_party_claim = spark.sql("""
    SELECT tp.ClaimDetailId, tp.ThirdPartyId, tp.PayTo
    FROM Bronze.AmigosPlus_AMP_Claim_ThirdPartyClaim tp
    WHERE tp.PayTo IN (1007, 1008)
""")
df_third_party_claim.createOrReplaceTempView("vw_ThirdPartyClaim")
print("[OK] vw_ThirdPartyClaim created")

In [ ]:
# Cell 15d: Vista temporal - LocalTeamTAT (IsCOR y LocalTeam desde FactClaimTAT)
# Equivale a #MaxIteratiopn + #LocalTeamTAT del SP
# Usa Max(Iterations) por ClaimHeaderId, luego trae LocalTeam e IsCOR
try:
    df_local_team_tat = spark.sql("""
        SELECT DISTINCT tat.LocalTeam, tat.ClaimHeaderId, tat.IsCOR
        FROM Gold.bgla_FactClaimTAT tat
        INNER JOIN (
            SELECT ClaimHeaderId, MAX(Iterations) AS MaxIteration
            FROM Gold.bgla_FactClaimTAT
            WHERE ClaimHeaderId IN (SELECT DISTINCT HeaderId FROM vw_ClaimRun)
            GROUP BY ClaimHeaderId
        ) mi ON mi.ClaimHeaderId = tat.ClaimHeaderId AND mi.MaxIteration = tat.Iterations
        WHERE tat.ClaimHeaderId IN (SELECT DISTINCT HeaderId FROM vw_ClaimRun)
    """)
    df_local_team_tat.createOrReplaceTempView("vw_LocalTeamTAT")
    print(f"[OK] vw_LocalTeamTAT created from Gold.bgla_FactClaimTAT ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â {df_local_team_tat.count():,} rows")
except Exception as e:
    # Fallback: stub vacÃƒÆ’Ã‚Â­o si FactClaimTAT no estÃƒÆ’Ã‚Â¡ disponible aÃƒÆ’Ã‚Âºn
    df_local_team_tat = spark.sql("""
        SELECT
            CAST(-1 AS BIGINT) AS ClaimHeaderId,
            CAST('_Non Defined IsCOR' AS STRING) AS IsCOR,
            CAST('_Non Defined LocalTeam' AS STRING) AS LocalTeam
        WHERE 1 = 0
    """)
    df_local_team_tat.createOrReplaceTempView("vw_LocalTeamTAT")
    print(f"[WARN] Gold.bgla_FactClaimTAT not available ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â using empty stub: {e}")

In [ ]:
# Cell 15e: Vista temporal - ClearingHouseLine (datos de ClearingHouse por lÃƒÆ’Ã‚Â­nea)
# Fuente: Bronze.Clearinghouse_Invoice_* (migrado de ETLStaging.Clearinghouse.Invoice_*)
# LÃƒÆ’Ã‚Â³gica: Claims cuyo LotNumber coincide con un BatchLotName existente en Invoice_BatchLot
df_clearing_house_line = spark.sql("""
    SELECT
        cd.ClaimDetailId,
        cld.ClaimLineDetailId,
        batlot.BatchLotName,
        CAST(batlot.FromDate AS DATE) AS BatchLotFromDate,
        CAST(tralot.ReceivedDate AS DATE) AS TransactionLotReceivedDate,
        tralot.Number AS TransactionLotNumber,
        tra.InvoiceNumber AS TransactionInvoiceNumber,
        cd.LotNumber,
        CASE WHEN tra.ClaimHeaderId IS NULL THEN 0 ELSE 1 END AS IsAutomatic
    FROM (
        SELECT HeaderId, ClaimDetailId, LotNumber
        FROM Bronze.AmigosPlus_AMP_Claim_Detail
        WHERE LotNumber IN (SELECT DISTINCT BatchLotName FROM Bronze.Clearinghouse_Invoice_BatchLot)
    ) cd
    INNER JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail cld ON cld.ClaimDetailId = cd.ClaimDetailId
    LEFT JOIN Bronze.Clearinghouse_Invoice_Transaction tra ON cd.HeaderId = tra.ClaimHeaderId
    LEFT JOIN Bronze.Clearinghouse_Invoice_TransactionLine traline
        ON traline.TransactionGUI = tra.TransactionGUI AND traline.ClaimLineDetailId = cld.ClaimLineDetailId
    LEFT JOIN Bronze.Clearinghouse_Invoice_TransactionLot tralot ON tralot.TransactionLotGUI = tra.TransactionLotGUI
    LEFT JOIN Bronze.Clearinghouse_Invoice_BatchLot batlot ON batlot.BatchLotGUI = tralot.BatchLotGUI
""")
df_clearing_house_line.createOrReplaceTempView("vw_ClearingHouseLine")
print(f"[OK] vw_ClearingHouseLine created ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â {df_clearing_house_line.count():,} rows")

In [ ]:
# ============================================================
# Cell 16: Vista temporal - BaseClaimsLines (valores base y pay por linea de claim)
# OPTIMIZACION: spark.sql() + cache() -- vista reutilizada en 3+ celdas
# FIX 2026-05-08: deduplicar vw_ClaimRun por ClaimDetailId antes del JOIN.
#   vw_ClaimRun tiene una fila por RunId/Factor (PAID + VOID + re-PAID -> N filas
#   para un mismo ClaimDetailId). Hacer JOIN directo multiplicaba cada linea x N runs;
#   ROW_NUMBER() asignaba LineNumber=1..N y DISTINCT no colapsaba (LineNumber difiere),
#   produciendo duplicados que rompian vw_MemberPPAAdjustment / vw_ProviderPPAAdjustment.
#   Solucion: subquery DISTINCT con solo claim-level columns (constantes por ClaimDetailId).
# ============================================================
df_base_claims = spark.sql("""
    SELECT DISTINCT
        TCR.HeaderId,
        TCR.PolicyId,
        TCR.MemberEligibilityId,
        TCR.MemberId,
        ld.ReferralHeaderId,
        ld.POSId,
        ld.TOSId,
        ld.ServiceProviderId,
        ld.ClaimDetailId,
        ld.ClaimLineDetailId,
        ld.ServiceIndicatorId,
        cd.ServiceIndicatorId AS DetailServiceIndicatorId,
        CAST(ld.FromDate AS DATE) AS FromDate,
        CAST(ld.ToDate AS DATE) AS ToDate,
        ld.XchangeDate,
        ld.XChangeRate,
        ld.CurrencyId AS LocalCurrencyId,
        COALESCE(pbc.BaseCurrencyId, -101) AS BaseCurrencyId,
        ld.XChangeRate AS BaseXChangeRate,
        COALESCE(ld.PayCurrencyIdToMember, -101) AS PayToMemberCurrencyId,
        COALESCE(ld.PayCurrencyXchangeRateToMember, 0) AS PayToMemberXChangeRate,
        COALESCE(ld.PayCurrencyIdToProvider, CASE WHEN n.ServiceIndicatorId IS NOT NULL THEN 192 ELSE ld.PayCurrencyIdToProvider END, -101) AS PayToProviderCurrencyId,
        COALESCE(ld.PayCurrencyXchangeRateToProvider, 0) AS PayToProviderXChangeRate,
        -- Amounts
        COALESCE(ld.OriginalCurrencyBilledAmount, CASE WHEN COALESCE(ld.XChangeRate, 0) = 0 THEN NULL ELSE ld.XchangeBilledAmount / ld.XChangeRate END) AS LocalBilledAmount,
        COALESCE(CAST(ld.XchangeBilledAmount AS DOUBLE), 0) AS BaseBilledAmount,
        COALESCE(CAST(ld.CoInsuranceAmount AS DOUBLE), 0) AS BaseCoInsuranceAmount,
        CASE
            WHEN COALESCE(ld.XchangeBilledAmount, 0) - COALESCE(sldv.BaseTotalIneligibleAmount, 0) < 0
                THEN COALESCE(ld.NetCoveredAmount, 0)
            ELSE COALESCE(ld.XchangeBilledAmount, 0) - COALESCE(sldv.BaseTotalIneligibleAmount, 0)
        END AS BaseAllowedAmount,
        COALESCE(CAST(ld.MemberLiabilityAmount AS DOUBLE), 0) AS BaseCopayAmount,
        COALESCE(CAST(ld.DeductibleAmount AS DOUBLE), 0) AS BaseDeductibleAmount,
        COALESCE(CAST(ld.MemberPaidAmount AS DOUBLE), 0) AS BaseMemberPaidAmount,
        COALESCE(CAST(ld.NetCoveredAmount AS DOUBLE), 0) AS BaseNetCoveredAmount,
        CASE WHEN n.ServiceIndicatorId IS NULL THEN COALESCE(CAST(ld.ProviderPaidAmount AS DOUBLE), 0) ELSE COALESCE(CAST(ld.NetCoveredAmount AS DOUBLE), 0) END AS BaseProviderPaidAmount,
        COALESCE(sldv.BaseTotalIneligibleAmount, 0) AS BaseTotalIneligibleAmount,
        COALESCE(sldv.BaseOtherIneligibleAmount, 0) AS BaseOtherIneligibleAmount,
        COALESCE(CASE WHEN n.ServiceIndicatorId IS NULL THEN ld.ProviderPaidAmount ELSE ld.NetCoveredAmount END, 0) + COALESCE(ld.MemberPaidAmount, 0) AS BaseTotalPaidAmount,
        COALESCE(sldv.BaseHB_DiscountAmount, 0) AS BaseHB_DiscountAmount,
        COALESCE(CAST(ld.PayCurrencyAmountToMember AS DOUBLE), 0) AS PayCurrencyAmountToMember,
        COALESCE(CAST(ld.PayCurrencyAmountToProvider AS DOUBLE), 0) AS PayCurrencyAmountToProvider,
        COALESCE(CAST(ld.COBPrePaidAmount AS DOUBLE), 0) AS COBPrePaidAmount,
        COALESCE(CAST(ld.LocalCOBPrePaidAmount AS DOUBLE), 0) AS LocalCOBPrePaidAmount,
        cd.COBMemberId,
        COALESCE(sldv.BaseWithHolding, 0) AS BaseWithHolding,
        COALESCE(sldv.BaseVAT, 0) AS BaseVAT,
        COALESCE(ld.HTHFeePaid, 0) AS HTHFeePaid,
        COALESCE(ld.HTHFeePercent, 0) AS HTHFeePercent,
        COALESCE(tp.PayTo, 0) AS PayToThirParty,
        ROW_NUMBER() OVER (PARTITION BY ld.ClaimDetailId ORDER BY ld.ClaimLineDetailId) AS LineNumber
    FROM (
        -- FIX: deduplicar claim-level columns (vw_ClaimRun tiene N filas/ClaimDetailId por runs)
        SELECT DISTINCT ClaimDetailId, HeaderId, PolicyId, MemberEligibilityId, MemberId
        FROM vw_ClaimRun
    ) TCR
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON cd.ClaimDetailId = TCR.ClaimDetailId
    INNER JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail ld ON ld.ClaimDetailId = TCR.ClaimDetailId
    LEFT JOIN vw_SubLineDetailValue sldv ON sldv.ClaimLineDetailId = ld.ClaimLineDetailId AND sldv.ClaimDetailId = ld.ClaimDetailId
    LEFT JOIN vw_NetworkMetadataXref n ON n.ServiceIndicatorId = cd.ServiceIndicatorId
    LEFT JOIN vw_PolicyBaseCurrency_Payment pbc ON cd.PolicyId = pbc.PolicyId AND ld.FromDate BETWEEN pbc.FromDate AND COALESCE(pbc.ToDate, DATE '9999-12-31')
    LEFT JOIN Bronze.AmigosPlus_AMP_Claim_ThirdPartyClaim tp ON tp.ClaimDetailId = ld.ClaimDetailId
""")
df_base_claims.cache()
df_base_claims.createOrReplaceTempView("vw_BaseClaimsLines")
print(f"[OK] vw_BaseClaimsLines cached -- {df_base_claims.count():,} rows")


In [ ]:
# ============================================================
# Cell 16b: Enriquecimiento de vw_BaseClaimsLines con
#           Silver.AMPview_Claim_LineDetailPayExchangeRates
#
# Replica EXACTA de los 2 UPDATEs de usp_DataPrepare_FactClaimCostData (lÃƒÆ’Ã‚Â­neas 327-353):
#
#   -- UPDATE #1 (Member): condicionado a PayToMemberCurrencyId IS NULL AND BaseMemberPaidAmount > 0
#   UPDATE cl
#   SET cl.PayToMemberCurrencyId  = CASE WHEN PayToThirParty = 1008
#                                        THEN ldxr.ClaimPayToThirdPartyCurrencyID
#                                        ELSE ldxr.ClaimPayToMemberCurrencyID END,
#       cl.PayToMemberXChangeRate = ISNULL(cl.PayToMemberXChangeRate,
#                                          CASE WHEN PayToThirParty = 1008
#                                               THEN ldxr.BaseToThirdPartyExchangeRate
#                                               ELSE ldxr.BaseToMemberExchangeRate END)
#
#   -- UPDATE #2 (Provider): condicionado a PayToProviderCurrencyId IS NULL AND BaseProviderPaidAmount > 0
#   UPDATE cl
#   SET cl.PayToProviderCurrencyId  = CASE WHEN PayToThirParty = 1007
#                                          THEN ldxr.ClaimPayToThirdPartyCurrencyID
#                                          ELSE ldxr.ClaimPayToProviderCurrencyID END,
#       cl.PayToProviderXChangeRate = ISNULL(cl.PayToProviderXChangeRate,
#                                            CASE WHEN PayToThirParty = 1007
#                                                 THEN ldxr.BaseToThirdPartyExchangeRate
#                                                 ELSE ldxr.BaseToProviderExchangeRate END)
#
# Notas de equivalencia:
#  - Sentinel de NULL en vw_BaseClaimsLines: -101 para currencyId, 0 para XChangeRate.
#  - El XChangeRate del SP solo se rellena dentro del UPDATE (es decir, cuando el
#    Currency era NULL), porque el WHERE filtra Currency IS NULL. Reproducimos esto:
#    XChangeRate solo se actualiza cuando ademÃƒÆ’Ã‚Â¡s el Currency original estaba en NULL.
#  - Si el Currency original no era NULL, pasa por COALESCE-no-op (Curr y XR sin cambio).
# ============================================================

df_base_claims_enriched = spark.sql("""
    SELECT
        b.HeaderId, b.PolicyId, b.MemberEligibilityId, b.MemberId,
        b.ReferralHeaderId, b.POSId, b.TOSId, b.ServiceProviderId,
        b.ClaimDetailId, b.ClaimLineDetailId, b.ServiceIndicatorId, b.DetailServiceIndicatorId,
        b.FromDate, b.ToDate, b.XchangeDate, b.XChangeRate,
        b.LocalCurrencyId, b.BaseCurrencyId, b.BaseXChangeRate,

        -- ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ PayToMember ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
        -- Currency: si era NULL (-101) y MemberPaid > 0, asigna desde ldxr (1008ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ThirdParty, elseÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢Member)
        CASE
            WHEN b.PayToMemberCurrencyId = -101 AND b.BaseMemberPaidAmount > 0 THEN
                CASE WHEN b.PayToThirParty = 1008
                     THEN COALESCE(ldxr.ClaimPayToThirdPartyCurrencyID, b.PayToMemberCurrencyId)
                     ELSE COALESCE(ldxr.ClaimPayToMemberCurrencyID,    b.PayToMemberCurrencyId)
                END
            ELSE b.PayToMemberCurrencyId
        END AS PayToMemberCurrencyId,

        -- XChangeRate: solo se rellena DENTRO del UPDATE (Currency era NULL Y MemberPaid > 0),
        -- y solo si el rate actual es 0 (sentinel de NULL): isnull(cl.XR, ...)
        CASE
            WHEN b.PayToMemberCurrencyId = -101 AND b.BaseMemberPaidAmount > 0
                 AND COALESCE(b.PayToMemberXChangeRate, 0) = 0 THEN
                CASE WHEN b.PayToThirParty = 1008
                     THEN COALESCE(ldxr.BaseToThirdPartyExchangeRate, b.PayToMemberXChangeRate)
                     ELSE COALESCE(ldxr.BaseToMemberExchangeRate,    b.PayToMemberXChangeRate)
                END
            ELSE b.PayToMemberXChangeRate
        END AS PayToMemberXChangeRate,

        -- ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ PayToProvider ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
        CASE
            WHEN b.PayToProviderCurrencyId = -101 AND b.BaseProviderPaidAmount > 0 THEN
                CASE WHEN b.PayToThirParty = 1007
                     THEN COALESCE(ldxr.ClaimPayToThirdPartyCurrencyID, b.PayToProviderCurrencyId)
                     ELSE COALESCE(ldxr.ClaimPayToProviderCurrencyID,   b.PayToProviderCurrencyId)
                END
            ELSE b.PayToProviderCurrencyId
        END AS PayToProviderCurrencyId,

        CASE
            WHEN b.PayToProviderCurrencyId = -101 AND b.BaseProviderPaidAmount > 0
                 AND COALESCE(b.PayToProviderXChangeRate, 0) = 0 THEN
                CASE WHEN b.PayToThirParty = 1007
                     THEN COALESCE(ldxr.BaseToThirdPartyExchangeRate, b.PayToProviderXChangeRate)
                     ELSE COALESCE(ldxr.BaseToProviderExchangeRate,   b.PayToProviderXChangeRate)
                END
            ELSE b.PayToProviderXChangeRate
        END AS PayToProviderXChangeRate,

        -- ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Resto de columnas sin cambios ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
        b.LocalBilledAmount, b.BaseBilledAmount, b.BaseCoInsuranceAmount, b.BaseAllowedAmount,
        b.BaseCopayAmount, b.BaseDeductibleAmount, b.BaseMemberPaidAmount, b.BaseNetCoveredAmount,
        b.BaseProviderPaidAmount, b.BaseTotalIneligibleAmount, b.BaseOtherIneligibleAmount,
        b.BaseTotalPaidAmount, b.BaseHB_DiscountAmount, b.PayCurrencyAmountToMember,
        b.PayCurrencyAmountToProvider, b.COBPrePaidAmount, b.LocalCOBPrePaidAmount,
        b.COBMemberId, b.BaseWithHolding, b.BaseVAT, b.HTHFeePaid, b.HTHFeePercent,
        b.PayToThirParty, b.LineNumber
    FROM vw_BaseClaimsLines b
    LEFT JOIN Silver.AMPview_Claim_LineDetailPayExchangeRates ldxr
      ON ldxr.ClaimDetailId     = b.ClaimDetailId
     AND ldxr.ClaimLineDetailId = b.ClaimLineDetailId
""")

df_base_claims_enriched.cache()
df_base_claims_enriched.createOrReplaceTempView("vw_BaseClaimsLines")
print(f"[OK] vw_BaseClaimsLines enriquecida con ldxr ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â {df_base_claims_enriched.count():,} rows")


In [ ]:
# Cell 18: Vista temporal - ClaimLineGrouping (agrupaciÃƒÆ’Ã‚Â³n de lÃƒÆ’Ã‚Â­neas por claim)
df_claim_line_groupping = spark.sql("""
    SELECT
        HeaderId, PolicyId, MemberEligibilityId, MemberId,
        CAST(NULL AS INT) AS ReferralHeaderId,
        -101 AS POSId, -101 AS TOSId, -101 AS ServiceProviderId,
        ClaimDetailId,
        -101 AS ServiceIndicatorId,
        DetailServiceIndicatorId,
        MIN(FromDate) AS FromDate,
        MAX(ToDate) AS ToDate,
        CAST(NULL AS DATE) AS XchangeDate,
        CAST(NULL AS DOUBLE) AS XChangeRate,
        -101 AS LocalCurrencyId,
        BaseCurrencyId,
        SUM(LocalBilledAmount) AS LocalBilledAmount,
        SUM(BaseBilledAmount) AS BaseBilledAmount,
        SUM(BaseCoInsuranceAmount) AS BaseCoInsuranceAmount,
        SUM(BaseAllowedAmount) AS BaseAllowedAmount,
        SUM(BaseCopayAmount) AS BaseCopayAmount,
        SUM(BaseDeductibleAmount) AS BaseDeductibleAmount,
        SUM(BaseMemberPaidAmount) AS BaseMemberPaidAmount,
        SUM(BaseNetCoveredAmount) AS BaseNetCoveredAmount,
        SUM(BaseProviderPaidAmount) AS BaseProviderPaidAmount,
        SUM(BaseTotalIneligibleAmount) AS BaseTotalIneligibleAmount,
        SUM(BaseOtherIneligibleAmount) AS BaseOtherIneligibleAmount,
        SUM(BaseTotalPaidAmount) AS BaseTotalPaidAmount,
        CAST(0 AS DOUBLE) AS BaseAllowedCmp,
        CAST(0 AS DOUBLE) AS BaseTotPaidCmp,
        CAST(0 AS DOUBLE) AS BaseFX_DiscountAmount,
        SUM(BaseHB_DiscountAmount) AS BaseHB_DiscountAmount,
        SUM(PayCurrencyAmountToMember) AS PayCurrencyAmountToMember,
        SUM(PayCurrencyAmountToProvider) AS PayCurrencyAmountToProvider,
        SUM(COBPrePaidAmount) AS COBPrePaidAmount,
        SUM(LocalCOBPrePaidAmount) AS LocalCOBPrePaidAmount,
        SUM(BaseWithHolding) AS BaseWithHoldingAmount,
        SUM(BaseVAT) AS BaseVATAmount,
        SUM(HTHFeePaid) AS HTHFeePaid,
        AVG(HTHFeePercent) AS HTHFeePercent,
        CAST(NULL AS DOUBLE) AS BaseXChangeRate,
        MAX(CASE WHEN BaseProviderPaidAmount > 0 AND PayToProviderCurrencyId = -101 THEN 192 ELSE PayToProviderCurrencyId END) AS PayToProviderCurrencyId,
        MAX(PayToProviderXChangeRate) AS PayToProviderXChangeRate,
        MAX(CASE WHEN BaseMemberPaidAmount > 0 AND PayToMemberCurrencyId = -101 THEN 192 ELSE PayToMemberCurrencyId END) AS PayToMemberCurrencyId,
        MAX(PayToMemberXChangeRate) AS PayToMemberXChangeRate
    FROM vw_BaseClaimsLines
    GROUP BY HeaderId, PolicyId, MemberEligibilityId, MemberId, DetailServiceIndicatorId, ClaimDetailId, BaseCurrencyId
""")
df_claim_line_groupping.createOrReplaceTempView("vw_ClaimLineGroupping")
print("[OK] vw_ClaimLineGroupping created")

In [ ]:
# ============================================================
# Cell 18b: Vista temporal - vw_ClaimsBaseCurrencyId
# Mapea ClaimDetailId -> currencies + totales pagados por claim.
# Equivalente al temp table #ClaimsBaseCurrencyId del SP usp_DataPrepare_FactClaimCostData.
# Columnas:
#   - BaseCurrencyId, PayToProviderCurrencyId, PayToMemberCurrencyId  (resoluciÃƒÆ’Ã‚Â³n de XChangeRate)
#   - BaseProviderPaidAmount / PayProviderPaidAmount  (totales en moneda base / pago)
#   - BaseMemberPaidAmount   / PayMemberPaidAmount    (totales en moneda base / pago)
#   Estos totales son consumidos por vw_*PPAAdjustment (window functions).
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_ClaimsBaseCurrencyId AS
    SELECT
        ClaimDetailId,
        MAX(BaseCurrencyId) AS BaseCurrencyId,
        MAX(CASE WHEN BaseProviderPaidAmount > 0 AND PayToProviderCurrencyId = -101 THEN 192
                 ELSE PayToProviderCurrencyId END) AS PayToProviderCurrencyId,
        MAX(CASE WHEN BaseMemberPaidAmount   > 0 AND PayToMemberCurrencyId   = -101 THEN 192
                 ELSE PayToMemberCurrencyId END) AS PayToMemberCurrencyId,
        SUM(BaseProviderPaidAmount)                       AS BaseProviderPaidAmount,
        SUM(COALESCE(PayCurrencyAmountToProvider, 0))     AS PayProviderPaidAmount,
        SUM(BaseMemberPaidAmount)                         AS BaseMemberPaidAmount,
        SUM(COALESCE(PayCurrencyAmountToMember, 0))       AS PayMemberPaidAmount
    FROM vw_BaseClaimsLines
    GROUP BY ClaimDetailId
""")
print("[OK] vw_ClaimsBaseCurrencyId created")


In [ ]:
# Cell 19: Vista temporal - vw_ProviderPPA (PPA Provider)
# Replica los 4 INSERT del SP a #PROVIDERS_PPA_TABLE en usp_DataPrepare_FactClaimCostData:
#   #1 ProviderBalance sin PaymentLog (run.PaymentId IS NULL,  NetworkMetadataXrefId IS NULL)
#   #2 ProviderBalance con PaymentLog  (PostedOn BETWEEN,      NetworkMetadataXrefId IS NULL)
#   #3 PROVIDERS Network con PaymentLog: TmpClaimPayableMulticurrency + INNER JOIN NetworkMetadataXref
#                                       (cubre redes GeoBlue/UHCI/Olympus, RunTypeId IN 15,16,17,18)
#   #4 PROVIDERS Network sin PaymentLog: TmpClaimPayableMulticurrency + INNER JOIN NetworkMetadataXref
#                                       (run.PaymentId IS NULL, IsProvider = 1)
df_provider_ppa = spark.sql(f"""
    SELECT
        ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId,
        SUM(ppa.BasePPABalance) AS BaseAmount,
        SUM(ppa.PayPPABalance) AS PaidAmount,
        ppa.XChangeRate,
        ppa.DetailRunId, ppa.JournalEntryId,
        ppa.PolicyId, ppa.postedon,
        ppa.PayClaimDetailId, ppa.PayDetailRunId
    FROM (
        -- INSERT #1 + #2: ProviderBalance directo (sin red externa)
        SELECT
            run.RunId,
            detail.HeaderId,
            detailrun.ClaimDetailId,
            detailrun.DetailRunId,
            pb.PayDetailRunId,
            -1 * COALESCE(pb.Balance, 0) AS BasePPABalance,
            -1 * COALESCE(pb.Balance / CASE WHEN pb.XChangeRate = 0 OR claims.BaseCurrencyId = CASE WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId ELSE claims.PayToMemberCurrencyId END THEN 1 ELSE pb.XChangeRate END, 0) AS PayPPABalance,
            CASE WHEN pb.XChangeRate = 0 OR claims.BaseCurrencyId = CASE WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId ELSE claims.PayToMemberCurrencyId END THEN 1 ELSE pb.XChangeRate END AS XChangeRate,
            journal.JournalEntryId,
            journal.PostedOn AS postedon,
            detail.PolicyId,
            paydetailrun.ClaimDetailId AS PayClaimDetailId
        FROM Bronze.AmigosPlus_AMP_Claim_ProviderBalance pb
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun paydetailrun ON paydetailrun.DetailRunId = pb.PayDetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run run ON run.RunId = paydetailrun.RunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun detailrun ON detailrun.DetailRunId = pb.DetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = detailrun.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run r ON r.RunId = detailrun.RunId
        INNER JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = paydetailrun.ClaimDetailId
        LEFT JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog ON paymentlog.PaymentId = run.PaymentId
        LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry journal ON journal.JournalEntryId = paymentlog.JournalEntryId
        LEFT JOIN Bronze.AmigosPlus_AMP_Claim_NetworkMetadataXref nm ON nm.RunTypeId = run.RunTypeId
        WHERE run.IsProvider = 1
          AND COALESCE(pb.PaymentTypeId, -1) <> 2
          AND pb.ReversedJournalEntryId IS NULL
          AND nm.NetworkMetadataXrefId IS NULL
          AND COALESCE(pb.Balance, 0) <> 0

        UNION ALL

        -- INSERT #3: PROVIDERS Network CON PaymentLog
        -- Fuente: Silver.TmpClaimPayableMulticurrency + INNER JOIN NetworkMetadataXref
        -- Cubre redes GeoBlue (500), UHCI (432), Olympus (773) con RunTypeId IN (15,16,17,18)
        SELECT
            run.RunId,
            detail.HeaderId,
            paydetailrun.ClaimDetailId,
            paydetailrun.DetailRunId,
            paydetailrun.DetailRunId AS PayDetailRunId,
            -1 * (pje.BaseAmount - pje.BaseAmountAdjusted) AS BasePPABalance,
            -1 * (pje.PaidAmount - pje.PaidAmountAdjusted) AS PayPPABalance,
            CASE WHEN (pje.BaseAmount - pje.BaseAmountAdjusted) = 0 OR (pje.PaidAmount - pje.PaidAmountAdjusted) = 0 THEN 1
                 ELSE (pje.PaidAmount - pje.PaidAmountAdjusted) / (pje.BaseAmount - pje.BaseAmountAdjusted) END AS XChangeRate,
            journal.JournalEntryId,
            journal.PostedOn AS postedon,
            detail.PolicyId,
            paydetailrun.ClaimDetailId AS PayClaimDetailId
        FROM Bronze.AmigosPlus_AMP_Claim_Run run
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_NetworkMetadataXref nm ON nm.RunTypeId = run.RunTypeId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun paydetailrun ON paydetailrun.RunId = run.RunId
        INNER JOIN Silver.TmpClaimPayableMulticurrency pje
                ON pje.ClaimDetailId = paydetailrun.ClaimDetailId
               AND pje.AmountTypeId = nm.AmountTypeId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = paydetailrun.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog ON paymentlog.PaymentId = run.PaymentId
        INNER JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry journal ON journal.JournalEntryId = paymentlog.JournalEntryId
        WHERE paymentlog.StatusId NOT IN (153, 156, 174, 264, 265, 275)
          AND (pje.BaseAmount - pje.BaseAmountAdjusted) <> 0
          AND journal.PostedOn BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}'

        UNION ALL

        -- INSERT #4: PROVIDERS Network SIN PaymentLog (run.PaymentId IS NULL)
        SELECT
            run.RunId,
            detail.HeaderId,
            paydetailrun.ClaimDetailId,
            paydetailrun.DetailRunId,
            paydetailrun.DetailRunId AS PayDetailRunId,
            -1 * (pje.BaseAmount - pje.BaseAmountAdjusted) AS BasePPABalance,
            -1 * (pje.PaidAmount - pje.PaidAmountAdjusted) AS PayPPABalance,
            CASE WHEN (pje.BaseAmount - pje.BaseAmountAdjusted) = 0 OR (pje.PaidAmount - pje.PaidAmountAdjusted) = 0 THEN 1
                 ELSE (pje.PaidAmount - pje.PaidAmountAdjusted) / (pje.BaseAmount - pje.BaseAmountAdjusted) END AS XChangeRate,
            CAST(NULL AS BIGINT) AS JournalEntryId,
            CAST(NULL AS TIMESTAMP) AS postedon,
            detail.PolicyId,
            paydetailrun.ClaimDetailId AS PayClaimDetailId
        FROM Bronze.AmigosPlus_AMP_Claim_Run run
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_NetworkMetadataXref nm ON nm.RunTypeId = run.RunTypeId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun paydetailrun ON paydetailrun.RunId = run.RunId
        INNER JOIN Silver.TmpClaimPayableMulticurrency pje
                ON pje.ClaimDetailId = paydetailrun.ClaimDetailId
               AND pje.AmountTypeId = nm.AmountTypeId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = paydetailrun.ClaimDetailId
        WHERE run.PaymentId IS NULL
          AND run.IsProvider = 1
          AND (pje.BaseAmount - pje.BaseAmountAdjusted) <> 0
    ) ppa
    GROUP BY ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId, ppa.DetailRunId, ppa.JournalEntryId, ppa.PolicyId, ppa.postedon, ppa.PayClaimDetailId, ppa.PayDetailRunId, ppa.XChangeRate
""")
df_provider_ppa.cache()
df_provider_ppa.createOrReplaceTempView("vw_ProviderPPA")
spark.table("vw_ProviderPPA").count()  # Force cache materialization

print("[OK] vw_ProviderPPA created (4 paths: ProviderBalance #1+#2 + Network #3+#4) [cached]")

In [ ]:
# Cell 20: Vista temporal - PolicyBalance (PPA Member)
df_member_ppa = spark.sql("""
    SELECT
        ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId,
        SUM(ppa.BasePPABalance) AS BaseAmount,
        SUM(ppa.PayPPABalance) AS PaidAmount,
        ppa.XChangeRate,
        ppa.DetailRunId, ppa.JournalEntryId,
        ppa.PolicyId, ppa.postedon,
        ppa.PayClaimDetailId, ppa.PayDetailRunId
    FROM (
        SELECT
            run.RunId,
            detail.HeaderId,
            detailrun.ClaimDetailId,
            detailrun.DetailRunId,
            pb.PayDetailRunId,
            -1 * COALESCE(pb.Balance, 0) AS BasePPABalance,
            -1 * COALESCE(pb.Balance / CASE WHEN pb.XChangeRate = 0 OR claims.BaseCurrencyId = CASE WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId ELSE claims.PayToMemberCurrencyId END OR (pb.XChangeRate IS NULL AND claims.BaseCurrencyId = 192) THEN 1 ELSE pb.XChangeRate END, 0) AS PayPPABalance,
            CASE WHEN pb.XChangeRate = 0 OR claims.BaseCurrencyId = CASE WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId ELSE claims.PayToMemberCurrencyId END OR (pb.XChangeRate IS NULL AND claims.BaseCurrencyId = 192) THEN 1 ELSE pb.XChangeRate END AS XChangeRate,
            journal.JournalEntryId,
            journal.PostedOn AS postedon,
            detail.PolicyId,
            paydetailrun.ClaimDetailId AS PayClaimDetailId
        FROM Bronze.AmigosPlus_AMP_Claim_PolicyBalance pb
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun detailrun ON detailrun.DetailRunId = pb.DetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = detailrun.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run r ON r.RunId = detailrun.RunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun paydetailrun ON paydetailrun.DetailRunId = pb.PayDetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run run ON run.RunId = paydetailrun.RunId
        LEFT JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = paydetailrun.ClaimDetailId
        LEFT JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog ON paymentlog.PaymentId = run.PaymentId
        LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry journal ON journal.JournalEntryId = paymentlog.JournalEntryId
        WHERE run.IsProvider = 0
          AND pb.DetailRunId <> pb.PayDetailRunId
          AND COALESCE(pb.PaymentTypeId, -1) <> 2
          AND pb.ReversedJournalEntryId IS NULL
          AND pb.PayDetailRunId IS NOT NULL
          AND pb.DetailRunId IS NOT NULL
          AND COALESCE(pb.Balance, 0) <> 0
    ) ppa
    GROUP BY ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId, ppa.DetailRunId, ppa.JournalEntryId, ppa.PolicyId, ppa.postedon, ppa.PayClaimDetailId, ppa.PayDetailRunId, ppa.XChangeRate
""")
df_member_ppa.cache()
df_member_ppa.createOrReplaceTempView("vw_MemberPPA")

spark.table("vw_MemberPPA").count()  # Force cache materializationprint("[OK] vw_MemberPPA created [cached]")

In [ ]:
# ============================================================
# Cell 20b: PPA Adjustment - Distribucion via Window Functions
# Reemplaza los 4 CTEs recursivos del SP (OPTION MAXRECURSION 1000):
#   #PROVIDERS_PPA_TABLE_ADJUSTMENT       -> vw_ProviderPPAAdjustment
#   #PROVIDERS_PPA_VOIDS_TABLE_ADJUSTMENT -> vw_ProviderPPAVoidAdjustment
#   #MEMBERS_PPA_TABLE_ADJUSTMENT         -> vw_MemberPPAAdjustment
#   #MEMBERS_PPA_VOIDS_TABLE_ADJUSTMENT   -> vw_MemberPPAVoidAdjustment
#
# Cada vista expone 4 columnas por linea (RunId, DetailRunId, ClaimDetailId,
# ClaimLineDetailId):
#   adj_base  = LinePaidBase  - alloc_base   (negativo para void)
#   adj_pay   = LinePaidPay   - alloc_pay    (negativo para void)
#   paid_base = LinePaidBase  completo       (negativo para void)
#   paid_pay  = LinePaidPay   completo       (negativo para void)
#
# Algoritmo equivalente al CTE recursivo:
#   cap = LEAST(ABS(ppa_total), total_paid)
#   alloc_i = LEAST(cap, CumPaid_i) - LEAST(cap, CumPaid_{i-1})
# Caso overshoot (ABS(PPA) > TotalPaid): el residual va a la ultima linea
# (replica el comportamiento del CTE recursivo de SQL Server).
# ============================================================


def cache_temp_view(view_name: str):
    """Materializa y re-registra una temp view para evitar recalcularla."""
    df = spark.table(view_name)
    df.cache()
    df.createOrReplaceTempView(view_name)
    df.count()
    print(f"[OK] {view_name} cached")
    return df


# ------------------------------------------------------------
# Crear Void PPA views anticipadamente (se necesitan tambien
# para los INSERT VOID PPA *).
# ------------------------------------------------------------
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_ProviderPPA_Void AS
    SELECT
        ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId,
        SUM(ppa.BasePPABalance) AS BaseAmount,
        SUM(ppa.PayPPABalance) AS PaidAmount,
        ppa.XChangeRate,
        ppa.DetailRunId, ppa.JournalEntryId,
        ppa.PolicyId, ppa.postedon,
        ppa.PayClaimDetailId, ppa.PayDetailRunId
    FROM (
        SELECT
            run.RunId, detail.HeaderId, detailrun.ClaimDetailId, detailrun.DetailRunId,
            pb.PayDetailRunId,
            COALESCE(pb.Balance, 0) AS BasePPABalance,
            COALESCE(
                pb.Balance / CASE
                    WHEN pb.XChangeRate = 0
                      OR claims.BaseCurrencyId = CASE
                            WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId
                            ELSE claims.PayToMemberCurrencyId
                         END
                    THEN 1 ELSE pb.XChangeRate
                END,
                0
            ) AS PayPPABalance,
            CASE
                WHEN pb.XChangeRate = 0
                  OR claims.BaseCurrencyId = CASE
                        WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId
                        ELSE claims.PayToMemberCurrencyId
                     END
                THEN 1 ELSE pb.XChangeRate
            END AS XChangeRate,
            journal.JournalEntryId, journal.PostedOn AS postedon, detail.PolicyId,
            paydetailrun.ClaimDetailId AS PayClaimDetailId
        FROM Bronze.AmigosPlus_AMP_Claim_ProviderBalance pb
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun detailrun ON detailrun.DetailRunId = pb.DetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = detailrun.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run r ON r.RunId = detailrun.RunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun paydetailrun ON paydetailrun.DetailRunId = pb.PayDetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run run ON run.RunId = paydetailrun.RunId
        INNER JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = paydetailrun.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog ON paymentlog.PaymentId = run.PaymentId
        INNER JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry journal ON journal.JournalEntryId = paymentlog.JournalEntryId
        LEFT JOIN Bronze.AmigosPlus_AMP_Claim_NetworkMetadataXref nm ON nm.RunTypeId = run.RunTypeId
        WHERE paymentlog.StatusId IN (153, 156, 174, 264, 265, 275)
          AND pb.DetailRunId <> pb.PayDetailRunId
          AND COALESCE(pb.PaymentTypeId, -1) <> 2
          AND pb.ReversedJournalEntryId IS NULL
          AND pb.PayDetailRunId IS NOT NULL
          AND pb.DetailRunId IS NOT NULL
          AND nm.NetworkMetadataXrefId IS NULL
          AND COALESCE(pb.Balance, 0) <> 0
          AND journal.PostedOn BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}'
    ) ppa
    GROUP BY ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId, ppa.DetailRunId, ppa.JournalEntryId,
             ppa.PolicyId, ppa.postedon, ppa.PayClaimDetailId, ppa.PayDetailRunId, ppa.XChangeRate
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_MemberPPA_Void AS
    SELECT
        ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId,
        SUM(ppa.BasePPABalance) AS BaseAmount,
        SUM(ppa.PayPPABalance) AS PaidAmount,
        ppa.XChangeRate,
        ppa.DetailRunId, ppa.JournalEntryId,
        ppa.PolicyId, ppa.postedon,
        ppa.PayClaimDetailId, ppa.PayDetailRunId
    FROM (
        SELECT
            run.RunId, detail.HeaderId, detailrun.ClaimDetailId, detailrun.DetailRunId,
            pb.PayDetailRunId,
            COALESCE(pb.Balance, 0) AS BasePPABalance,
            COALESCE(
                pb.Balance / CASE
                    WHEN pb.XChangeRate = 0
                      OR claims.BaseCurrencyId = CASE
                            WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId
                            ELSE claims.PayToMemberCurrencyId
                         END
                      OR (pb.XChangeRate IS NULL AND claims.BaseCurrencyId = 192)
                    THEN 1 ELSE pb.XChangeRate
                END,
                0
            ) AS PayPPABalance,
            CASE
                WHEN pb.XChangeRate = 0
                  OR claims.BaseCurrencyId = CASE
                        WHEN r.IsProvider = 1 THEN claims.PayToProviderCurrencyId
                        ELSE claims.PayToMemberCurrencyId
                     END
                  OR (pb.XChangeRate IS NULL AND claims.BaseCurrencyId = 192)
                THEN 1 ELSE pb.XChangeRate
            END AS XChangeRate,
            journal.JournalEntryId, journal.PostedOn AS postedon, detail.PolicyId,
            paydetailrun.ClaimDetailId AS PayClaimDetailId
        FROM Bronze.AmigosPlus_AMP_Claim_PolicyBalance pb
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun detailrun ON detailrun.DetailRunId = pb.DetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail detail ON detail.ClaimDetailId = detailrun.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run r ON r.RunId = detailrun.RunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun paydetailrun ON paydetailrun.DetailRunId = pb.PayDetailRunId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run run ON run.RunId = paydetailrun.RunId
        INNER JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = paydetailrun.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog ON paymentlog.PaymentId = run.PaymentId
        INNER JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry journal ON journal.JournalEntryId = paymentlog.JournalEntryId
        LEFT JOIN Bronze.AmigosPlus_AMP_Claim_NetworkMetadataXref nm ON nm.RunTypeId = run.RunTypeId
        WHERE paymentlog.StatusId IN (153, 156, 174, 264, 265, 275)
          AND pb.DetailRunId <> pb.PayDetailRunId
          AND COALESCE(pb.PaymentTypeId, -1) <> 2
          AND pb.ReversedJournalEntryId IS NULL
          AND pb.PayDetailRunId IS NOT NULL
          AND pb.DetailRunId IS NOT NULL
          AND COALESCE(pb.Balance, 0) <> 0
          AND journal.PostedOn BETWEEN '{FROM_DATE_STR}' AND '{TO_DATE_STR}'
    ) ppa
    GROUP BY ppa.RunId, ppa.HeaderId, ppa.ClaimDetailId, ppa.DetailRunId, ppa.JournalEntryId,
             ppa.PolicyId, ppa.postedon, ppa.PayClaimDetailId, ppa.PayDetailRunId, ppa.XChangeRate
""")
print("[OK] vw_ProviderPPA_Void, vw_MemberPPA_Void created")


# ------------------------------------------------------------
# 1) Provider PPA Non-Void Adjustment
# ------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_ProviderPPAAdjustment AS
    WITH ppa_agg AS (
        SELECT RunId, PayDetailRunId, PayClaimDetailId,
               SUM(BaseAmount) AS PPABase, SUM(PaidAmount) AS PPAPay
        FROM vw_ProviderPPA
        GROUP BY RunId, PayDetailRunId, PayClaimDetailId
    ),
    ppa_lines AS (
        SELECT
            ppa.RunId,
            ppa.PayDetailRunId AS DetailRunId,
            ppa.PayClaimDetailId AS ClaimDetailId,
            tcl.ClaimLineDetailId,
            ppa.PPABase, ppa.PPAPay,
            claims.BaseProviderPaidAmount AS TotalPaidBase,
            claims.PayProviderPaidAmount  AS TotalPaidPay,
            tcl.BaseProviderPaidAmount AS LinePaidBase,
            COALESCE(tcl.PayCurrencyAmountToProvider, 0) AS LinePaidPay,
            tcl.LineNumber,
            SUM(tcl.BaseProviderPaidAmount) OVER w AS CumPaidBase,
            SUM(COALESCE(tcl.PayCurrencyAmountToProvider, 0)) OVER w AS CumPaidPay,
            COALESCE(SUM(tcl.BaseProviderPaidAmount) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidBase,
            COALESCE(SUM(COALESCE(tcl.PayCurrencyAmountToProvider, 0)) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidPay,
            MAX(tcl.LineNumber) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ) AS MaxLineNumber
        FROM ppa_agg ppa
        INNER JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = ppa.PayClaimDetailId
        INNER JOIN vw_BaseClaimsLines tcl       ON tcl.ClaimDetailId    = ppa.PayClaimDetailId
        WHERE ppa.PPABase < 0 OR ppa.PPAPay < 0
        WINDOW w AS (
            PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
    )
    SELECT RunId, DetailRunId, ClaimDetailId, ClaimLineDetailId,
        CAST(LinePaidBase - (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN ABS(PPABase) - LEAST(LEAST(ABS(PPABase), TotalPaidBase), PrevCumPaidBase)
                 ELSE LEAST(LEAST(ABS(PPABase), TotalPaidBase), CumPaidBase)
                      - LEAST(LEAST(ABS(PPABase), TotalPaidBase), PrevCumPaidBase)
            END
        ) AS DOUBLE) AS adj_base,
        CAST(LinePaidPay - (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN ABS(PPAPay) - LEAST(LEAST(ABS(PPAPay), TotalPaidPay), PrevCumPaidPay)
                 ELSE LEAST(LEAST(ABS(PPAPay), TotalPaidPay), CumPaidPay)
                      - LEAST(LEAST(ABS(PPAPay), TotalPaidPay), PrevCumPaidPay)
            END
        ) AS DOUBLE) AS adj_pay,
        CAST(LinePaidBase AS DOUBLE) AS paid_base,
        CAST(LinePaidPay  AS DOUBLE) AS paid_pay,
        LineNumber
    FROM ppa_lines
""")
print("[OK] vw_ProviderPPAAdjustment created")
_df_ppa_adj = cache_temp_view("vw_ProviderPPAAdjustment")


# ------------------------------------------------------------
# 2) Provider PPA Void Adjustment
# ------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_ProviderPPAVoidAdjustment AS
    WITH ppa_agg AS (
        SELECT RunId, PayDetailRunId, PayClaimDetailId,
               SUM(BaseAmount) AS PPABase, SUM(PaidAmount) AS PPAPay
        FROM vw_ProviderPPA_Void
        GROUP BY RunId, PayDetailRunId, PayClaimDetailId
    ),
    ppa_lines AS (
        SELECT
            ppa.RunId,
            ppa.PayDetailRunId AS DetailRunId,
            ppa.PayClaimDetailId AS ClaimDetailId,
            tcl.ClaimLineDetailId,
            ppa.PPABase, ppa.PPAPay,
            claims.BaseProviderPaidAmount AS TotalPaidBase,
            claims.PayProviderPaidAmount  AS TotalPaidPay,
            tcl.BaseProviderPaidAmount AS LinePaidBase,
            COALESCE(tcl.PayCurrencyAmountToProvider, 0) AS LinePaidPay,
            tcl.LineNumber,
            SUM(tcl.BaseProviderPaidAmount) OVER w AS CumPaidBase,
            SUM(COALESCE(tcl.PayCurrencyAmountToProvider, 0)) OVER w AS CumPaidPay,
            COALESCE(SUM(tcl.BaseProviderPaidAmount) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidBase,
            COALESCE(SUM(COALESCE(tcl.PayCurrencyAmountToProvider, 0)) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidPay,
            MAX(tcl.LineNumber) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ) AS MaxLineNumber
        FROM ppa_agg ppa
        INNER JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = ppa.PayClaimDetailId
        INNER JOIN vw_BaseClaimsLines tcl       ON tcl.ClaimDetailId    = ppa.PayClaimDetailId
        WHERE ppa.PPABase > 0 OR ppa.PPAPay > 0
        WINDOW w AS (
            PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
    )
    SELECT RunId, DetailRunId, ClaimDetailId, ClaimLineDetailId,
        CAST(-LinePaidBase + (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN PPABase - LEAST(LEAST(PPABase, TotalPaidBase), PrevCumPaidBase)
                 ELSE LEAST(LEAST(PPABase, TotalPaidBase), CumPaidBase)
                      - LEAST(LEAST(PPABase, TotalPaidBase), PrevCumPaidBase)
            END
        ) AS DOUBLE) AS adj_base,
        CAST(-LinePaidPay + (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN PPAPay - LEAST(LEAST(PPAPay, TotalPaidPay), PrevCumPaidPay)
                 ELSE LEAST(LEAST(PPAPay, TotalPaidPay), CumPaidPay)
                      - LEAST(LEAST(PPAPay, TotalPaidPay), PrevCumPaidPay)
            END
        ) AS DOUBLE) AS adj_pay,
        CAST(-LinePaidBase AS DOUBLE) AS paid_base,
        CAST(-LinePaidPay  AS DOUBLE) AS paid_pay,
        LineNumber
    FROM ppa_lines
""")
print("[OK] vw_ProviderPPAVoidAdjustment created")
_df_ppa_v_adj = cache_temp_view("vw_ProviderPPAVoidAdjustment")


# ------------------------------------------------------------
# 3) Member PPA Non-Void Adjustment
# ------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_MemberPPAAdjustment AS
    WITH ppa_agg AS (
        SELECT RunId, PayDetailRunId, PayClaimDetailId,
               SUM(BaseAmount) AS PPABase, SUM(PaidAmount) AS PPAPay
        FROM vw_MemberPPA
        GROUP BY RunId, PayDetailRunId, PayClaimDetailId
    ),
    ppa_lines AS (
        SELECT
            ppa.RunId,
            ppa.PayDetailRunId AS DetailRunId,
            ppa.PayClaimDetailId AS ClaimDetailId,
            tcl.ClaimLineDetailId,
            ppa.PPABase, ppa.PPAPay,
            claims.BaseMemberPaidAmount AS TotalPaidBase,
            claims.PayMemberPaidAmount  AS TotalPaidPay,
            tcl.BaseMemberPaidAmount AS LinePaidBase,
            COALESCE(tcl.PayCurrencyAmountToMember, 0) AS LinePaidPay,
            tcl.LineNumber,
            SUM(tcl.BaseMemberPaidAmount) OVER w AS CumPaidBase,
            SUM(COALESCE(tcl.PayCurrencyAmountToMember, 0)) OVER w AS CumPaidPay,
            COALESCE(SUM(tcl.BaseMemberPaidAmount) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidBase,
            COALESCE(SUM(COALESCE(tcl.PayCurrencyAmountToMember, 0)) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidPay,
            MAX(tcl.LineNumber) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ) AS MaxLineNumber
        FROM ppa_agg ppa
        INNER JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = ppa.PayClaimDetailId
        INNER JOIN vw_BaseClaimsLines tcl       ON tcl.ClaimDetailId    = ppa.PayClaimDetailId
        WHERE ppa.PPABase < 0 OR ppa.PPAPay < 0
        WINDOW w AS (
            PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
    )
    SELECT RunId, DetailRunId, ClaimDetailId, ClaimLineDetailId,
        CAST(LinePaidBase - (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN ABS(PPABase) - LEAST(LEAST(ABS(PPABase), TotalPaidBase), PrevCumPaidBase)
                 ELSE LEAST(LEAST(ABS(PPABase), TotalPaidBase), CumPaidBase)
                      - LEAST(LEAST(ABS(PPABase), TotalPaidBase), PrevCumPaidBase)
            END
        ) AS DOUBLE) AS adj_base,
        CAST(LinePaidPay - (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN ABS(PPAPay) - LEAST(LEAST(ABS(PPAPay), TotalPaidPay), PrevCumPaidPay)
                 ELSE LEAST(LEAST(ABS(PPAPay), TotalPaidPay), CumPaidPay)
                      - LEAST(LEAST(ABS(PPAPay), TotalPaidPay), PrevCumPaidPay)
            END
        ) AS DOUBLE) AS adj_pay,
        CAST(LinePaidBase AS DOUBLE) AS paid_base,
        CAST(LinePaidPay  AS DOUBLE) AS paid_pay,
        LineNumber
    FROM ppa_lines
""")
print("[OK] vw_MemberPPAAdjustment created")
_df_mppa_adj = cache_temp_view("vw_MemberPPAAdjustment")


# ------------------------------------------------------------
# 4) Member PPA Void Adjustment
# ------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_MemberPPAVoidAdjustment AS
    WITH ppa_agg AS (
        SELECT RunId, PayDetailRunId, PayClaimDetailId,
               SUM(BaseAmount) AS PPABase, SUM(PaidAmount) AS PPAPay
        FROM vw_MemberPPA_Void
        GROUP BY RunId, PayDetailRunId, PayClaimDetailId
    ),
    ppa_lines AS (
        SELECT
            ppa.RunId,
            ppa.PayDetailRunId AS DetailRunId,
            ppa.PayClaimDetailId AS ClaimDetailId,
            tcl.ClaimLineDetailId,
            ppa.PPABase, ppa.PPAPay,
            claims.BaseMemberPaidAmount AS TotalPaidBase,
            claims.PayMemberPaidAmount  AS TotalPaidPay,
            tcl.BaseMemberPaidAmount AS LinePaidBase,
            COALESCE(tcl.PayCurrencyAmountToMember, 0) AS LinePaidPay,
            tcl.LineNumber,
            SUM(tcl.BaseMemberPaidAmount) OVER w AS CumPaidBase,
            SUM(COALESCE(tcl.PayCurrencyAmountToMember, 0)) OVER w AS CumPaidPay,
            COALESCE(SUM(tcl.BaseMemberPaidAmount) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidBase,
            COALESCE(SUM(COALESCE(tcl.PayCurrencyAmountToMember, 0)) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
                ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0) AS PrevCumPaidPay,
            MAX(tcl.LineNumber) OVER (
                PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ) AS MaxLineNumber
        FROM ppa_agg ppa
        INNER JOIN vw_ClaimsBaseCurrencyId claims ON claims.ClaimDetailId = ppa.PayClaimDetailId
        INNER JOIN vw_BaseClaimsLines tcl       ON tcl.ClaimDetailId    = ppa.PayClaimDetailId
        WHERE ppa.PPABase > 0 OR ppa.PPAPay > 0
        WINDOW w AS (
            PARTITION BY ppa.RunId, ppa.PayDetailRunId, ppa.PayClaimDetailId
            ORDER BY tcl.LineNumber ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
    )
    SELECT RunId, DetailRunId, ClaimDetailId, ClaimLineDetailId,
        CAST(-LinePaidBase + (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN PPABase - LEAST(LEAST(PPABase, TotalPaidBase), PrevCumPaidBase)
                 ELSE LEAST(LEAST(PPABase, TotalPaidBase), CumPaidBase)
                      - LEAST(LEAST(PPABase, TotalPaidBase), PrevCumPaidBase)
            END
        ) AS DOUBLE) AS adj_base,
        CAST(-LinePaidPay + (
            CASE WHEN LineNumber = MaxLineNumber
                 THEN PPAPay - LEAST(LEAST(PPAPay, TotalPaidPay), PrevCumPaidPay)
                 ELSE LEAST(LEAST(PPAPay, TotalPaidPay), CumPaidPay)
                      - LEAST(LEAST(PPAPay, TotalPaidPay), PrevCumPaidPay)
            END
        ) AS DOUBLE) AS adj_pay,
        CAST(-LinePaidBase AS DOUBLE) AS paid_base,
        CAST(-LinePaidPay  AS DOUBLE) AS paid_pay,
        LineNumber
    FROM ppa_lines
""")
print("[OK] vw_MemberPPAVoidAdjustment created")
_df_mppa_v_adj = cache_temp_view("vw_MemberPPAVoidAdjustment")


In [ ]:
# ============================================================
# Cell 20c: Materializar vistas pesadas como Delta tables
# ------------------------------------------------------------
# Motivo: Cell 21 (staging SELECT) hace ~20 LEFT JOINs sobre
# temp views complejas (window funcs + agregaciones). En pool
# Medium 1-2 nodos el DAG colapsa con LIVY_JOB_STATE_DEAD (OOM).
#
# Solucion: persistir cada vista pesada como Delta table en
# Silver. Esto:
#   1) Corta el DAG en stages independientes (libera memoria)
#   2) Habilita stats Delta -> el optimizador hace BHJ con dims
#   3) Idempotente: overwrite cada run
#   4) Re-registra la temp view apuntando a la tabla fisica
#
# Vistas materializadas:
#   vw_BaseClaimsLines           -> Silver.tmp_FCP_BaseClaimsLines
#   vw_ClaimRun                  -> Silver.tmp_FCP_ClaimRun
#   vw_ProviderPPAAdjustment     -> Silver.tmp_FCP_ProviderPPAAdj
#   vw_ProviderPPAVoidAdjustment -> Silver.tmp_FCP_ProviderPPAVoidAdj
#   vw_MemberPPAAdjustment       -> Silver.tmp_FCP_MemberPPAAdj
#   vw_MemberPPAVoidAdjustment   -> Silver.tmp_FCP_MemberPPAVoidAdj
# ============================================================

# AQE + skew handling para joins desbalanceados
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Auto-broadcast hasta 10 MB para que dims chicas se broadcastee
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))


def materialize_view_as_delta(view_name: str, table_name: str, partition_cols=None):
    """
    Persiste una temp view como Delta table y re-registra la temp view
    apuntando a la tabla fisica. Idempotente (overwrite cada run).
    """
    df = spark.table(view_name)
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.saveAsTable(table_name)
    spark.table(table_name).createOrReplaceTempView(view_name)
    n = spark.table(table_name).count()
    print(f"[OK] {view_name:35s} -> {table_name:45s} {n:>12,} rows")
    return n


spark.sql("CREATE SCHEMA IF NOT EXISTS Silver")

# Orden: BaseClaimsLines primero (lo consumen los 4 PPAAdjustment).
# Al re-registrar la temp view apuntando a la tabla Delta, los PPAAdjustment
# (que son temp views lazy) leeran de Delta en su proxima evaluacion.
materialize_view_as_delta("vw_BaseClaimsLines",           "Silver.tmp_FCP_BaseClaimsLines")
materialize_view_as_delta("vw_ClaimRun",                  "Silver.tmp_FCP_ClaimRun")
materialize_view_as_delta("vw_ProviderPPAAdjustment",     "Silver.tmp_FCP_ProviderPPAAdj")
materialize_view_as_delta("vw_ProviderPPAVoidAdjustment", "Silver.tmp_FCP_ProviderPPAVoidAdj")
materialize_view_as_delta("vw_MemberPPAAdjustment",       "Silver.tmp_FCP_MemberPPAAdj")
materialize_view_as_delta("vw_MemberPPAVoidAdjustment",   "Silver.tmp_FCP_MemberPPAVoidAdj")

# Stats para CBO / broadcast decisions
for t in [
    "Silver.tmp_FCP_BaseClaimsLines",
    "Silver.tmp_FCP_ClaimRun",
    "Silver.tmp_FCP_ProviderPPAAdj",
    "Silver.tmp_FCP_ProviderPPAVoidAdj",
    "Silver.tmp_FCP_MemberPPAAdj",
    "Silver.tmp_FCP_MemberPPAVoidAdj",
]:
    try:
        spark.sql(f"ANALYZE TABLE {t} COMPUTE STATISTICS")
    except Exception as _e:
        print(f"[WARN] ANALYZE {t} skipped: {_e}")

print("\n[OK] Materializacion completada. Cell 21 ahora lee de Delta tables.")


In [ ]:
# ============================================================
# Cell 21: Create staging table - Silver.TmpFactClaimPayment
# OPTIMIZACIÃƒâ€œN: FusiÃƒÂ³n de RUN MEMBER + RUN PROVIDER en UNA sola query
# Usa CASE WHEN TMPcr.IsProvider para diferenciar, eliminando el UNION ALL
# que duplicaba 20+ JOINs con doble escaneo completo de tablas base.
# ============================================================
df_staging = spark.sql("""
    SELECT
        cd.HeaderId AS ClaimHeaderId,
        cd.HeaderId AS ClaimNumber,
        COALESCE(cd.RefNo, '_Undefined') AS ClaimReferenceNumber,
        COALESCE(cd.ClaimDetailId, -101) AS ClaimDetailId,
        DENSE_RANK() OVER (PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId ASC) AS ClaimVersion,
        DENSE_RANK() OVER (PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId DESC) AS ControlVersion,
        vld.ClaimLineDetailId,
        DENSE_RANK() OVER (PARTITION BY cd.ClaimDetailId ORDER BY vld.ClaimLineDetailId ASC) AS ClaimLineSequenceNumber,
        vld.ReferralHeaderId AS ClamReferralHeaderId,
        COALESCE(cd.UUID, '_Undefined') AS ClaimUUID,
        COALESCE(cl.StatusId, -101) AS ClaimStatusId,
        COALESCE(cl.ReasonId, -101) AS ClaimReasonId,
        COALESCE(CAST(cd.IssuedDate AS DATE), DATE '1900-01-01') AS ClaimStatusReasonDate,
        COALESCE(cd.MemberEligibilityId, -101) AS ClaimMemberEligibilityId,
        COALESCE(cd.BillingProviderId, -101) AS ClaimBillingProviderId,
        'Billing' AS ClaimBillingProviderCategory,
        COALESCE(vld.ServiceProviderId, -101) AS ClaimServiceProviderId,
        'Service' AS ClaimServiceProviderCategory,
        COALESCE(cd.ScheduleId, -1) AS ClaimScheduleId,
        COALESCE(cd.ServiceIndicatorId, -101) AS ClaimDetailServiceIndicatorId,
        COALESCE(vld.ServiceIndicatorId, -101) AS ClaimLineDetailServiceIndicatorId,
        CAST(vld.FromDate AS DATE) AS ClaimServiceFromDate,
        CAST(vld.ToDate AS DATE) AS ClaimServiceToDate,
        COALESCE(vld.LocalCurrencyId, -101) AS ClaimLocalCurrencyId,
        vld.XChangeRate AS ClaimLocalXChangeRate,
        vld.XchangeDate AS ClaimLocalXChangeDate,
        COALESCE(vld.BaseCurrencyId, -101) AS ClaimBaseCurrencyId,
        COALESCE(CAST(cd.ReceivedDate AS DATE), DATE '1900-01-01') AS ClaimReceivedDate,
        -101 AS ClaimGlobalLocationId,
        COALESCE(ProvAddr.CountryId, -101) AS ClaimCountryId,
        -101 AS ClaimStateId,
        COALESCE(ProvAddr.CityId, -101) AS ClaimCityId,
        COALESCE(cd.AccumulatorScopeId, -1) AS ClaimAccumulatorScopeId,
        COALESCE(cd.BillTypeId, -1) AS ClaimBillTypeId,
        -- Ã¢â€â‚¬Ã¢â€â‚¬ Condicional Member vs Provider Ã¢â€â‚¬Ã¢â€â‚¬
        CASE WHEN TMPcr.IsProvider = 0 THEN 1008 ELSE 1007 END AS ClaimPayToId,
        COALESCE(Payee.PayeeId, -101) AS ClaimPayeeId,
        COALESCE(cd.PolicyId, -101) AS PolicyId,
        -- Ã¢â€â‚¬Ã¢â€â‚¬ Policy Dimension Ã¢â€â‚¬Ã¢â€â‚¬
        COALESCE(pp.ModeOfPaymentId, -101) AS PolicyModeOfPaymentId,
        COALESCE(pp.PaymentMethodId, -1) AS PolicyPaymentMethodId,
        COALESCE(pp.ReceivedMethodId, -101) AS PolicyReceivedMethodId,
        COALESCE(pp.GroupId, -101) AS PolicyGroupId,
        COALESCE(pp.FamilyTypeId, -101) AS PolicyFamilyTypeId,
        COALESCE(pp.ProductId, -101) AS PolicyProductId,
        COALESCE(pp.EntityId, -101) AS PolicyEntityId,
        COALESCE(pp.PlanId, -101) AS PolicyPlanId,
        COALESCE(pp.BusinessModeId, -101) AS PolicyBusinessModeId,
        COALESCE(pp.BusinessTypeId, -101) AS PolicyBusinessTypeId,
        COALESCE(pp.InsuranceBusinessId, -101) AS PolicyInsuranceBusinessId,
        COALESCE(pp.CompanyId, -101) AS PolicyCompanyId,
        pp.PolicyIssueDate,
        pp.PolicyRenewalDate,
        pp.PolicyAnniversaryDate,
        pp.PolicyApplicationReceivedDate,
        pp.PolicyEffectiveDate,
        pp.PolicyCancelDate,
        pp.PolicyDeathBenefitPeriodDate,
        COALESCE(pp.AgentHierarchyId, -101) AS PolicyAgentHierarchyId,
        COALESCE(pp.RegionId, -101) AS PolicyRegionId,
        COALESCE(pp.MemberOwnerId, -101) AS PolicyMemberOwnerId,
        pp.PolicyGlobalLocationId,
        pp.PolicyCountryId,
        pp.PolicyStateId,
        pp.PolicyCityId,
        -- Run info
        COALESCE(TMPcr.RunId, -101) AS ClaimRunId,
        TMPcr.DetailRunId AS ClaimDetailRunId,
        TMPcr.InsuranceBusinessId AS ClaimRunInsuranceBusinessId,
        COALESCE(TMPcr.RunTypeId, -101) AS ClaimRunTypeId,
        CAST(TMPcr.CreatedOn AS DATE) AS ClaimRunCreatedOnDate,
        CAST(TMPcr.PrintedOn AS DATE) AS ClaimRunPrintedOnDate,
        COALESCE(TMPcr.Status, -101) AS ClaimRunStatusId,
        COALESCE(TMPcr.PaymentMethodId, -101) AS ClaimRunPaymentMethodId,
        cdbl.DiagnosticId AS ClaimPrimaryDiagnosticId,
        cpbl.ProcedureId AS ClaimPrimaryProcedureId,
        cd.DiagnosticType AS ClaimDiagnosticType,
        vld.POSId AS ClaimPOSId,
        vld.TOSId AS ClaimTOSId,
        -- Payment info
        COALESCE(cp.PaymentId, -1) AS PaymentId,
        COALESCE(cp.PaymentNumber, -1) AS PaymentNumber,
        COALESCE(CAST(cp.PrintedDate AS DATE), DATE '1900-01-01') AS PaymentPrintedDate,
        COALESCE(cp.StatusId, -101) AS PaymentStatusId,
        COALESCE(PaymentVoid.ReasonId, -101) AS PaymentVoidReasonId,
        COALESCE(CAST(TMPcr.postedon AS DATE), DATE '1900-01-01') AS PaymentPostedDate,
        COALESCE(CAST(PaymentVoid.PaymentLogDate AS DATE), DATE '1900-01-01') AS PaymentVoidDate,
        -- Payment Details
        cp.BankId AS PaymentBankId,
        cp.GatewayTransactionId AS PaymentGatewayTransactionId,
        cp.Amount AS PaymentAmount,
        cp.ForeignAmount AS PaymentForeignAmount,
        cp.BankTransmissionId AS PaymentBankTransmissionId,
        TMPcr.JournalEntryId,
        TMPcr.Notes AS JournalNotes,
        PaymentVoid.JournalEntryId AS VoidJournalEntryId,
        jeVoid.Notes AS VoidJournalNotes,
        -- Ã¢â€â‚¬Ã¢â€â‚¬ Currency: condicional Member vs Provider Ã¢â€â‚¬Ã¢â€â‚¬
        CASE WHEN TMPcr.IsProvider = 0 THEN vld.PayToMemberCurrencyId ELSE -101 END AS ClaimPaymentCurrencyToMemberId,
        CASE WHEN TMPcr.IsProvider = 0 THEN vld.PayToMemberXChangeRate ELSE CAST(0 AS DOUBLE) END AS ClaimPaymentToMemberXchangeRate,
        CASE WHEN TMPcr.IsProvider = 1 THEN vld.PayToProviderCurrencyId ELSE -101 END AS ClaimPaymentCurrencyToProviderId,
        CASE WHEN TMPcr.IsProvider = 1 THEN vld.PayToProviderXChangeRate ELSE CAST(0 AS DOUBLE) END AS ClaimPaymentToProviderXchangeRate,
        -- ThirdParty / Account
        COALESCE(tp.ThirdPartyId, -101) AS ClaimPaymentThirdPartyId,
        COALESCE(NULLIF(cd.AccountNo, ''), '_Undefined') AS ClaimAccountNumber,
        -- Ã¢â€â‚¬Ã¢â€â‚¬ Amounts: condicional Member vs Provider Ã¢â€â‚¬Ã¢â€â‚¬
        TMPcr.Factor * vld.LocalBilledAmount AS LocalBilledAmount,
        TMPcr.Factor * vld.BaseAllowedAmount AS BaseAllowedAmount,
        TMPcr.Factor * vld.BaseBilledAmount AS BaseBilledAmount,
        TMPcr.Factor * vld.BaseCoInsuranceAmount AS BaseCoInsuranceAmount,
        TMPcr.Factor * CAST(0 AS DOUBLE) AS BaseAllowedCmp,
        TMPcr.Factor * vld.BaseCopayAmount AS BaseCopayAmount,
        TMPcr.Factor * vld.BaseDeductibleAmount AS BaseDeductibleAmount,
        TMPcr.Factor * vld.BaseNetCoveredAmount AS BaseNetCoveredAmount,
        CASE WHEN TMPcr.IsProvider = 0 THEN TMPcr.Factor * vld.BaseMemberPaidAmount ELSE CAST(0 AS DOUBLE) END AS BaseMemberPaidAmount,
        CASE WHEN TMPcr.IsProvider = 1 THEN TMPcr.Factor * vld.BaseProviderPaidAmount ELSE CAST(0 AS DOUBLE) END AS BaseProviderPaidAmount,
        TMPcr.Factor * vld.BaseHB_DiscountAmount AS BaseDiscountAmount,
        TMPcr.Factor * vld.BaseTotalIneligibleAmount AS BaseTotalIneligibleAmount,
        TMPcr.Factor * vld.BaseOtherIneligibleAmount AS BaseOtherIneligibleAmount,
        TMPcr.Factor * vld.BaseTotalPaidAmount AS BaseTotalPaidAmount,
        TMPcr.Factor * vld.BaseWithHolding AS BaseWithHoldingAmount,
        TMPcr.Factor * vld.BaseVAT AS BaseVATAmount,
        CASE WHEN TMPcr.IsProvider = 0 THEN TMPcr.Factor * vld.PayCurrencyAmountToMember ELSE CAST(0 AS DOUBLE) END AS PayMemberPaidAmount,
        CASE WHEN TMPcr.IsProvider = 1 THEN TMPcr.Factor * vld.PayCurrencyAmountToProvider ELSE CAST(0 AS DOUBLE) END AS PayProviderPaidAmount,
        vld.BaseXChangeRate,
        -- Ã¢â€â‚¬Ã¢â€â‚¬ USD Amounts (comunes) Ã¢â€â‚¬Ã¢â€â‚¬
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseAllowedAmount ELSE vld.BaseAllowedAmount * vld.BaseXChangeRate END AS USD_AllowedAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseBilledAmount ELSE vld.BaseBilledAmount * vld.BaseXChangeRate END AS USD_BilledAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCoInsuranceAmount ELSE vld.BaseCoInsuranceAmount * vld.BaseXChangeRate END AS USD_CoInsuranceAmount,
        TMPcr.Factor * CAST(0 AS DOUBLE) AS USD_AllowedCmp,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCopayAmount ELSE vld.BaseCopayAmount * vld.BaseXChangeRate END AS USD_CopayAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseDeductibleAmount ELSE vld.BaseDeductibleAmount * vld.BaseXChangeRate END AS USD_DeductibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseNetCoveredAmount ELSE vld.BaseNetCoveredAmount * vld.BaseXChangeRate END AS USD_NetCoveredAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseHB_DiscountAmount ELSE vld.BaseHB_DiscountAmount * vld.BaseXChangeRate END AS USD_DiscountAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseTotalIneligibleAmount ELSE vld.BaseTotalIneligibleAmount * vld.BaseXChangeRate END AS USD_TotalIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseOtherIneligibleAmount ELSE vld.BaseOtherIneligibleAmount * vld.BaseXChangeRate END AS USD_OtherIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseWithHolding ELSE vld.BaseWithHolding * vld.BaseXChangeRate END AS USD_WithHoldingAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseVAT ELSE vld.BaseVAT * vld.BaseXChangeRate END AS USD_VATAmount,
        -- Ã¢â€â‚¬Ã¢â€â‚¬ USD Member/Provider Paid: condicional Ã¢â€â‚¬Ã¢â€â‚¬
        CASE WHEN TMPcr.IsProvider = 0 THEN
            TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseMemberPaidAmount
                ELSE CASE WHEN vld.PayToMemberCurrencyId = 192 THEN vld.PayCurrencyAmountToMember
                    ELSE CAST(NULL AS DOUBLE) END
            END
        ELSE CAST(0 AS DOUBLE)
        END AS USD_MemberPaidAmount,
        CASE WHEN TMPcr.IsProvider = 1 THEN
            TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseProviderPaidAmount
                ELSE CASE WHEN vld.PayToProviderCurrencyId = 192 THEN vld.PayCurrencyAmountToProvider
                    ELSE CAST(NULL AS DOUBLE) END
            END
        ELSE CAST(0 AS DOUBLE)
        END AS USD_ProviderPaidAmount,
        CASE WHEN TMPcr.IsProvider = 0 THEN
            TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseTotalPaidAmount
                ELSE CASE WHEN vld.PayToMemberCurrencyId = 192 THEN vld.PayCurrencyAmountToMember
                    ELSE CAST(NULL AS DOUBLE) END
            END
        ELSE
            TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseTotalPaidAmount
                ELSE CASE WHEN vld.PayToProviderCurrencyId = 192 THEN vld.PayCurrencyAmountToProvider
                    ELSE CAST(NULL AS DOUBLE) END
            END
        END AS USD_TotalPaidAmount,
        TMPcr.Factor,
        -- Claim processing
        procDate.FromDate AS ClaimProcessedDate,
        CASE WHEN procDate.StatusId = 1017 AND procDate.ReasonId IN (1094, 1095, 1096, 1773, 1827, 1828, 1829, 1933) THEN 1 ELSE 0 END AS ClaimProcessRowFlag,
        TMPcr.Processed AS RunProcessed,
        1 AS ActualRowFlag,
        current_timestamp() AS LoadDate,
        -- PaidSource: condicional
        CASE
            WHEN TMPcr.IsProvider = 0 AND PaymentVoid.PaymentLogDate IS NULL THEN 'PAID MEMBER'
            WHEN TMPcr.IsProvider = 0 AND PaymentVoid.PaymentLogDate IS NOT NULL THEN 'VOID MEMBER'
            WHEN TMPcr.IsProvider = 1 AND PaymentVoid.PaymentLogDate IS NULL THEN 'PAID PROVIDER'
            ELSE 'VOID PROVIDER'
        END AS PaidSource,
        -- Ã¢â€â‚¬Ã¢â€â‚¬ FIX: PAID/VOID regular Ã¢â‚¬â€ NO aplica PPA aqui (PPA va en INSERTs separados).
        --      Adjusted = Paid (en SQL Server, regular PAID/VOID tiene Adj == Paid).
        -- BaseClaimProviderPaymentAdjustedAmount = LinePaidBase
        CASE WHEN TMPcr.IsProvider = 1
             THEN TMPcr.Factor * CAST(vld.BaseProviderPaidAmount AS DOUBLE)
             ELSE CAST(0 AS DOUBLE)
        END AS BaseClaimProviderPaymentAdjustedAmount,
        -- PayClaimProviderPaymentAdjustedAmount = LinePaidPay
        CASE WHEN TMPcr.IsProvider = 1
             THEN TMPcr.Factor * CAST(vld.PayCurrencyAmountToProvider AS DOUBLE)
             ELSE CAST(0 AS DOUBLE)
        END AS PayClaimProviderPaymentAdjustedAmount,
        -- USD_ClaimProviderPaymentAdjustedAmount = LinePaid en USD (igual que USD_ProviderPaidAmount)
        CASE WHEN TMPcr.IsProvider = 1 THEN
            TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseProviderPaidAmount
                ELSE CASE WHEN vld.PayToProviderCurrencyId = 192 THEN vld.PayCurrencyAmountToProvider
                    ELSE CAST(NULL AS DOUBLE) END
            END
        ELSE CAST(0 AS DOUBLE)
        END AS USD_ClaimProviderPaymentAdjustedAmount,
        CASE
            WHEN TMPcr.IsProvider = 0 THEN
                CASE WHEN vld.PayToMemberCurrencyId = 192 THEN CAST(1 AS DOUBLE)
                     WHEN vld.BaseCurrencyId = 192 THEN CAST(1.0 / CASE WHEN vld.PayToMemberXChangeRate = 0 THEN 1 ELSE vld.PayToMemberXChangeRate END AS DOUBLE)
                     ELSE CAST(NULL AS DOUBLE)
                END
            ELSE
                CASE WHEN vld.PayToProviderCurrencyId = 192 THEN CAST(1 AS DOUBLE)
                     WHEN vld.BaseCurrencyId = 192 THEN CAST(1.0 / CASE WHEN vld.PayToProviderXChangeRate = 0 THEN 1 ELSE vld.PayToProviderXChangeRate END AS DOUBLE)
                     ELSE CAST(NULL AS DOUBLE)
                END
        END AS PayCurrencyToUSDXChangeRate,
        -- BaseClaimMemberPaymentAdjustedAmount
        -- BaseClaimMemberPaymentAdjustedAmount = LinePaidBase
        CASE WHEN TMPcr.IsProvider = 0
             THEN TMPcr.Factor * CAST(vld.BaseMemberPaidAmount AS DOUBLE)
             ELSE CAST(0 AS DOUBLE)
        END AS BaseClaimMemberPaymentAdjustedAmount,
        -- PayClaimMemberPaymentAdjustedAmount = LinePaidPay
        CASE WHEN TMPcr.IsProvider = 0
             THEN TMPcr.Factor * CAST(vld.PayCurrencyAmountToMember AS DOUBLE)
             ELSE CAST(0 AS DOUBLE)
        END AS PayClaimMemberPaymentAdjustedAmount,
        -- USD_ClaimMemberPaymentAdjustedAmount = LinePaid en USD
        CASE WHEN TMPcr.IsProvider = 0 THEN
            TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseMemberPaidAmount
                ELSE CASE WHEN vld.PayToMemberCurrencyId = 192 THEN vld.PayCurrencyAmountToMember
                    ELSE CAST(NULL AS DOUBLE) END
            END
        ELSE CAST(0 AS DOUBLE)
        END AS USD_ClaimMemberPaymentAdjustedAmount,
        -- Network / Flags (comunes)
        CASE
            WHEN cd.ServiceIndicatorId = 500 THEN 'GeoBlue'
            WHEN cd.ServiceIndicatorId = 432 THEN 'UHCI'
            WHEN cd.ServiceIndicatorId = 422 THEN 'Bupa'
            WHEN cd.ServiceIndicatorId = 773 THEN 'Olympus'
            WHEN IBRun.Max_InsuranceBusinessId = 58 THEN 'TBSL'
            ELSE 'Bupa'
        END AS ServiceNetwork,
        CASE
            WHEN IBRun.HeaderId IS NULL THEN 'NOT PAID'
            WHEN IBRun.min_isprovider <> IBRun.max_isprovider THEN 'SPLIT PAYMENT'
            WHEN IBRun.max_isprovider = 1 THEN 'PROVIDER PAID'
            ELSE 'MEMBER PAID'
        END AS Paid_To_Calculated,
        CASE WHEN cd.ServiceIndicatorId = 432 THEN 'INN' ELSE 'OON' END AS INN_OON,
        cd.MemberId,
        DATEDIFF(t.ServiceFromDate, t.ReceivedDate) AS ClaimGap,
        CASE WHEN cd.ServiceIndicatorId IN (500, 432, 773) THEN 5 ELSE cd.ClaimReceivedMethodId END AS ClaimReceivedMethodId,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 12) THEN 1 ELSE 0 END AS PolicyFirstYear,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 1) THEN 1 ELSE 0 END AS PolicyFirstMonth,
        0 AS FirstClaim,
        CASE WHEN rm.ReceivedMethodId IS NOT NULL THEN 1 ELSE 0 END AS IsDigital,
        COALESCE(ltTAT.IsCOR, '_Non Defined IsCOR') AS IsCOR,
        COALESCE(ltTAT.LocalTeam, '_Non Defined LocalTeam') AS LocalTeam,
        COALESCE(cd.IsFastTrack, FALSE) AS IsFastTrack,
        vld.HTHFeePaid,
        vld.HTHFeePercent,
        0 AS IsDeleted,
        COALESCE(pcn.Name, '_Not defined ProviderNetworkClassName') AS ProviderNetworkClassName,
        0 AS BrackedClaimKey,
        0 AS BrackedMemberKey,
        CAST(0 AS DOUBLE) AS IncomeByInsured,
        CAST(0 AS DOUBLE) AS SpendByInsured,
        0 AS ClaimsOutliers,
        COALESCE(ClaimSub.SenderEmailAddress, '_Not defined SenderEmailAddress') AS SubmissionSenderEmailAddress,
        COALESCE(ClaimSub.SenderFullName, '_Not defined SenderFullName') AS SubmissionSenderFullName,
        COALESCE(ClaimSub.InboundChannel, '_Not defined InboundChannel') AS SubmissionInboundChannel,
        COALESCE(ClaimSub.TrackingNumber, '_Not defined TrackingNumber') AS SubmissionTrackingNumber,
        COALESCE(r.RoleReferences, '_Not defined SenderType') AS SubmissionSenderType,
        '_Not defined' AS ClaimValidation,
        ch.BatchLotName,
        ch.BatchLotFromDate,
        ch.TransactionLotReceivedDate,
        ch.TransactionLotNumber,
        ch.TransactionInvoiceNumber,
        ch.LotNumber,
        COALESCE(ch.IsAutomatic, 0) AS IsAutomatic,
        CASE WHEN vld.ServiceIndicatorId IN (316,346,349,620,651,654,709) THEN 1 ELSE 0 END AS IsICU,
        CASE WHEN vld.ServiceIndicatorId IN (197,198,679,708,720,769) THEN 1 ELSE 0 END AS IsEmergency,
        0 AS IsHospital_Emergency,
        CASE WHEN cpbl.ProcedureId IN (7734,7736,40957,7737,41474,7738,7739,7756,41220,7757,7758) THEN 1 ELSE 0 END AS IsChildbirths,
        CASE WHEN cpbl.ProcedureId IN (7745,7746,7747,40958,7748,7749,7750,7751,41475,7752,7753,7754,7759,7760,7761) THEN 1 ELSE 0 END AS IsCesareanSection,
        -- PPA Balance columns (0 para PAID/VOID; se llenan en INSERT PPA *)
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalancePaymentCurrency,
        TMPcr.Factor * vld.BaseVAT * COALESCE(NULLIF(CASE WHEN TMPcr.IsProvider = 1 THEN vld.PayToProviderXChangeRate ELSE vld.PayToMemberXChangeRate END, 0), 1) AS VATPaymentCurrency,
        TMPcr.Factor * vld.BaseWithHolding * COALESCE(NULLIF(CASE WHEN TMPcr.IsProvider = 1 THEN vld.PayToProviderXChangeRate ELSE vld.PayToMemberXChangeRate END, 0), 1) AS WithholdingPaymentCurrency
    FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
    INNER JOIN vw_ClaimRun TMPcr ON TMPcr.ClaimDetailId = cd.ClaimDetailId
    INNER JOIN vw_BaseClaimsLines vld ON vld.ClaimDetailId = cd.ClaimDetailId
    LEFT JOIN vw_ClaimLog_Payment cl ON cd.ClaimDetailId = cl.ClaimDetailId AND cl.rn = 1
    LEFT JOIN vw_ProcessedDate_Payment procDate ON cd.ClaimDetailId = procDate.ClaimDetailId AND procDate.Cont = 1
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_Payment cp ON cp.PaymentId = TMPcr.PaymentId
    LEFT JOIN vw_PaymentLogVoid PaymentVoid ON cp.PaymentId = PaymentVoid.PaymentId AND PaymentVoid.JournalEntryId = TMPcr.JournalEntryId
    LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry jeVoid ON jeVoid.JournalEntryId = PaymentVoid.JournalEntryId
    LEFT JOIN vw_DiagnosticLine_Payment cdbl ON cdbl.ClaimLineDetailId = vld.ClaimLineDetailId AND cdbl.ID = 1
    LEFT JOIN vw_ProcedureLine_Payment cpbl ON cpbl.ClaimLineDetailId = vld.ClaimLineDetailId AND cpbl.ID = 1
    LEFT JOIN vw_InsuranceBusinessRun IBRun ON TMPcr.HeaderId = IBRun.HeaderId
    LEFT JOIN vw_TmpReceived_Payment t ON cd.HeaderId = t.HeaderId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderAddress ProvAddr ON cd.ProviderAddressId = ProvAddr.ProviderAddressId
    LEFT JOIN Bronze.AmigosPlus_AMP_Policy_Member Member ON cd.MemberId = Member.MemberId
    LEFT JOIN vw_Payee_Payment Payee ON Member.ContactBaseId = Payee.ContactBaseId AND Payee.RowNum = 1
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderNetworkClass pcn ON pcn.ProviderNetworkClassId = cd.ProviderNetworkClassId
    LEFT JOIN vw_ClaimSubmission_Payment ClaimSub ON ClaimSub.ClaimHeaderId = cd.HeaderId AND ClaimSub.PolicyId = cd.PolicyId AND ClaimSub.CONT = 1
    LEFT JOIN Bronze.Common_Roles r ON r.RoleId = ClaimSub.SenderTypeId
    LEFT JOIN vw_ReceivedMethodDigital_Payment rm ON cd.ClaimReceivedMethodId = rm.ReceivedMethodId
    LEFT JOIN vw_PolicyHistory pp ON cd.PolicyId = pp.PolicyId
    LEFT JOIN vw_ThirdPartyClaim tp ON tp.ClaimDetailId = cd.ClaimDetailId
        AND tp.PayTo = CASE WHEN TMPcr.IsProvider = 0 THEN 1008 ELSE 1007 END
    LEFT JOIN vw_LocalTeamTAT ltTAT ON cd.HeaderId = ltTAT.ClaimHeaderId
    LEFT JOIN vw_ClearingHouseLine ch ON ch.ClaimLineDetailId = vld.ClaimLineDetailId
    -- FIX: PPA Adjustment LEFT JOINs eliminados.
    -- Causaban contaminacion: matcheaban con la fila PAID/VOID original
    -- (porque vw_*PPAAdjustment expone RunId del PAGO ORIGINAL) y sobrescribian
    -- el LinePaid via COALESCE. Resultado: la fila "PAID 910" desaparecia y solo
    -- quedaba "PAID 492.44" (PPA-adjusted). Las filas PPA ahora se generan en
    -- los INSERTs separados (PPA PROVIDER / PPA MEMBER / VOID PPA *).
""")

# Escribir a Silver staging con optimizaciÃƒÂ³n Delta
df_staging.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Silver.TmpFactClaimPayment")

print(f"[OK] Silver.TmpFactClaimPayment created Ã¢â‚¬â€ {spark.table('Silver.TmpFactClaimPayment').count():,} rows")

In [ ]:
# ============================================================
# Cell: Align regular PAID PROVIDER adjusted amounts with SQL PPA adjustment
# ------------------------------------------------------------
# SQL Server keeps provider paid amounts in PAID PROVIDER and writes PPA
# balances separately as PPA PROVIDER, but the PAID PROVIDER adjusted
# columns can come from the line-level Provider PPA adjustment table.
# Default adjusted values to 0, then apply vw_ProviderPPAAdjustment when
# a matching run/detail/line exists.
# ============================================================
spark.sql("""
    UPDATE Silver.TmpFactClaimPayment
    SET
        BaseClaimProviderPaymentAdjustedAmount = CAST(0 AS DOUBLE),
        PayClaimProviderPaymentAdjustedAmount  = CAST(0 AS DOUBLE),
        USD_ClaimProviderPaymentAdjustedAmount  = CAST(0 AS DOUBLE)
    WHERE PaidSource = 'PAID PROVIDER'
      AND JournalEntryId IS NOT NULL
""")

spark.sql("""
    MERGE INTO Silver.TmpFactClaimPayment AS tgt
    USING (
        SELECT
            RunId,
            DetailRunId,
            ClaimDetailId,
            ClaimLineDetailId,
            SUM(CAST(adj_base AS DOUBLE)) AS AdjBaseAmount,
            SUM(CAST(adj_pay AS DOUBLE)) AS AdjPayAmount,
            SUM(CAST(paid_base AS DOUBLE)) AS PaidBaseAmount,
            SUM(CAST(paid_pay AS DOUBLE)) AS PaidPayAmount
        FROM vw_ProviderPPAAdjustment
        GROUP BY RunId, DetailRunId, ClaimDetailId, ClaimLineDetailId
    ) AS adj
    ON tgt.PaidSource = 'PAID PROVIDER'
   AND tgt.JournalEntryId IS NOT NULL
   AND tgt.ClaimRunId = adj.RunId
   AND tgt.ClaimDetailRunId = adj.DetailRunId
   AND tgt.ClaimDetailId = adj.ClaimDetailId
   AND tgt.ClaimLineDetailId = adj.ClaimLineDetailId
    WHEN MATCHED THEN UPDATE SET
        tgt.BaseClaimProviderPaymentAdjustedAmount = adj.AdjBaseAmount,
        tgt.PayClaimProviderPaymentAdjustedAmount  = adj.AdjPayAmount,
        tgt.USD_ClaimProviderPaymentAdjustedAmount  = CAST(COALESCE(adj.AdjPayAmount * tgt.PayCurrencyToUSDXChangeRate, adj.AdjBaseAmount) AS DOUBLE)
""")
print("[OK] Regular PAID PROVIDER adjusted amounts aligned with Provider PPA adjustment rows")

## Secciones PPA ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â usp_DataPrepare_FactClaimCostData

El SP original genera 6 tipos de `PaidSource`:
1. **PAID MEMBER / VOID MEMBER** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Incluidos en la celda anterior (fusionados con `CASE WHEN IsProvider`)
2. **PAID PROVIDER / VOID PROVIDER** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Incluidos en la celda anterior
3. **PPA PROVIDER** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Ajustes PPA Provider (ProviderBalance)
4. **PPA MEMBER** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Ajustes PPA Member (PolicyBalance)
5. **VOID PPA PROVIDER** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Anulaciones PPA Provider
6. **VOID PPA MEMBER** ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Anulaciones PPA Member

### Dependencia externa
- **`usp_DataPrepare_FactClaimPayableMulticurrency`**: El SP original ejecuta este procedimiento al inicio.  
  Crea la tabla `__TMPBGLAEDW__ClaimPayableMulticurrency` usada para montos de network providers.  
  **Fuente esperada**: `ETLStaging.dbo.__TMPBGLAEDW__ClaimPayableMulticurrency`  
  **AcciÃƒÆ’Ã‚Â³n**: Migrar como notebook separado (`nb_FactClaimPayableMulticurrency`) o como tabla Bronze/Silver.

### Nota sobre PPA Adjustment CTEs
El SP usa CTEs recursivos (`OPTION MAXRECURSION 1000`) para distribuir balances PPA por lÃƒÆ’Ã‚Â­nea de claim:
- `#PROVIDERS_PPA_TABLE_ADJUSTMENT` ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ `vw_ProviderPPAAdjustment`
- `#PROVIDERS_PPA_VOIDS_TABLE_ADJUSTMENT` ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ `vw_ProviderPPAVoidAdjustment`
- `#MEMBERS_PPA_TABLE_ADJUSTMENT` ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ `vw_MemberPPAAdjustment`
- `#MEMBERS_PPA_VOIDS_TABLE_ADJUSTMENT` ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ `vw_MemberPPAVoidAdjustment`

**ImplementaciÃƒÆ’Ã‚Â³n**: Window functions (waterfall allocation) en la celda 20b:
- `cap = LEAST(ABS(ppa_total), total_paid)` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â limita PPA al total pagado
- `alloc_i = LEAST(cap, CumPaid_i) - LEAST(cap, CumPaid_{i-1})` ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â distribuciÃƒÆ’Ã‚Â³n por lÃƒÆ’Ã‚Â­nea
- `amount_nonvoid = LinePaid - alloc` | `amount_void = -LinePaid + alloc`

Las 4 vistas de ajuste se consumen en la celda PAID MEMBER/PROVIDER vÃƒÆ’Ã‚Â­a LEFT JOIN con alias PPPA_A, VPPPA_A, MPPA_A, VMPPA_A.

In [ ]:
# ============================================================
# Cell: Carga de vw_ClaimPayableMulticurrency
# Fuente: Silver.TmpClaimPayableMulticurrency
#   (generada por nb_FactClaimPayableMulticurrency)
#
# Montos ajustados multicurrency para network providers.
# Se usa como LEFT JOIN (alias cpm) en PAID PROVIDER y
# como INNER JOIN (alias pje) en PPA PROVIDER (Network).
#
# Columnas: ClaimPayableJEId, ClaimDetailId, AmountTypeId,
#   BaseAmount, PaidAmount, BaseAmountAdjusted, PaidAmountAdjusted,
#   RecordedBaseAmountAdjusted, HeaderId, PolicyId,
#   MemberEligibilityId, MemberId
# ============================================================
try:
    df_cpm = spark.table("Silver.TmpClaimPayableMulticurrency")
    df_cpm.cache()
    df_cpm.createOrReplaceTempView("vw_ClaimPayableMulticurrency")
    cpm_count = df_cpm.count()
    print(f"[OK] vw_ClaimPayableMulticurrency created ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â {cpm_count:,} rows")
except Exception as e:
    # Fallback: crear vista vacÃƒÆ’Ã‚Â­a si nb_FactClaimPayableMulticurrency no se ha ejecutado
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW vw_ClaimPayableMulticurrency AS
        SELECT
            CAST(NULL AS BIGINT)       AS ClaimPayableJEId,
            CAST(NULL AS BIGINT)       AS ClaimDetailId,
            CAST(NULL AS BIGINT)       AS PolicyId,
            CAST(NULL AS BIGINT)       AS MemberEligibilityId,
            CAST(NULL AS BIGINT)       AS MemberId,
            CAST(NULL AS BIGINT)       AS HeaderId,
            CAST(NULL AS INT)          AS AmountTypeId,
            CAST(0 AS DOUBLE)  AS BaseAmount,
            CAST(0 AS DOUBLE)  AS PaidAmount,
            CAST(0 AS DOUBLE)  AS RecordedBaseAmountAdjusted,
            CAST(0 AS DOUBLE)  AS BaseAmountAdjusted,
            CAST(0 AS DOUBLE)  AS PaidAmountAdjusted
        WHERE 1=0
    """)
    print(f"[WARN] Silver.TmpClaimPayableMulticurrency no disponible ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â vista vacÃƒÆ’Ã‚Â­a creada")
    print(f"       Error: {e}")
    print(f"       AcciÃƒÆ’Ã‚Â³n: Ejecutar nb_FactClaimPayableMulticurrency antes de este notebook")

In [ ]:
# ============================================================
# Cell: INSERT PPA PROVIDER (aggregate, SQL-compatible)
# ------------------------------------------------------------
# SQL Server does NOT write provider PPA adjustments as line-level
# PAID PROVIDER rows. It writes them as separate PPA PROVIDER rows
# at claim-detail level (ClaimLineDetailId = -101), using the PPA
# balance itself. This prevents PPA rows from replacing regular
# PAID PROVIDER lines in the final ActualRowFlag consolidation.
# ============================================================
spark.sql("""
    INSERT INTO Silver.TmpFactClaimPayment
    WITH PPA AS (
        SELECT
            RunId,
            PayDetailRunId,
            ClaimDetailId,
            HeaderId,
            JournalEntryId,
            MAX(PolicyId) AS PolicyId,
            MAX(postedon) AS postedon,
            MAX(XChangeRate) AS XChangeRate,
            CAST(SUM(BaseAmount) AS DOUBLE) AS BaseAmount,
            CAST(SUM(PaidAmount) AS DOUBLE) AS PaidAmount
        FROM vw_ProviderPPA
        GROUP BY RunId, PayDetailRunId, ClaimDetailId, HeaderId, JournalEntryId
    ),
    VLD AS (
        SELECT
            ClaimDetailId,
            MAX(ReferralHeaderId) AS ReferralHeaderId,
            MAX(ServiceProviderId) AS ServiceProviderId,
            MAX(ServiceIndicatorId) AS ServiceIndicatorId,
            MAX(DetailServiceIndicatorId) AS DetailServiceIndicatorId,
            MIN(FromDate) AS FromDate,
            MAX(ToDate) AS ToDate,
            CAST(NULL AS DATE) AS XchangeDate,
            CAST(NULL AS DOUBLE) AS XChangeRate,
            MAX(LocalCurrencyId) AS LocalCurrencyId,
            MAX(BaseCurrencyId) AS BaseCurrencyId,
            MAX(PayToProviderCurrencyId) AS PayToProviderCurrencyId,
            CAST(NULL AS DOUBLE) AS PayToProviderXChangeRate,
            CAST(NULL AS DOUBLE) AS BaseXChangeRate,
            SUM(LocalBilledAmount) AS LocalBilledAmount,
            SUM(BaseAllowedAmount) AS BaseAllowedAmount,
            SUM(BaseBilledAmount) AS BaseBilledAmount,
            SUM(BaseCoInsuranceAmount) AS BaseCoInsuranceAmount,
            SUM(BaseCopayAmount) AS BaseCopayAmount,
            SUM(BaseDeductibleAmount) AS BaseDeductibleAmount,
            SUM(BaseNetCoveredAmount) AS BaseNetCoveredAmount,
            SUM(BaseHB_DiscountAmount) AS BaseHB_DiscountAmount,
            SUM(BaseTotalIneligibleAmount) AS BaseTotalIneligibleAmount,
            SUM(BaseOtherIneligibleAmount) AS BaseOtherIneligibleAmount,
            SUM(BaseWithHolding) AS BaseWithHolding,
            SUM(BaseVAT) AS BaseVAT,
            MAX(POSId) AS POSId,
            MAX(TOSId) AS TOSId,
            SUM(HTHFeePaid) AS HTHFeePaid,
            AVG(HTHFeePercent) AS HTHFeePercent
        FROM vw_BaseClaimsLines
        GROUP BY ClaimDetailId
    )
    SELECT
        cd.HeaderId AS ClaimHeaderId,
        cd.HeaderId AS ClaimNumber,
        COALESCE(cd.RefNo, '_Undefined') AS ClaimReferenceNumber,
        COALESCE(cd.ClaimDetailId, -101) AS ClaimDetailId,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId ASC) AS ClaimVersion,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId DESC) AS ControlVersion,
        CAST(-101 AS BIGINT) AS ClaimLineDetailId,
        CAST(1 AS INT) AS ClaimLineSequenceNumber,
        vld.ReferralHeaderId AS ClamReferralHeaderId,
        COALESCE(cd.UUID, '_Undefined') AS ClaimUUID,
        COALESCE(cl.StatusId, -101) AS ClaimStatusId,
        COALESCE(cl.ReasonId, -101) AS ClaimReasonId,
        COALESCE(CAST(cd.IssuedDate AS DATE), DATE '1899-12-31') AS ClaimStatusReasonDate,
        COALESCE(cd.MemberEligibilityId, -101) AS ClaimMemberEligibilityId,
        COALESCE(cd.BillingProviderId, -101) AS ClaimBillingProviderId,
        'Billing' AS ClaimBillingProviderCategory,
        COALESCE(vld.ServiceProviderId, -101) AS ClaimServiceProviderId,
        'Service' AS ClaimServiceProviderCategory,
        COALESCE(cd.ScheduleId, -1) AS ClaimScheduleId,
        COALESCE(cd.ServiceIndicatorId, -101) AS ClaimDetailServiceIndicatorId,
        COALESCE(vld.ServiceIndicatorId, -101) AS ClaimLineDetailServiceIndicatorId,
        vld.FromDate AS ClaimServiceFromDate,
        vld.ToDate AS ClaimServiceToDate,
        COALESCE(vld.LocalCurrencyId, -101) AS ClaimLocalCurrencyId,
        vld.XChangeRate AS ClaimLocalXChangeRate,
        vld.XchangeDate AS ClaimLocalXChangeDate,
        COALESCE(vld.BaseCurrencyId, -101) AS ClaimBaseCurrencyId,
        cd.ReceivedDate AS ClaimReceivedDate,
        -101 AS ClaimGlobalLocationId,
        COALESCE(ProviderAddress.CountryId, -101) AS ClaimCountryId,
        -101 AS ClaimStateId,
        COALESCE(ProviderAddress.CityId, -101) AS ClaimCityId,
        COALESCE(cd.AccumulatorScopeId, -1) AS ClaimAccumulatorScopeId,
        COALESCE(cd.BillTypeId, -1) AS ClaimBillTypeId,
        1007 AS ClaimPayToId,
        COALESCE(Payee.PayeeId, -101) AS ClaimPayeeId,
        COALESCE(cd.PolicyId, -101) AS PolicyId,
        COALESCE(pp.ModeOfPaymentId, -101) AS PolicyModeOfPaymentId,
        COALESCE(pp.PaymentMethodId, -1) AS PolicyPaymentMethodId,
        COALESCE(pp.ReceivedMethodId, -101) AS PolicyReceivedMethodId,
        COALESCE(pp.GroupId, -101) AS PolicyGroupId,
        COALESCE(pp.FamilyTypeId, -101) AS PolicyFamilyTypeId,
        COALESCE(pp.ProductId, -101) AS PolicyProductId,
        COALESCE(pp.EntityId, -101) AS PolicyEntityId,
        COALESCE(pp.PlanId, -101) AS PolicyPlanId,
        COALESCE(pp.BusinessModeId, -101) AS PolicyBusinessModeId,
        COALESCE(pp.BusinessTypeId, -101) AS PolicyBusinessTypeId,
        COALESCE(pp.InsuranceBusinessId, -101) AS PolicyInsuranceBusinessId,
        COALESCE(pp.CompanyId, -101) AS PolicyCompanyId,
        pp.PolicyIssueDate, pp.PolicyRenewalDate, pp.PolicyAnniversaryDate,
        pp.PolicyApplicationReceivedDate, pp.PolicyEffectiveDate, pp.PolicyCancelDate,
        pp.PolicyDeathBenefitPeriodDate,
        COALESCE(pp.AgentHierarchyId, -101) AS PolicyAgentHierarchyId,
        COALESCE(pp.RegionId, -101) AS PolicyRegionId,
        COALESCE(pp.MemberOwnerId, -101) AS PolicyMemberOwnerId,
        pp.PolicyGlobalLocationId, pp.PolicyCountryId, pp.PolicyStateId, pp.PolicyCityId,
        COALESCE(TMPcr.RunId, -101) AS ClaimRunId,
        TMPcr.DetailRunId AS ClaimDetailRunId,
        TMPcr.InsuranceBusinessId AS ClaimRunInsuranceBusinessId,
        COALESCE(TMPcr.RunTypeId, -101) AS ClaimRunTypeId,
        CAST(TMPcr.CreatedOn AS DATE) AS ClaimRunCreatedOnDate,
        CAST(TMPcr.PrintedOn AS DATE) AS ClaimRunPrintedOnDate,
        COALESCE(TMPcr.Status, -101) AS ClaimRunStatusId,
        COALESCE(TMPcr.PaymentMethodId, -101) AS ClaimRunPaymentMethodId,
        -101 AS ClaimPrimaryDiagnosticId,
        -101 AS ClaimPrimaryProcedureId,
        cd.DiagnosticType AS ClaimDiagnosticType,
        vld.POSId AS ClaimPOSId,
        vld.TOSId AS ClaimTOSId,
        COALESCE(cp.PaymentId, -1) AS PaymentId,
        COALESCE(cp.PaymentNumber, -1) AS PaymentNumber,
        COALESCE(CAST(cp.PrintedDate AS DATE), DATE '1899-12-31') AS PaymentPrintedDate,
        COALESCE(cp.StatusId, -101) AS PaymentStatusId,
        COALESCE(PaymentVoid.ReasonId, -101) AS PaymentVoidReasonId,
        COALESCE(CAST(PPA.postedon AS DATE), DATE '1899-12-31') AS PaymentPostedDate,
        COALESCE(CAST(PaymentVoid.PaymentLogDate AS DATE), DATE '1899-12-31') AS PaymentVoidDate,
        cp.BankId AS PaymentBankId,
        cp.GatewayTransactionId AS PaymentGatewayTransactionId,
        cp.Amount AS PaymentAmount,
        cp.ForeignAmount AS PaymentForeignAmount,
        cp.BankTransmissionId AS PaymentBankTransmissionId,
        TMPcr.JournalEntryId,
        TMPcr.Notes AS JournalNotes,
        PaymentVoid.JournalEntryId AS VoidJournalEntryId,
        jeVoid.Notes AS VoidJournalNotes,
        -101 AS ClaimPaymentCurrencyToMemberId,
        CAST(0 AS DOUBLE) AS ClaimPaymentToMemberXchangeRate,
        COALESCE(vld.PayToProviderCurrencyId, -101) AS ClaimPaymentCurrencyToProviderId,
        PPA.XChangeRate AS ClaimPaymentToProviderXchangeRate,
        COALESCE(tp.ThirdPartyId, -101) AS ClaimPaymentThirdPartyId,
        COALESCE(NULLIF(cd.AccountNo, ''), '_Undefined') AS ClaimAccountNumber,
        TMPcr.Factor * vld.LocalBilledAmount AS LocalBilledAmount,
        TMPcr.Factor * vld.BaseAllowedAmount AS BaseAllowedAmount,
        TMPcr.Factor * vld.BaseBilledAmount AS BaseBilledAmount,
        TMPcr.Factor * vld.BaseCoInsuranceAmount AS BaseCoInsuranceAmount,
        CAST(0 AS DOUBLE) AS BaseAllowedCmp,
        TMPcr.Factor * vld.BaseCopayAmount AS BaseCopayAmount,
        TMPcr.Factor * vld.BaseDeductibleAmount AS BaseDeductibleAmount,
        TMPcr.Factor * vld.BaseNetCoveredAmount AS BaseNetCoveredAmount,
        CAST(0 AS DOUBLE) AS BaseMemberPaidAmount,
        PPA.BaseAmount AS BaseProviderPaidAmount,
        TMPcr.Factor * vld.BaseHB_DiscountAmount AS BaseDiscountAmount,
        TMPcr.Factor * vld.BaseTotalIneligibleAmount AS BaseTotalIneligibleAmount,
        TMPcr.Factor * vld.BaseOtherIneligibleAmount AS BaseOtherIneligibleAmount,
        PPA.BaseAmount AS BaseTotalPaidAmount,
        TMPcr.Factor * vld.BaseWithHolding AS BaseWithHoldingAmount,
        TMPcr.Factor * vld.BaseVAT AS BaseVATAmount,
        CAST(0 AS DOUBLE) AS PayMemberPaidAmount,
        CASE WHEN vld.BaseCurrencyId = 192 AND vld.PayToProviderCurrencyId = 192 THEN PPA.BaseAmount ELSE PPA.PaidAmount END AS PayProviderPaidAmount,
        vld.BaseXChangeRate,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseAllowedAmount ELSE vld.BaseAllowedAmount * vld.BaseXChangeRate END AS USD_AllowedAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseBilledAmount ELSE vld.BaseBilledAmount * vld.BaseXChangeRate END AS USD_BilledAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCoInsuranceAmount ELSE vld.BaseCoInsuranceAmount * vld.BaseXChangeRate END AS USD_CoInsuranceAmount,
        CAST(0 AS DOUBLE) AS USD_AllowedCmp,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCopayAmount ELSE vld.BaseCopayAmount * vld.BaseXChangeRate END AS USD_CopayAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseDeductibleAmount ELSE vld.BaseDeductibleAmount * vld.BaseXChangeRate END AS USD_DeductibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseNetCoveredAmount ELSE vld.BaseNetCoveredAmount * vld.BaseXChangeRate END AS USD_NetCoveredAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseHB_DiscountAmount ELSE vld.BaseHB_DiscountAmount * vld.BaseXChangeRate END AS USD_DiscountAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseTotalIneligibleAmount ELSE vld.BaseTotalIneligibleAmount * vld.BaseXChangeRate END AS USD_TotalIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseOtherIneligibleAmount ELSE vld.BaseOtherIneligibleAmount * vld.BaseXChangeRate END AS USD_OtherIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseWithHolding ELSE vld.BaseWithHolding * vld.BaseXChangeRate END AS USD_WithHoldingAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseVAT ELSE vld.BaseVAT * vld.BaseXChangeRate END AS USD_VATAmount,
        CAST(0 AS DOUBLE) AS USD_MemberPaidAmount,
        CASE WHEN vld.PayToProviderCurrencyId = 192 THEN CASE WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount ELSE PPA.PaidAmount END
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE PPA.PaidAmount * NULLIF(PPA.XChangeRate, 0) END AS USD_ProviderPaidAmount,
        CASE WHEN vld.PayToProviderCurrencyId = 192 THEN CASE WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount ELSE PPA.PaidAmount END
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE PPA.PaidAmount * NULLIF(PPA.XChangeRate, 0) END AS USD_TotalPaidAmount,
        TMPcr.Factor,
        cl.FromDate AS ClaimProcessedDate,
        CASE WHEN cl.StatusId = 1017 AND cl.ReasonId IN (1094, 1095, 1096, 1773, 1827, 1828, 1829, 1933) THEN 1 ELSE 0 END AS ClaimProcessRowFlag,
        TMPcr.Processed AS RunProcessed,
        1 AS ActualRowFlag,
        current_timestamp() AS LoadDate,
        'PPA PROVIDER' AS PaidSource,
        CAST(0 AS DOUBLE) AS BaseClaimProviderPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS PayClaimProviderPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS USD_ClaimProviderPaymentAdjustedAmount,
        CASE WHEN vld.PayToProviderCurrencyId = 192 THEN 1
             WHEN vld.BaseCurrencyId = 192 THEN CAST(1 / CASE WHEN PPA.XChangeRate = 0 THEN 1 ELSE PPA.XChangeRate END AS DOUBLE)
             ELSE NULL END AS PayCurrencyToUSDXChangeRate,
        CAST(0 AS DOUBLE) AS BaseClaimMemberPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS PayClaimMemberPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS USD_ClaimMemberPaymentAdjustedAmount,
        CASE WHEN cd.ServiceIndicatorId = 500 THEN 'GeoBlue' WHEN cd.ServiceIndicatorId = 432 THEN 'UHCI' WHEN cd.ServiceIndicatorId = 422 THEN 'Bupa' WHEN cd.ServiceIndicatorId = 773 THEN 'Olympus' WHEN IBRun.Max_InsuranceBusinessId = 58 THEN 'TBSL' ELSE 'Bupa' END AS ServiceNetwork,
        CASE WHEN IBRun.HeaderId IS NULL THEN 'NOT PAID' WHEN IBRun.min_isprovider <> IBRun.max_isprovider THEN 'SPLIT PAYMENT' WHEN IBRun.max_isprovider = 1 THEN 'PROVIDER PAID' ELSE 'MEMBER PAID' END AS Paid_To_Calculated,
        CASE WHEN cd.ServiceIndicatorId = 432 THEN 'INN' ELSE 'OON' END AS INN_OON,
        cd.MemberId,
        DATEDIFF(t.ServiceFromDate, t.ReceivedDate) AS ClaimGap,
        CASE WHEN cd.ServiceIndicatorId IN (500, 432, 773) THEN 5 ELSE cd.ClaimReceivedMethodId END AS ClaimReceivedMethodId,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 12) THEN 1 ELSE 0 END AS PolicyFirstYear,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 1) THEN 1 ELSE 0 END AS PolicyFirstMonth,
        0 AS FirstClaim,
        0 AS IsDigital,
        COALESCE(ltTAT.IsCOR, '_Non Defined IsCOR') AS IsCOR,
        COALESCE(ltTAT.LocalTeam, '_Non Defined LocalTeam') AS LocalTeam,
        cd.IsFastTrack,
        vld.HTHFeePaid,
        vld.HTHFeePercent,
        0 AS IsDeleted,
        COALESCE(pcn.Name, '_Not defined ProviderNetworkClassName') AS ProviderNetworkClassName,
        0 AS BrackedClaimKey,
        0 AS BrackedMemberKey,
        CAST(0 AS DOUBLE) AS IncomeByInsured,
        CAST(0 AS DOUBLE) AS SpendByInsured,
        0 AS ClaimsOutliers,
        COALESCE(ClaimSub.SenderEmailAddress, '_Not defined SenderEmailAddress') AS SubmissionSenderEmailAddress,
        COALESCE(ClaimSub.SenderFullName, '_Not defined SenderFullName') AS SubmissionSenderFullName,
        COALESCE(ClaimSub.InboundChannel, '_Not defined InboundChannel') AS SubmissionInboundChannel,
        COALESCE(ClaimSub.TrackingNumber, '_Not defined TrackingNumber') AS SubmissionTrackingNumber,
        COALESCE(r.RoleReferences, '_Not defined SenderType') AS SubmissionSenderType,
        '_Not defined' AS ClaimValidation,
        ch.BatchLotName,
        ch.BatchLotFromDate,
        ch.TransactionLotReceivedDate,
        ch.TransactionLotNumber,
        ch.TransactionInvoiceNumber,
        ch.LotNumber,
        COALESCE(ch.IsAutomatic, 0) AS IsAutomatic,
        CAST(NULL AS INT) AS IsICU,
        CAST(NULL AS INT) AS IsEmergency,
        CAST(NULL AS INT) AS IsHospital_Emergency,
        CAST(NULL AS INT) AS IsChildbirths,
        CAST(NULL AS INT) AS IsCesareanSection,
        PPA.BaseAmount AS ClaimProviderPPABalanceBaseCurrenty,
        PPA.PaidAmount AS ClaimProviderPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalancePaymentCurrency,
        TMPcr.Factor * vld.BaseVAT * COALESCE(NULLIF(PPA.XChangeRate, 0), 1) AS VATPaymentCurrency,
        TMPcr.Factor * vld.BaseWithHolding * COALESCE(NULLIF(PPA.XChangeRate, 0), 1) AS WithholdingPaymentCurrency
    FROM PPA
    INNER JOIN vw_ClaimRun TMPcr ON TMPcr.RunId = PPA.RunId
        AND TMPcr.DetailRunId = PPA.PayDetailRunId
        AND TMPcr.JournalEntryId = PPA.JournalEntryId
    INNER JOIN VLD vld ON vld.ClaimDetailId = PPA.ClaimDetailId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON cd.ClaimDetailId = PPA.ClaimDetailId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderAddress ProviderAddress ON cd.ProviderAddressId = ProviderAddress.ProviderAddressId
    LEFT JOIN vw_ClaimLog_Payment cl ON PPA.ClaimDetailId = cl.ClaimDetailId AND cl.rn = 1
    LEFT JOIN vw_PolicyHistory pp ON pp.PolicyId = cd.PolicyId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_Payment cp ON cp.PaymentId = TMPcr.PaymentId
    LEFT JOIN vw_ThirdPartyClaim tp ON tp.ClaimDetailId = cd.ClaimDetailId AND tp.PayTo = 1007
    LEFT JOIN Bronze.AmigosPlus_AMP_Policy_Member Member ON cd.MemberId = Member.MemberId
    LEFT JOIN vw_Payee_Payment Payee ON Member.ContactBaseId = Payee.ContactBaseId AND Payee.RowNum = 1
    LEFT JOIN vw_PaymentLogVoid PaymentVoid ON cp.PaymentId = PaymentVoid.PaymentId AND PaymentVoid.JournalEntryId = TMPcr.JournalEntryId
    LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry jeVoid ON jeVoid.JournalEntryId = PaymentVoid.JournalEntryId
    LEFT JOIN vw_InsuranceBusinessRun IBRun ON TMPcr.HeaderId = IBRun.HeaderId
    LEFT JOIN vw_TmpReceived_Payment t ON cd.HeaderId = t.HeaderId
    LEFT JOIN vw_LocalTeamTAT ltTAT ON cd.HeaderId = ltTAT.ClaimHeaderId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderNetworkClass pcn ON pcn.ProviderNetworkClassId = cd.ProviderNetworkClassId
    LEFT JOIN vw_ClaimSubmission_Payment ClaimSub ON ClaimSub.ClaimHeaderId = cd.HeaderId AND ClaimSub.PolicyId = cd.PolicyId AND ClaimSub.CONT = 1
    LEFT JOIN Bronze.Common_Roles r ON r.RoleId = ClaimSub.SenderTypeId
    LEFT JOIN (SELECT DISTINCT BatchLotName, BatchLotFromDate, TransactionLotReceivedDate, TransactionLotNumber, TransactionInvoiceNumber, LotNumber, IsAutomatic, ClaimDetailId FROM vw_ClearingHouseLine) ch ON ch.ClaimDetailId = cd.ClaimDetailId
""")
print("[OK] PPA PROVIDER aggregate rows inserted into Silver.TmpFactClaimPayment")

In [ ]:
# ============================================================
# Cell: INSERT PAID MEMBER (PPA-adjusted, line-level) -- Bug #3
# Emite UNA fila por (ClaimLineDetailId) con la cantidad post-PPA
# (adj_base / adj_pay) y PaidSource = 'PAID MEMBER'.
# Solo se emiten lineas que recibieron distribucion PPA (adj <> paid).
# ============================================================
spark.sql("""
    INSERT INTO Silver.TmpFactClaimPayment
    WITH PPA AS (
        SELECT
            adj.RunId,
            adj.DetailRunId AS PayDetailRunId,
            adj.ClaimDetailId,
            adj.ClaimLineDetailId,
            adj.LineNumber,
            CAST(adj.adj_base AS DOUBLE) AS BaseAmount,
            CAST(adj.adj_pay  AS DOUBLE) AS PaidAmount,
            ppa.HeaderId,
            ppa.JournalEntryId,
            ppa.PolicyId,
            ppa.postedon,
            ppa.XChangeRate
        FROM vw_MemberPPAAdjustment adj
        INNER JOIN (
            SELECT RunId, PayDetailRunId, PayClaimDetailId,
                   MAX(HeaderId)       AS HeaderId,
                   MAX(JournalEntryId) AS JournalEntryId,
                   MAX(PolicyId)       AS PolicyId,
                   MAX(postedon)       AS postedon,
                   MAX(XChangeRate)    AS XChangeRate
            FROM vw_MemberPPA
            GROUP BY RunId, PayDetailRunId, PayClaimDetailId
        ) ppa
          ON ppa.RunId            = adj.RunId
         AND ppa.PayDetailRunId   = adj.DetailRunId
         AND ppa.PayClaimDetailId = adj.ClaimDetailId
        WHERE adj.adj_base <> adj.paid_base OR adj.adj_pay <> adj.paid_pay
    )
    SELECT
        cd.HeaderId AS ClaimHeaderId,
        cd.HeaderId AS ClaimNumber,
        COALESCE(cd.RefNo, '_Undefined') AS ClaimReferenceNumber,
        COALESCE(cd.ClaimDetailId, -101) AS ClaimDetailId,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId ASC) AS ClaimVersion,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId DESC) AS ControlVersion,
        PPA.ClaimLineDetailId,
        DENSE_RANK() OVER(PARTITION BY cd.ClaimDetailId ORDER BY PPA.ClaimLineDetailId ASC) AS ClaimLineSequenceNumber,
        vld.ReferralHeaderId AS ClamReferralHeaderId,
        COALESCE(cd.UUID, '_Undefined') AS ClaimUUID,
        COALESCE(cl.StatusId, -101) AS ClaimStatusId,
        COALESCE(cl.ReasonId, -101) AS ClaimReasonId,
        COALESCE(CAST(cd.IssuedDate AS DATE), DATE '1899-12-31') AS ClaimStatusReasonDate,
        COALESCE(cd.MemberEligibilityId, -101) AS ClaimMemberEligibilityId,
        COALESCE(cd.BillingProviderId, -101) AS ClaimBillingProviderId,
        'Billing' AS ClaimBillingProviderCategory,
        COALESCE(vld.ServiceProviderId, -101) AS ClaimServiceProviderId,
        'Service' AS ClaimServiceProviderCategory,
        COALESCE(cd.ScheduleId, -1) AS ClaimScheduleId,
        COALESCE(cd.ServiceIndicatorId, -101) AS ClaimDetailServiceIndicatorId,
        COALESCE(vld.ServiceIndicatorId, -101) AS ClaimLineDetailServiceIndicatorId,
        vld.FromDate AS ClaimServiceFromDate,
        vld.ToDate AS ClaimServiceToDate,
        vld.LocalCurrencyId AS ClaimLocalCurrencyId,
        vld.XChangeRate AS ClaimLocalXChangeRate,
        vld.XchangeDate AS ClaimLocalXChangeDate,
        vld.BaseCurrencyId AS ClaimBaseCurrencyId,
        cd.ReceivedDate AS ClaimReceivedDate,
        -101 AS ClaimGlobalLocationId,
        COALESCE(ProviderAddress.CountryId, -101) AS ClaimCountryId,
        -101 AS ClaimStateId,
        COALESCE(ProviderAddress.CityId, -101) AS ClaimCityId,
        COALESCE(cd.AccumulatorScopeId, -1) AS ClaimAccumulatorScopeId,
        COALESCE(cd.BillTypeId, -1) AS ClaimBillTypeId,
        1008 AS ClaimPayToId,
        COALESCE(Payee.PayeeId, -101) AS ClaimPayeeId,
        COALESCE(cd.PolicyId, -101) AS PolicyId,
        COALESCE(pp.ModeOfPaymentId, -101) AS PolicyModeOfPaymentId,
        COALESCE(pp.PaymentMethodId, -1) AS PolicyPaymentMethodId,
        COALESCE(pp.ReceivedMethodId, -101) AS PolicyReceivedMethodId,
        COALESCE(pp.GroupId, -101) AS PolicyGroupId,
        COALESCE(pp.FamilyTypeId, -101) AS PolicyFamilyTypeId,
        COALESCE(pp.ProductId, -101) AS PolicyProductId,
        COALESCE(pp.EntityId, -101) AS PolicyEntityId,
        COALESCE(pp.PlanId, -101) AS PolicyPlanId,
        COALESCE(pp.BusinessModeId, -101) AS PolicyBusinessModeId,
        COALESCE(pp.BusinessTypeId, -101) AS PolicyBusinessTypeId,
        COALESCE(pp.InsuranceBusinessId, -101) AS PolicyInsuranceBusinessId,
        COALESCE(pp.CompanyId, -101) AS PolicyCompanyId,
        pp.PolicyIssueDate, pp.PolicyRenewalDate, pp.PolicyAnniversaryDate,
        pp.PolicyApplicationReceivedDate, pp.PolicyEffectiveDate, pp.PolicyCancelDate,
        pp.PolicyDeathBenefitPeriodDate,
        COALESCE(pp.AgentHierarchyId, -101) AS PolicyAgentHierarchyId,
        COALESCE(pp.RegionId, -101) AS PolicyRegionId,
        COALESCE(pp.MemberOwnerId, -101) AS PolicyMemberOwnerId,
        pp.PolicyGlobalLocationId, pp.PolicyCountryId, pp.PolicyStateId, pp.PolicyCityId,
        COALESCE(TMPcr.RunId, -101) AS ClaimRunId,
        TMPcr.DetailRunId AS ClaimDetailRunId,
        TMPcr.InsuranceBusinessId AS ClaimRunInsuranceBusinessId,
        COALESCE(TMPcr.RunTypeId, -101) AS ClaimRunTypeId,
        CAST(TMPcr.CreatedOn AS DATE) AS ClaimRunCreatedOnDate,
        CAST(TMPcr.PrintedOn AS DATE) AS ClaimRunPrintedOnDate,
        COALESCE(TMPcr.Status, -101) AS ClaimRunStatusId,
        COALESCE(TMPcr.PaymentMethodId, -101) AS ClaimRunPaymentMethodId,
        -101 AS ClaimPrimaryDiagnosticId,
        -101 AS ClaimPrimaryProcedureId,
        cd.DiagnosticType AS ClaimDiagnosticType,
        vld.POSId AS ClaimPOSId,
        vld.TOSId AS ClaimTOSId,
        COALESCE(cp.PaymentId, -1) AS PaymentId,
        COALESCE(cp.PaymentNumber, -1) AS PaymentNumber,
        COALESCE(CAST(cp.PrintedDate AS DATE), DATE '1899-12-31') AS PaymentPrintedDate,
        COALESCE(cp.StatusId, -101) AS PaymentStatusId,
        COALESCE(PaymentVoid.ReasonId, -101) AS PaymentVoidReasonId,
        COALESCE(CAST(PPA.postedon AS DATE), DATE '1899-12-31') AS PaymentPostedDate,
        COALESCE(CAST(PaymentVoid.PaymentLogDate AS DATE), DATE '1899-12-31') AS PaymentVoidDate,
        cp.BankId AS PaymentBankId, cp.GatewayTransactionId AS PaymentGatewayTransactionId,
        cp.Amount AS PaymentAmount, cp.ForeignAmount AS PaymentForeignAmount,
        cp.BankTransmissionId AS PaymentBankTransmissionId,
        TMPcr.JournalEntryId, TMPcr.Notes AS JournalNotes,
        PaymentVoid.JournalEntryId AS VoidJournalEntryId, jeVoid.Notes AS VoidJournalNotes,
        COALESCE(vld.PayToMemberCurrencyId, -101) AS ClaimPaymentCurrencyToMemberId,
        PPA.XChangeRate AS ClaimPaymentToMemberXchangeRate,
        -101 AS ClaimPaymentCurrencyToProviderId,
        CAST(0 AS DOUBLE) AS ClaimPaymentToProviderXchangeRate,
        COALESCE(tp.ThirdPartyId, -101) AS ClaimPaymentThirdPartyId,
        COALESCE(NULLIF(cd.AccountNo, ''), '_Undefined') AS ClaimAccountNumber,
        TMPcr.Factor * vld.LocalBilledAmount AS LocalBilledAmount,
        TMPcr.Factor * vld.BaseAllowedAmount AS BaseAllowedAmount,
        TMPcr.Factor * vld.BaseBilledAmount AS BaseBilledAmount,
        TMPcr.Factor * vld.BaseCoInsuranceAmount AS BaseCoInsuranceAmount,
        CAST(0 AS DOUBLE) AS BaseAllowedCmp,
        TMPcr.Factor * vld.BaseCopayAmount AS BaseCopayAmount,
        TMPcr.Factor * vld.BaseDeductibleAmount AS BaseDeductibleAmount,
        TMPcr.Factor * vld.BaseNetCoveredAmount AS BaseNetCoveredAmount,
        PPA.BaseAmount AS BaseMemberPaidAmount,
        CAST(0 AS DOUBLE) AS BaseProviderPaidAmount,
        TMPcr.Factor * vld.BaseHB_DiscountAmount AS BaseDiscountAmount,
        TMPcr.Factor * vld.BaseTotalIneligibleAmount AS BaseTotalIneligibleAmount,
        TMPcr.Factor * vld.BaseOtherIneligibleAmount AS BaseOtherIneligibleAmount,
        PPA.BaseAmount AS BaseTotalPaidAmount,
        TMPcr.Factor * vld.BaseWithHolding AS BaseWithHoldingAmount,
        TMPcr.Factor * vld.BaseVAT AS BaseVATAmount,
        PPA.PaidAmount AS PayMemberPaidAmount,
        CAST(0 AS DOUBLE) AS PayProviderPaidAmount,
        vld.BaseXChangeRate,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseAllowedAmount ELSE vld.BaseAllowedAmount * vld.BaseXChangeRate END AS USD_AllowedAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseBilledAmount ELSE vld.BaseBilledAmount * vld.BaseXChangeRate END AS USD_BilledAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCoInsuranceAmount ELSE vld.BaseCoInsuranceAmount * vld.BaseXChangeRate END AS USD_CoInsuranceAmount,
        CAST(0 AS DOUBLE) AS USD_AllowedCmp,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCopayAmount ELSE vld.BaseCopayAmount * vld.BaseXChangeRate END AS USD_CopayAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseDeductibleAmount ELSE vld.BaseDeductibleAmount * vld.BaseXChangeRate END AS USD_DeductibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseNetCoveredAmount ELSE vld.BaseNetCoveredAmount * vld.BaseXChangeRate END AS USD_NetCoveredAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseHB_DiscountAmount ELSE vld.BaseHB_DiscountAmount * vld.BaseXChangeRate END AS USD_DiscountAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseTotalIneligibleAmount ELSE vld.BaseTotalIneligibleAmount * vld.BaseXChangeRate END AS USD_TotalIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseOtherIneligibleAmount ELSE vld.BaseOtherIneligibleAmount * vld.BaseXChangeRate END AS USD_OtherIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseWithHolding ELSE vld.BaseWithHolding * vld.BaseXChangeRate END AS USD_WithHoldingAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseVAT ELSE vld.BaseVAT * vld.BaseXChangeRate END AS USD_VATAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_MemberPaidAmount,
        CAST(0 AS DOUBLE) AS USD_ProviderPaidAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_TotalPaidAmount,
        TMPcr.Factor,
        cl.FromDate AS ClaimProcessedDate,
        CASE WHEN cl.StatusId = 1017 AND cl.ReasonId IN (1094, 1095, 1096, 1773, 1827, 1828, 1829, 1933) THEN 1 ELSE 0 END AS ClaimProcessRowFlag,
        TMPcr.Processed AS RunProcessed,
        1 AS ActualRowFlag,
        current_timestamp() AS LoadDate,
        'PAID MEMBER' AS PaidSource,
        CAST(0 AS DOUBLE) AS BaseClaimProviderPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS PayClaimProviderPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS USD_ClaimProviderPaymentAdjustedAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN 1
             WHEN vld.BaseCurrencyId = 192 THEN CAST(1 / CASE WHEN vld.PayToMemberXChangeRate = 0 THEN 1 ELSE vld.PayToMemberXChangeRate END AS DOUBLE)
             ELSE NULL END AS PayCurrencyToUSDXChangeRate,
        PPA.BaseAmount AS BaseClaimMemberPaymentAdjustedAmount,
        PPA.PaidAmount AS PayClaimMemberPaymentAdjustedAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_ClaimMemberPaymentAdjustedAmount,
        CASE WHEN cd.ServiceIndicatorId = 500 THEN 'GeoBlue' WHEN cd.ServiceIndicatorId = 432 THEN 'UHCI' WHEN cd.ServiceIndicatorId = 422 THEN 'Bupa' WHEN cd.ServiceIndicatorId = 773 THEN 'Olympus' WHEN IBRun.Max_InsuranceBusinessId = 58 THEN 'TBSL' ELSE 'Bupa' END AS ServiceNetwork,
        CASE WHEN IBRun.HeaderId IS NULL THEN 'NOT PAID' WHEN IBRun.min_isprovider <> IBRun.max_isprovider THEN 'SPLIT PAYMENT' WHEN IBRun.max_isprovider = 1 THEN 'PROVIDER PAID' ELSE 'MEMBER PAID' END AS Paid_To_Calculated,
        CASE WHEN cd.ServiceIndicatorId = 432 THEN 'INN' ELSE 'OON' END AS INN_OON,
        cd.MemberId,
        DATEDIFF(t.ServiceFromDate, t.ReceivedDate) AS ClaimGap,
        CASE WHEN cd.ServiceIndicatorId IN (500, 432, 773) THEN 5 ELSE cd.ClaimReceivedMethodId END AS ClaimReceivedMethodId,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 12) THEN 1 ELSE 0 END AS PolicyFirstYear,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 1) THEN 1 ELSE 0 END AS PolicyFirstMonth,
        0 AS FirstClaim, 0 AS IsDigital,
        COALESCE(ltTAT.IsCOR, '_Non Defined IsCOR') AS IsCOR,
        COALESCE(ltTAT.LocalTeam, '_Non Defined LocalTeam') AS LocalTeam,
        cd.IsFastTrack, vld.HTHFeePaid, vld.HTHFeePercent,
        0 AS IsDeleted,
        COALESCE(pcn.Name, '_Not defined ProviderNetworkClassName') AS ProviderNetworkClassName,
        0 AS BrackedClaimKey, 0 AS BrackedMemberKey,
        CAST(0 AS DOUBLE) AS IncomeByInsured, CAST(0 AS DOUBLE) AS SpendByInsured,
        0 AS ClaimsOutliers,
        COALESCE(ClaimSub.SenderEmailAddress, '_Not defined SenderEmailAddress') AS SubmissionSenderEmailAddress,
        COALESCE(ClaimSub.SenderFullName, '_Not defined SenderFullName') AS SubmissionSenderFullName,
        COALESCE(ClaimSub.InboundChannel, '_Not defined InboundChannel') AS SubmissionInboundChannel,
        COALESCE(ClaimSub.TrackingNumber, '_Not defined TrackingNumber') AS SubmissionTrackingNumber,
        COALESCE(r.RoleReferences, '_Not defined SenderType') AS SubmissionSenderType,
        '_Not defined' AS ClaimValidation,
        ch.BatchLotName, ch.BatchLotFromDate, ch.TransactionLotReceivedDate,
        ch.TransactionLotNumber, ch.TransactionInvoiceNumber, ch.LotNumber,
        COALESCE(ch.IsAutomatic, 0) AS IsAutomatic,
        CAST(NULL AS INT) AS IsICU, CAST(NULL AS INT) AS IsEmergency,
        CAST(NULL AS INT) AS IsHospital_Emergency, CAST(NULL AS INT) AS IsChildbirths,
        CAST(NULL AS INT) AS IsCesareanSection,
        -- PPA Balance: Member Non-Void
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalancePaymentCurrency,
        PPA.BaseAmount AS ClaimMemberPPABalanceBaseCurrenty,
        PPA.PaidAmount AS ClaimMemberPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalancePaymentCurrency,
        TMPcr.Factor * vld.BaseVAT * COALESCE(NULLIF(vld.PayToMemberXChangeRate, 0), 1) AS VATPaymentCurrency,
        TMPcr.Factor * vld.BaseWithHolding * COALESCE(NULLIF(vld.PayToMemberXChangeRate, 0), 1) AS WithholdingPaymentCurrency
    FROM PPA
    INNER JOIN vw_ClaimRun TMPcr ON TMPcr.RunId = PPA.RunId AND TMPcr.DetailRunId = PPA.PayDetailRunId AND TMPcr.JournalEntryId = PPA.JournalEntryId
    INNER JOIN vw_BaseClaimsLines vld ON vld.ClaimLineDetailId = PPA.ClaimLineDetailId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON cd.ClaimDetailId = PPA.ClaimDetailId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderAddress ProviderAddress ON cd.ProviderAddressId = ProviderAddress.ProviderAddressId
    LEFT JOIN vw_ClaimLog_Payment cl ON PPA.ClaimDetailId = cl.ClaimDetailId AND cl.rn = 1
    LEFT JOIN vw_PolicyHistory pp ON pp.PolicyId = cd.PolicyId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_Payment cp ON cp.PaymentId = TMPcr.PaymentId
    LEFT JOIN vw_ThirdPartyClaim tp ON tp.ClaimDetailId = cd.ClaimDetailId AND tp.PayTo = 1008
    LEFT JOIN Bronze.AmigosPlus_AMP_Policy_Member Member ON cd.MemberId = Member.MemberId
    LEFT JOIN vw_Payee_Payment Payee ON Member.ContactBaseId = Payee.ContactBaseId AND Payee.RowNum = 1
    LEFT JOIN vw_PaymentLogVoid PaymentVoid ON cp.PaymentId = PaymentVoid.PaymentId AND PaymentVoid.JournalEntryId = TMPcr.JournalEntryId
    LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry jeVoid ON jeVoid.JournalEntryId = PaymentVoid.JournalEntryId
    LEFT JOIN vw_InsuranceBusinessRun IBRun ON TMPcr.HeaderId = IBRun.HeaderId
    LEFT JOIN vw_TmpReceived_Payment t ON cd.HeaderId = t.HeaderId
    LEFT JOIN vw_LocalTeamTAT ltTAT ON cd.HeaderId = ltTAT.ClaimHeaderId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderNetworkClass pcn ON pcn.ProviderNetworkClassId = cd.ProviderNetworkClassId
    LEFT JOIN vw_ClaimSubmission_Payment ClaimSub ON ClaimSub.ClaimHeaderId = cd.HeaderId AND ClaimSub.PolicyId = cd.PolicyId AND ClaimSub.CONT = 1
    LEFT JOIN Bronze.Common_Roles r ON r.RoleId = ClaimSub.SenderTypeId
    LEFT JOIN (SELECT DISTINCT BatchLotName, BatchLotFromDate, TransactionLotReceivedDate, TransactionLotNumber, TransactionInvoiceNumber, LotNumber, IsAutomatic, ClaimDetailId FROM vw_ClearingHouseLine) ch ON ch.ClaimDetailId = cd.ClaimDetailId
""")
print(f"[OK] PAID MEMBER (PPA-adjusted, line-level) rows inserted into Silver.TmpFactClaimPayment")


In [ ]:
# ============================================================
# Cell: INSERT VOID PROVIDER (PPA-adjusted, line-level) -- Bug #3
# Emite UNA fila por (ClaimLineDetailId) con la cantidad post-PPA
# (adj_base / adj_pay -- ya negados en vw_ProviderPPAVoidAdjustment)
# y PaidSource = 'VOID PROVIDER'.
# Solo se emiten lineas que recibieron distribucion PPA (adj <> paid).
# ============================================================
spark.sql("""
    INSERT INTO Silver.TmpFactClaimPayment
    WITH PPA AS (
        SELECT
            adj.RunId,
            adj.DetailRunId AS PayDetailRunId,
            adj.ClaimDetailId,
            adj.ClaimLineDetailId,
            adj.LineNumber,
            CAST(adj.adj_base AS DOUBLE) AS BaseAmount,
            CAST(adj.adj_pay  AS DOUBLE) AS PaidAmount,
            ppa.HeaderId,
            ppa.JournalEntryId,
            ppa.PolicyId,
            ppa.postedon,
            ppa.XChangeRate
        FROM vw_ProviderPPAVoidAdjustment adj
        INNER JOIN (
            SELECT RunId, PayDetailRunId, PayClaimDetailId,
                   MAX(HeaderId)       AS HeaderId,
                   MAX(JournalEntryId) AS JournalEntryId,
                   MAX(PolicyId)       AS PolicyId,
                   MAX(postedon)       AS postedon,
                   MAX(XChangeRate)    AS XChangeRate
            FROM vw_ProviderPPA_Void
            GROUP BY RunId, PayDetailRunId, PayClaimDetailId
        ) ppa
          ON ppa.RunId            = adj.RunId
         AND ppa.PayDetailRunId   = adj.DetailRunId
         AND ppa.PayClaimDetailId = adj.ClaimDetailId
        WHERE adj.adj_base <> adj.paid_base OR adj.adj_pay <> adj.paid_pay
    )
    SELECT
        cd.HeaderId AS ClaimHeaderId,
        cd.HeaderId AS ClaimNumber,
        COALESCE(cd.RefNo, '_Undefined') AS ClaimReferenceNumber,
        COALESCE(cd.ClaimDetailId, -101) AS ClaimDetailId,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId ASC) AS ClaimVersion,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId DESC) AS ControlVersion,
        PPA.ClaimLineDetailId,
        DENSE_RANK() OVER(PARTITION BY cd.ClaimDetailId ORDER BY PPA.ClaimLineDetailId ASC) AS ClaimLineSequenceNumber,
        vld.ReferralHeaderId AS ClamReferralHeaderId,
        COALESCE(cd.UUID, '_Undefined') AS ClaimUUID,
        COALESCE(cl.StatusId, -101) AS ClaimStatusId,
        COALESCE(cl.ReasonId, -101) AS ClaimReasonId,
        COALESCE(CAST(cd.IssuedDate AS DATE), DATE '1899-12-31') AS ClaimStatusReasonDate,
        COALESCE(cd.MemberEligibilityId, -101) AS ClaimMemberEligibilityId,
        COALESCE(cd.BillingProviderId, -101) AS ClaimBillingProviderId,
        'Billing' AS ClaimBillingProviderCategory,
        COALESCE(vld.ServiceProviderId, -101) AS ClaimServiceProviderId,
        'Service' AS ClaimServiceProviderCategory,
        COALESCE(cd.ScheduleId, -1) AS ClaimScheduleId,
        COALESCE(cd.ServiceIndicatorId, -101) AS ClaimDetailServiceIndicatorId,
        COALESCE(vld.ServiceIndicatorId, -101) AS ClaimLineDetailServiceIndicatorId,
        vld.FromDate AS ClaimServiceFromDate,
        vld.ToDate AS ClaimServiceToDate,
        vld.LocalCurrencyId AS ClaimLocalCurrencyId,
        vld.XChangeRate AS ClaimLocalXChangeRate,
        vld.XchangeDate AS ClaimLocalXChangeDate,
        vld.BaseCurrencyId AS ClaimBaseCurrencyId,
        cd.ReceivedDate AS ClaimReceivedDate,
        -101 AS ClaimGlobalLocationId,
        COALESCE(ProviderAddress.CountryId, -101) AS ClaimCountryId,
        -101 AS ClaimStateId,
        COALESCE(ProviderAddress.CityId, -101) AS ClaimCityId,
        COALESCE(cd.AccumulatorScopeId, -1) AS ClaimAccumulatorScopeId,
        COALESCE(cd.BillTypeId, -1) AS ClaimBillTypeId,
        1007 AS ClaimPayToId,
        COALESCE(Payee.PayeeId, -101) AS ClaimPayeeId,
        COALESCE(cd.PolicyId, -101) AS PolicyId,
        COALESCE(pp.ModeOfPaymentId, -101) AS PolicyModeOfPaymentId,
        COALESCE(pp.PaymentMethodId, -1) AS PolicyPaymentMethodId,
        COALESCE(pp.ReceivedMethodId, -101) AS PolicyReceivedMethodId,
        COALESCE(pp.GroupId, -101) AS PolicyGroupId,
        COALESCE(pp.FamilyTypeId, -101) AS PolicyFamilyTypeId,
        COALESCE(pp.ProductId, -101) AS PolicyProductId,
        COALESCE(pp.EntityId, -101) AS PolicyEntityId,
        COALESCE(pp.PlanId, -101) AS PolicyPlanId,
        COALESCE(pp.BusinessModeId, -101) AS PolicyBusinessModeId,
        COALESCE(pp.BusinessTypeId, -101) AS PolicyBusinessTypeId,
        COALESCE(pp.InsuranceBusinessId, -101) AS PolicyInsuranceBusinessId,
        COALESCE(pp.CompanyId, -101) AS PolicyCompanyId,
        pp.PolicyIssueDate, pp.PolicyRenewalDate, pp.PolicyAnniversaryDate,
        pp.PolicyApplicationReceivedDate, pp.PolicyEffectiveDate, pp.PolicyCancelDate,
        pp.PolicyDeathBenefitPeriodDate,
        COALESCE(pp.AgentHierarchyId, -101) AS PolicyAgentHierarchyId,
        COALESCE(pp.RegionId, -101) AS PolicyRegionId,
        COALESCE(pp.MemberOwnerId, -101) AS PolicyMemberOwnerId,
        pp.PolicyGlobalLocationId, pp.PolicyCountryId, pp.PolicyStateId, pp.PolicyCityId,
        COALESCE(TMPcr.RunId, -101) AS ClaimRunId,
        TMPcr.DetailRunId AS ClaimDetailRunId,
        TMPcr.InsuranceBusinessId AS ClaimRunInsuranceBusinessId,
        COALESCE(TMPcr.RunTypeId, -101) AS ClaimRunTypeId,
        CAST(TMPcr.CreatedOn AS DATE) AS ClaimRunCreatedOnDate,
        CAST(TMPcr.PrintedOn AS DATE) AS ClaimRunPrintedOnDate,
        COALESCE(TMPcr.Status, -101) AS ClaimRunStatusId,
        COALESCE(TMPcr.PaymentMethodId, -101) AS ClaimRunPaymentMethodId,
        -101 AS ClaimPrimaryDiagnosticId,
        -101 AS ClaimPrimaryProcedureId,
        cd.DiagnosticType AS ClaimDiagnosticType,
        vld.POSId AS ClaimPOSId,
        vld.TOSId AS ClaimTOSId,
        COALESCE(cp.PaymentId, -1) AS PaymentId,
        COALESCE(cp.PaymentNumber, -1) AS PaymentNumber,
        COALESCE(CAST(cp.PrintedDate AS DATE), DATE '1899-12-31') AS PaymentPrintedDate,
        COALESCE(cp.StatusId, -101) AS PaymentStatusId,
        COALESCE(PaymentVoid.ReasonId, -101) AS PaymentVoidReasonId,
        COALESCE(CAST(PPA.postedon AS DATE), DATE '1899-12-31') AS PaymentPostedDate,
        COALESCE(CAST(PaymentVoid.PaymentLogDate AS DATE), DATE '1899-12-31') AS PaymentVoidDate,
        cp.BankId AS PaymentBankId, cp.GatewayTransactionId AS PaymentGatewayTransactionId,
        cp.Amount AS PaymentAmount, cp.ForeignAmount AS PaymentForeignAmount,
        cp.BankTransmissionId AS PaymentBankTransmissionId,
        TMPcr.JournalEntryId, TMPcr.Notes AS JournalNotes,
        PaymentVoid.JournalEntryId AS VoidJournalEntryId, jeVoid.Notes AS VoidJournalNotes,
        -101 AS ClaimPaymentCurrencyToMemberId,
        CAST(0 AS DOUBLE) AS ClaimPaymentToMemberXchangeRate,
        COALESCE(vld.PayToProviderCurrencyId, -101) AS ClaimPaymentCurrencyToProviderId,
        PPA.XChangeRate AS ClaimPaymentToProviderXchangeRate,
        COALESCE(tp.ThirdPartyId, -101) AS ClaimPaymentThirdPartyId,
        COALESCE(NULLIF(cd.AccountNo, ''), '_Undefined') AS ClaimAccountNumber,
        TMPcr.Factor * vld.LocalBilledAmount AS LocalBilledAmount,
        TMPcr.Factor * vld.BaseAllowedAmount AS BaseAllowedAmount,
        TMPcr.Factor * vld.BaseBilledAmount AS BaseBilledAmount,
        TMPcr.Factor * vld.BaseCoInsuranceAmount AS BaseCoInsuranceAmount,
        CAST(0 AS DOUBLE) AS BaseAllowedCmp,
        TMPcr.Factor * vld.BaseCopayAmount AS BaseCopayAmount,
        TMPcr.Factor * vld.BaseDeductibleAmount AS BaseDeductibleAmount,
        TMPcr.Factor * vld.BaseNetCoveredAmount AS BaseNetCoveredAmount,
        CAST(0 AS DOUBLE) AS BaseMemberPaidAmount,
        PPA.BaseAmount AS BaseProviderPaidAmount,
        TMPcr.Factor * vld.BaseHB_DiscountAmount AS BaseDiscountAmount,
        TMPcr.Factor * vld.BaseTotalIneligibleAmount AS BaseTotalIneligibleAmount,
        TMPcr.Factor * vld.BaseOtherIneligibleAmount AS BaseOtherIneligibleAmount,
        PPA.BaseAmount AS BaseTotalPaidAmount,
        TMPcr.Factor * vld.BaseWithHolding AS BaseWithHoldingAmount,
        TMPcr.Factor * vld.BaseVAT AS BaseVATAmount,
        CAST(0 AS DOUBLE) AS PayMemberPaidAmount,
        PPA.PaidAmount AS PayProviderPaidAmount,
        vld.BaseXChangeRate,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseAllowedAmount ELSE vld.BaseAllowedAmount * vld.BaseXChangeRate END AS USD_AllowedAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseBilledAmount ELSE vld.BaseBilledAmount * vld.BaseXChangeRate END AS USD_BilledAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCoInsuranceAmount ELSE vld.BaseCoInsuranceAmount * vld.BaseXChangeRate END AS USD_CoInsuranceAmount,
        CAST(0 AS DOUBLE) AS USD_AllowedCmp,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCopayAmount ELSE vld.BaseCopayAmount * vld.BaseXChangeRate END AS USD_CopayAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseDeductibleAmount ELSE vld.BaseDeductibleAmount * vld.BaseXChangeRate END AS USD_DeductibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseNetCoveredAmount ELSE vld.BaseNetCoveredAmount * vld.BaseXChangeRate END AS USD_NetCoveredAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseHB_DiscountAmount ELSE vld.BaseHB_DiscountAmount * vld.BaseXChangeRate END AS USD_DiscountAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseTotalIneligibleAmount ELSE vld.BaseTotalIneligibleAmount * vld.BaseXChangeRate END AS USD_TotalIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseOtherIneligibleAmount ELSE vld.BaseOtherIneligibleAmount * vld.BaseXChangeRate END AS USD_OtherIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseWithHolding ELSE vld.BaseWithHolding * vld.BaseXChangeRate END AS USD_WithHoldingAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseVAT ELSE vld.BaseVAT * vld.BaseXChangeRate END AS USD_VATAmount,
        CAST(0 AS DOUBLE) AS USD_MemberPaidAmount,
        CASE WHEN vld.PayToProviderCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_ProviderPaidAmount,
        CASE WHEN vld.PayToProviderCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_TotalPaidAmount,
        TMPcr.Factor,
        cl.FromDate AS ClaimProcessedDate,
        CASE WHEN cl.StatusId = 1017 AND cl.ReasonId IN (1094, 1095, 1096, 1773, 1827, 1828, 1829, 1933) THEN 1 ELSE 0 END AS ClaimProcessRowFlag,
        TMPcr.Processed AS RunProcessed,
        1 AS ActualRowFlag,
        current_timestamp() AS LoadDate,
        'VOID PROVIDER' AS PaidSource,
        PPA.BaseAmount AS BaseClaimProviderPaymentAdjustedAmount,
        PPA.PaidAmount AS PayClaimProviderPaymentAdjustedAmount,
        CASE WHEN vld.PayToProviderCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_ClaimProviderPaymentAdjustedAmount,
        CASE WHEN vld.PayToProviderCurrencyId = 192 THEN 1
             WHEN vld.BaseCurrencyId = 192 THEN CAST(1 / CASE WHEN vld.PayToProviderXChangeRate = 0 THEN 1 ELSE vld.PayToProviderXChangeRate END AS DOUBLE)
             ELSE NULL END AS PayCurrencyToUSDXChangeRate,
        CAST(0 AS DOUBLE) AS BaseClaimMemberPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS PayClaimMemberPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS USD_ClaimMemberPaymentAdjustedAmount,
        CASE WHEN cd.ServiceIndicatorId = 500 THEN 'GeoBlue' WHEN cd.ServiceIndicatorId = 432 THEN 'UHCI' WHEN cd.ServiceIndicatorId = 422 THEN 'Bupa' WHEN cd.ServiceIndicatorId = 773 THEN 'Olympus' WHEN IBRun.Max_InsuranceBusinessId = 58 THEN 'TBSL' ELSE 'Bupa' END AS ServiceNetwork,
        CASE WHEN IBRun.HeaderId IS NULL THEN 'NOT PAID' WHEN IBRun.min_isprovider <> IBRun.max_isprovider THEN 'SPLIT PAYMENT' WHEN IBRun.max_isprovider = 1 THEN 'PROVIDER VOID' ELSE 'MEMBER VOID' END AS Paid_To_Calculated,
        CASE WHEN cd.ServiceIndicatorId = 432 THEN 'INN' ELSE 'OON' END AS INN_OON,
        cd.MemberId,
        DATEDIFF(t.ServiceFromDate, t.ReceivedDate) AS ClaimGap,
        CASE WHEN cd.ServiceIndicatorId IN (500, 432, 773) THEN 5 ELSE cd.ClaimReceivedMethodId END AS ClaimReceivedMethodId,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 12) THEN 1 ELSE 0 END AS PolicyFirstYear,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 1) THEN 1 ELSE 0 END AS PolicyFirstMonth,
        0 AS FirstClaim, 0 AS IsDigital,
        COALESCE(ltTAT.IsCOR, '_Non Defined IsCOR') AS IsCOR,
        COALESCE(ltTAT.LocalTeam, '_Non Defined LocalTeam') AS LocalTeam,
        cd.IsFastTrack, vld.HTHFeePaid, vld.HTHFeePercent,
        0 AS IsDeleted,
        COALESCE(pcn.Name, '_Not defined ProviderNetworkClassName') AS ProviderNetworkClassName,
        0 AS BrackedClaimKey, 0 AS BrackedMemberKey,
        CAST(0 AS DOUBLE) AS IncomeByInsured, CAST(0 AS DOUBLE) AS SpendByInsured,
        0 AS ClaimsOutliers,
        COALESCE(ClaimSub.SenderEmailAddress, '_Not defined SenderEmailAddress') AS SubmissionSenderEmailAddress,
        COALESCE(ClaimSub.SenderFullName, '_Not defined SenderFullName') AS SubmissionSenderFullName,
        COALESCE(ClaimSub.InboundChannel, '_Not defined InboundChannel') AS SubmissionInboundChannel,
        COALESCE(ClaimSub.TrackingNumber, '_Not defined TrackingNumber') AS SubmissionTrackingNumber,
        COALESCE(r.RoleReferences, '_Not defined SenderType') AS SubmissionSenderType,
        '_Not defined' AS ClaimValidation,
        ch.BatchLotName, ch.BatchLotFromDate, ch.TransactionLotReceivedDate,
        ch.TransactionLotNumber, ch.TransactionInvoiceNumber, ch.LotNumber,
        COALESCE(ch.IsAutomatic, 0) AS IsAutomatic,
        CAST(NULL AS INT) AS IsICU, CAST(NULL AS INT) AS IsEmergency,
        CAST(NULL AS INT) AS IsHospital_Emergency, CAST(NULL AS INT) AS IsChildbirths,
        CAST(NULL AS INT) AS IsCesareanSection,
        -- PPA Balance: Provider Void
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalancePaymentCurrency,
        PPA.BaseAmount AS ClaimProviderVoidPPABalanceBaseCurrenty,
        PPA.PaidAmount AS ClaimProviderVoidPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberVoidPPABalancePaymentCurrency,
        TMPcr.Factor * vld.BaseVAT * COALESCE(NULLIF(vld.PayToProviderXChangeRate, 0), 1) AS VATPaymentCurrency,
        TMPcr.Factor * vld.BaseWithHolding * COALESCE(NULLIF(vld.PayToProviderXChangeRate, 0), 1) AS WithholdingPaymentCurrency
    FROM PPA
    INNER JOIN vw_ClaimRun TMPcr ON TMPcr.RunId = PPA.RunId AND TMPcr.DetailRunId = PPA.PayDetailRunId AND TMPcr.JournalEntryId = PPA.JournalEntryId
    INNER JOIN vw_BaseClaimsLines vld ON vld.ClaimLineDetailId = PPA.ClaimLineDetailId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON cd.ClaimDetailId = PPA.ClaimDetailId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderAddress ProviderAddress ON cd.ProviderAddressId = ProviderAddress.ProviderAddressId
    LEFT JOIN vw_ClaimLog_Payment cl ON PPA.ClaimDetailId = cl.ClaimDetailId AND cl.rn = 1
    LEFT JOIN vw_PolicyHistory pp ON pp.PolicyId = cd.PolicyId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_Payment cp ON cp.PaymentId = TMPcr.PaymentId
    LEFT JOIN vw_ThirdPartyClaim tp ON tp.ClaimDetailId = cd.ClaimDetailId AND tp.PayTo = 1007
    LEFT JOIN Bronze.AmigosPlus_AMP_Policy_Member Member ON cd.MemberId = Member.MemberId
    LEFT JOIN vw_Payee_Payment Payee ON Member.ContactBaseId = Payee.ContactBaseId AND Payee.RowNum = 1
    LEFT JOIN vw_PaymentLogVoid PaymentVoid ON cp.PaymentId = PaymentVoid.PaymentId AND PaymentVoid.JournalEntryId = TMPcr.JournalEntryId
    LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry jeVoid ON jeVoid.JournalEntryId = PaymentVoid.JournalEntryId
    LEFT JOIN vw_InsuranceBusinessRun IBRun ON TMPcr.HeaderId = IBRun.HeaderId
    LEFT JOIN vw_TmpReceived_Payment t ON cd.HeaderId = t.HeaderId
    LEFT JOIN vw_LocalTeamTAT ltTAT ON cd.HeaderId = ltTAT.ClaimHeaderId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderNetworkClass pcn ON pcn.ProviderNetworkClassId = cd.ProviderNetworkClassId
    LEFT JOIN vw_ClaimSubmission_Payment ClaimSub ON ClaimSub.ClaimHeaderId = cd.HeaderId AND ClaimSub.PolicyId = cd.PolicyId AND ClaimSub.CONT = 1
    LEFT JOIN Bronze.Common_Roles r ON r.RoleId = ClaimSub.SenderTypeId
    LEFT JOIN (SELECT DISTINCT BatchLotName, BatchLotFromDate, TransactionLotReceivedDate, TransactionLotNumber, TransactionInvoiceNumber, LotNumber, IsAutomatic, ClaimDetailId FROM vw_ClearingHouseLine) ch ON ch.ClaimDetailId = cd.ClaimDetailId
""")
print(f"[OK] VOID PROVIDER (PPA-adjusted, line-level) rows inserted into Silver.TmpFactClaimPayment")


In [ ]:
# ============================================================
# Cell: INSERT VOID MEMBER (PPA-adjusted, line-level) -- Bug #3
# Emite UNA fila por (ClaimLineDetailId) con la cantidad post-PPA
# (adj_base / adj_pay -- ya negados en vw_MemberPPAVoidAdjustment)
# y PaidSource = 'VOID MEMBER'.
# Solo se emiten lineas que recibieron distribucion PPA (adj <> paid).
# ============================================================
spark.sql("""
    INSERT INTO Silver.TmpFactClaimPayment
    WITH PPA AS (
        SELECT
            adj.RunId,
            adj.DetailRunId AS PayDetailRunId,
            adj.ClaimDetailId,
            adj.ClaimLineDetailId,
            adj.LineNumber,
            CAST(adj.adj_base AS DOUBLE) AS BaseAmount,
            CAST(adj.adj_pay  AS DOUBLE) AS PaidAmount,
            ppa.HeaderId,
            ppa.JournalEntryId,
            ppa.PolicyId,
            ppa.postedon,
            ppa.XChangeRate
        FROM vw_MemberPPAVoidAdjustment adj
        INNER JOIN (
            SELECT RunId, PayDetailRunId, PayClaimDetailId,
                   MAX(HeaderId)       AS HeaderId,
                   MAX(JournalEntryId) AS JournalEntryId,
                   MAX(PolicyId)       AS PolicyId,
                   MAX(postedon)       AS postedon,
                   MAX(XChangeRate)    AS XChangeRate
            FROM vw_MemberPPA_Void
            GROUP BY RunId, PayDetailRunId, PayClaimDetailId
        ) ppa
          ON ppa.RunId            = adj.RunId
         AND ppa.PayDetailRunId   = adj.DetailRunId
         AND ppa.PayClaimDetailId = adj.ClaimDetailId
        WHERE adj.adj_base <> adj.paid_base OR adj.adj_pay <> adj.paid_pay
    )
    SELECT
        cd.HeaderId AS ClaimHeaderId,
        cd.HeaderId AS ClaimNumber,
        COALESCE(cd.RefNo, '_Undefined') AS ClaimReferenceNumber,
        COALESCE(cd.ClaimDetailId, -101) AS ClaimDetailId,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId ASC) AS ClaimVersion,
        DENSE_RANK() OVER(PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId DESC) AS ControlVersion,
        PPA.ClaimLineDetailId,
        DENSE_RANK() OVER(PARTITION BY cd.ClaimDetailId ORDER BY PPA.ClaimLineDetailId ASC) AS ClaimLineSequenceNumber,
        vld.ReferralHeaderId AS ClamReferralHeaderId,
        COALESCE(cd.UUID, '_Undefined') AS ClaimUUID,
        COALESCE(cl.StatusId, -101) AS ClaimStatusId,
        COALESCE(cl.ReasonId, -101) AS ClaimReasonId,
        COALESCE(CAST(cd.IssuedDate AS DATE), DATE '1899-12-31') AS ClaimStatusReasonDate,
        COALESCE(cd.MemberEligibilityId, -101) AS ClaimMemberEligibilityId,
        COALESCE(cd.BillingProviderId, -101) AS ClaimBillingProviderId,
        'Billing' AS ClaimBillingProviderCategory,
        COALESCE(vld.ServiceProviderId, -101) AS ClaimServiceProviderId,
        'Service' AS ClaimServiceProviderCategory,
        COALESCE(cd.ScheduleId, -1) AS ClaimScheduleId,
        COALESCE(cd.ServiceIndicatorId, -101) AS ClaimDetailServiceIndicatorId,
        COALESCE(vld.ServiceIndicatorId, -101) AS ClaimLineDetailServiceIndicatorId,
        vld.FromDate AS ClaimServiceFromDate,
        vld.ToDate AS ClaimServiceToDate,
        vld.LocalCurrencyId AS ClaimLocalCurrencyId,
        vld.XChangeRate AS ClaimLocalXChangeRate,
        vld.XchangeDate AS ClaimLocalXChangeDate,
        vld.BaseCurrencyId AS ClaimBaseCurrencyId,
        cd.ReceivedDate AS ClaimReceivedDate,
        -101 AS ClaimGlobalLocationId,
        COALESCE(ProviderAddress.CountryId, -101) AS ClaimCountryId,
        -101 AS ClaimStateId,
        COALESCE(ProviderAddress.CityId, -101) AS ClaimCityId,
        COALESCE(cd.AccumulatorScopeId, -1) AS ClaimAccumulatorScopeId,
        COALESCE(cd.BillTypeId, -1) AS ClaimBillTypeId,
        1008 AS ClaimPayToId,
        COALESCE(Payee.PayeeId, -101) AS ClaimPayeeId,
        COALESCE(cd.PolicyId, -101) AS PolicyId,
        COALESCE(pp.ModeOfPaymentId, -101) AS PolicyModeOfPaymentId,
        COALESCE(pp.PaymentMethodId, -1) AS PolicyPaymentMethodId,
        COALESCE(pp.ReceivedMethodId, -101) AS PolicyReceivedMethodId,
        COALESCE(pp.GroupId, -101) AS PolicyGroupId,
        COALESCE(pp.FamilyTypeId, -101) AS PolicyFamilyTypeId,
        COALESCE(pp.ProductId, -101) AS PolicyProductId,
        COALESCE(pp.EntityId, -101) AS PolicyEntityId,
        COALESCE(pp.PlanId, -101) AS PolicyPlanId,
        COALESCE(pp.BusinessModeId, -101) AS PolicyBusinessModeId,
        COALESCE(pp.BusinessTypeId, -101) AS PolicyBusinessTypeId,
        COALESCE(pp.InsuranceBusinessId, -101) AS PolicyInsuranceBusinessId,
        COALESCE(pp.CompanyId, -101) AS PolicyCompanyId,
        pp.PolicyIssueDate, pp.PolicyRenewalDate, pp.PolicyAnniversaryDate,
        pp.PolicyApplicationReceivedDate, pp.PolicyEffectiveDate, pp.PolicyCancelDate,
        pp.PolicyDeathBenefitPeriodDate,
        COALESCE(pp.AgentHierarchyId, -101) AS PolicyAgentHierarchyId,
        COALESCE(pp.RegionId, -101) AS PolicyRegionId,
        COALESCE(pp.MemberOwnerId, -101) AS PolicyMemberOwnerId,
        pp.PolicyGlobalLocationId, pp.PolicyCountryId, pp.PolicyStateId, pp.PolicyCityId,
        COALESCE(TMPcr.RunId, -101) AS ClaimRunId,
        TMPcr.DetailRunId AS ClaimDetailRunId,
        TMPcr.InsuranceBusinessId AS ClaimRunInsuranceBusinessId,
        COALESCE(TMPcr.RunTypeId, -101) AS ClaimRunTypeId,
        CAST(TMPcr.CreatedOn AS DATE) AS ClaimRunCreatedOnDate,
        CAST(TMPcr.PrintedOn AS DATE) AS ClaimRunPrintedOnDate,
        COALESCE(TMPcr.Status, -101) AS ClaimRunStatusId,
        COALESCE(TMPcr.PaymentMethodId, -101) AS ClaimRunPaymentMethodId,
        -101 AS ClaimPrimaryDiagnosticId, -101 AS ClaimPrimaryProcedureId,
        cd.DiagnosticType AS ClaimDiagnosticType,
        vld.POSId AS ClaimPOSId, vld.TOSId AS ClaimTOSId,
        COALESCE(cp.PaymentId, -1) AS PaymentId,
        COALESCE(cp.PaymentNumber, -1) AS PaymentNumber,
        COALESCE(CAST(cp.PrintedDate AS DATE), DATE '1899-12-31') AS PaymentPrintedDate,
        COALESCE(cp.StatusId, -101) AS PaymentStatusId,
        COALESCE(PaymentVoid.ReasonId, -101) AS PaymentVoidReasonId,
        COALESCE(CAST(PPA.postedon AS DATE), DATE '1899-12-31') AS PaymentPostedDate,
        COALESCE(CAST(PaymentVoid.PaymentLogDate AS DATE), DATE '1899-12-31') AS PaymentVoidDate,
        cp.BankId AS PaymentBankId, cp.GatewayTransactionId AS PaymentGatewayTransactionId,
        cp.Amount AS PaymentAmount, cp.ForeignAmount AS PaymentForeignAmount,
        cp.BankTransmissionId AS PaymentBankTransmissionId,
        TMPcr.JournalEntryId, TMPcr.Notes AS JournalNotes,
        PaymentVoid.JournalEntryId AS VoidJournalEntryId, jeVoid.Notes AS VoidJournalNotes,
        COALESCE(vld.PayToMemberCurrencyId, -101) AS ClaimPaymentCurrencyToMemberId,
        PPA.XChangeRate AS ClaimPaymentToMemberXchangeRate,
        -101 AS ClaimPaymentCurrencyToProviderId,
        CAST(0 AS DOUBLE) AS ClaimPaymentToProviderXchangeRate,
        COALESCE(tp.ThirdPartyId, -101) AS ClaimPaymentThirdPartyId,
        COALESCE(NULLIF(cd.AccountNo, ''), '_Undefined') AS ClaimAccountNumber,
        TMPcr.Factor * vld.LocalBilledAmount AS LocalBilledAmount,
        TMPcr.Factor * vld.BaseAllowedAmount AS BaseAllowedAmount,
        TMPcr.Factor * vld.BaseBilledAmount AS BaseBilledAmount,
        TMPcr.Factor * vld.BaseCoInsuranceAmount AS BaseCoInsuranceAmount,
        CAST(0 AS DOUBLE) AS BaseAllowedCmp,
        TMPcr.Factor * vld.BaseCopayAmount AS BaseCopayAmount,
        TMPcr.Factor * vld.BaseDeductibleAmount AS BaseDeductibleAmount,
        TMPcr.Factor * vld.BaseNetCoveredAmount AS BaseNetCoveredAmount,
        PPA.BaseAmount AS BaseMemberPaidAmount,
        CAST(0 AS DOUBLE) AS BaseProviderPaidAmount,
        TMPcr.Factor * vld.BaseHB_DiscountAmount AS BaseDiscountAmount,
        TMPcr.Factor * vld.BaseTotalIneligibleAmount AS BaseTotalIneligibleAmount,
        TMPcr.Factor * vld.BaseOtherIneligibleAmount AS BaseOtherIneligibleAmount,
        PPA.BaseAmount AS BaseTotalPaidAmount,
        TMPcr.Factor * vld.BaseWithHolding AS BaseWithHoldingAmount,
        TMPcr.Factor * vld.BaseVAT AS BaseVATAmount,
        PPA.PaidAmount AS PayMemberPaidAmount,
        CAST(0 AS DOUBLE) AS PayProviderPaidAmount,
        vld.BaseXChangeRate,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseAllowedAmount ELSE vld.BaseAllowedAmount * vld.BaseXChangeRate END AS USD_AllowedAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseBilledAmount ELSE vld.BaseBilledAmount * vld.BaseXChangeRate END AS USD_BilledAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCoInsuranceAmount ELSE vld.BaseCoInsuranceAmount * vld.BaseXChangeRate END AS USD_CoInsuranceAmount,
        CAST(0 AS DOUBLE) AS USD_AllowedCmp,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseCopayAmount ELSE vld.BaseCopayAmount * vld.BaseXChangeRate END AS USD_CopayAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseDeductibleAmount ELSE vld.BaseDeductibleAmount * vld.BaseXChangeRate END AS USD_DeductibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseNetCoveredAmount ELSE vld.BaseNetCoveredAmount * vld.BaseXChangeRate END AS USD_NetCoveredAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseHB_DiscountAmount ELSE vld.BaseHB_DiscountAmount * vld.BaseXChangeRate END AS USD_DiscountAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseTotalIneligibleAmount ELSE vld.BaseTotalIneligibleAmount * vld.BaseXChangeRate END AS USD_TotalIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseOtherIneligibleAmount ELSE vld.BaseOtherIneligibleAmount * vld.BaseXChangeRate END AS USD_OtherIneligibleAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseWithHolding ELSE vld.BaseWithHolding * vld.BaseXChangeRate END AS USD_WithHoldingAmount,
        TMPcr.Factor * CASE WHEN vld.BaseCurrencyId = 192 THEN vld.BaseVAT ELSE vld.BaseVAT * vld.BaseXChangeRate END AS USD_VATAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_MemberPaidAmount,
        CAST(0 AS DOUBLE) AS USD_ProviderPaidAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_TotalPaidAmount,
        TMPcr.Factor,
        cl.FromDate AS ClaimProcessedDate,
        CASE WHEN cl.StatusId = 1017 AND cl.ReasonId IN (1094, 1095, 1096, 1773, 1827, 1828, 1829, 1933) THEN 1 ELSE 0 END AS ClaimProcessRowFlag,
        TMPcr.Processed AS RunProcessed, 1 AS ActualRowFlag,
        current_timestamp() AS LoadDate,
        'VOID MEMBER' AS PaidSource,
        CAST(0 AS DOUBLE) AS BaseClaimProviderPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS PayClaimProviderPaymentAdjustedAmount,
        CAST(0 AS DOUBLE) AS USD_ClaimProviderPaymentAdjustedAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN 1
             WHEN vld.BaseCurrencyId = 192 THEN CAST(1 / CASE WHEN vld.PayToMemberXChangeRate = 0 THEN 1 ELSE vld.PayToMemberXChangeRate END AS DOUBLE)
             ELSE NULL END AS PayCurrencyToUSDXChangeRate,
        PPA.BaseAmount AS BaseClaimMemberPaymentAdjustedAmount,
        PPA.PaidAmount AS PayClaimMemberPaymentAdjustedAmount,
        CASE WHEN vld.PayToMemberCurrencyId = 192 THEN PPA.PaidAmount
             WHEN vld.BaseCurrencyId = 192 THEN PPA.BaseAmount
             ELSE CAST(NULL AS DOUBLE) END AS USD_ClaimMemberPaymentAdjustedAmount,
        CASE WHEN cd.ServiceIndicatorId = 500 THEN 'GeoBlue' WHEN cd.ServiceIndicatorId = 432 THEN 'UHCI' WHEN cd.ServiceIndicatorId = 422 THEN 'Bupa' WHEN cd.ServiceIndicatorId = 773 THEN 'Olympus' WHEN IBRun.Max_InsuranceBusinessId = 58 THEN 'TBSL' ELSE 'Bupa' END AS ServiceNetwork,
        CASE WHEN IBRun.HeaderId IS NULL THEN 'NOT PAID' WHEN IBRun.min_isprovider <> IBRun.max_isprovider THEN 'SPLIT PAYMENT' WHEN IBRun.max_isprovider = 1 THEN 'PROVIDER VOID' ELSE 'MEMBER VOID' END AS Paid_To_Calculated,
        CASE WHEN cd.ServiceIndicatorId = 432 THEN 'INN' ELSE 'OON' END AS INN_OON,
        cd.MemberId, DATEDIFF(t.ServiceFromDate, t.ReceivedDate) AS ClaimGap,
        CASE WHEN cd.ServiceIndicatorId IN (500, 432, 773) THEN 5 ELSE cd.ClaimReceivedMethodId END AS ClaimReceivedMethodId,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 12) THEN 1 ELSE 0 END AS PolicyFirstYear,
        CASE WHEN vld.FromDate BETWEEN pp.PolicyIssueDate AND ADD_MONTHS(pp.PolicyIssueDate, 1) THEN 1 ELSE 0 END AS PolicyFirstMonth,
        0 AS FirstClaim, 0 AS IsDigital,
        COALESCE(ltTAT.IsCOR, '_Non Defined IsCOR') AS IsCOR,
        COALESCE(ltTAT.LocalTeam, '_Non Defined LocalTeam') AS LocalTeam,
        cd.IsFastTrack, vld.HTHFeePaid, vld.HTHFeePercent, 0 AS IsDeleted,
        COALESCE(pcn.Name, '_Not defined ProviderNetworkClassName') AS ProviderNetworkClassName,
        0 AS BrackedClaimKey, 0 AS BrackedMemberKey,
        CAST(0 AS DOUBLE) AS IncomeByInsured, CAST(0 AS DOUBLE) AS SpendByInsured,
        0 AS ClaimsOutliers,
        COALESCE(ClaimSub.SenderEmailAddress, '_Not defined SenderEmailAddress') AS SubmissionSenderEmailAddress,
        COALESCE(ClaimSub.SenderFullName, '_Not defined SenderFullName') AS SubmissionSenderFullName,
        COALESCE(ClaimSub.InboundChannel, '_Not defined InboundChannel') AS SubmissionInboundChannel,
        COALESCE(ClaimSub.TrackingNumber, '_Not defined TrackingNumber') AS SubmissionTrackingNumber,
        COALESCE(r.RoleReferences, '_Not defined SenderType') AS SubmissionSenderType,
        '_Not defined' AS ClaimValidation,
        ch.BatchLotName, ch.BatchLotFromDate, ch.TransactionLotReceivedDate,
        ch.TransactionLotNumber, ch.TransactionInvoiceNumber, ch.LotNumber,
        COALESCE(ch.IsAutomatic, 0) AS IsAutomatic,
        CAST(NULL AS INT) AS IsICU, CAST(NULL AS INT) AS IsEmergency,
        CAST(NULL AS INT) AS IsHospital_Emergency, CAST(NULL AS INT) AS IsChildbirths,
        CAST(NULL AS INT) AS IsCesareanSection,
        -- PPA Balance: Member Void
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimMemberPPABalancePaymentCurrency,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalanceBaseCurrenty,
        CAST(0 AS DOUBLE) AS ClaimProviderVoidPPABalancePaymentCurrency,
        PPA.BaseAmount AS ClaimMemberVoidPPABalanceBaseCurrenty,
        PPA.PaidAmount AS ClaimMemberVoidPPABalancePaymentCurrency,
        TMPcr.Factor * vld.BaseVAT * COALESCE(NULLIF(vld.PayToMemberXChangeRate, 0), 1) AS VATPaymentCurrency,
        TMPcr.Factor * vld.BaseWithHolding * COALESCE(NULLIF(vld.PayToMemberXChangeRate, 0), 1) AS WithholdingPaymentCurrency
    FROM PPA
    INNER JOIN vw_ClaimRun TMPcr ON TMPcr.RunId = PPA.RunId AND TMPcr.DetailRunId = PPA.PayDetailRunId AND TMPcr.JournalEntryId = PPA.JournalEntryId
    INNER JOIN vw_BaseClaimsLines vld ON vld.ClaimLineDetailId = PPA.ClaimLineDetailId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON cd.ClaimDetailId = PPA.ClaimDetailId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderAddress ProviderAddress ON cd.ProviderAddressId = ProviderAddress.ProviderAddressId
    LEFT JOIN vw_ClaimLog_Payment cl ON PPA.ClaimDetailId = cl.ClaimDetailId AND cl.rn = 1
    LEFT JOIN vw_PolicyHistory pp ON pp.PolicyId = cd.PolicyId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_Payment cp ON cp.PaymentId = TMPcr.PaymentId
    LEFT JOIN vw_ThirdPartyClaim tp ON tp.ClaimDetailId = cd.ClaimDetailId AND tp.PayTo = 1008
    LEFT JOIN Bronze.AmigosPlus_AMP_Policy_Member Member ON cd.MemberId = Member.MemberId
    LEFT JOIN vw_Payee_Payment Payee ON Member.ContactBaseId = Payee.ContactBaseId AND Payee.RowNum = 1
    LEFT JOIN vw_PaymentLogVoid PaymentVoid ON cp.PaymentId = PaymentVoid.PaymentId AND PaymentVoid.JournalEntryId = TMPcr.JournalEntryId
    LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry jeVoid ON jeVoid.JournalEntryId = PaymentVoid.JournalEntryId
    LEFT JOIN vw_InsuranceBusinessRun IBRun ON TMPcr.HeaderId = IBRun.HeaderId
    LEFT JOIN vw_TmpReceived_Payment t ON cd.HeaderId = t.HeaderId
    LEFT JOIN vw_LocalTeamTAT ltTAT ON cd.HeaderId = ltTAT.ClaimHeaderId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_ProviderNetworkClass pcn ON pcn.ProviderNetworkClassId = cd.ProviderNetworkClassId
    LEFT JOIN vw_ClaimSubmission_Payment ClaimSub ON ClaimSub.ClaimHeaderId = cd.HeaderId AND ClaimSub.PolicyId = cd.PolicyId AND ClaimSub.CONT = 1
    LEFT JOIN Bronze.Common_Roles r ON r.RoleId = ClaimSub.SenderTypeId
    LEFT JOIN (SELECT DISTINCT BatchLotName, BatchLotFromDate, TransactionLotReceivedDate, TransactionLotNumber, TransactionInvoiceNumber, LotNumber, IsAutomatic, ClaimDetailId FROM vw_ClearingHouseLine) ch ON ch.ClaimDetailId = cd.ClaimDetailId
""")
print(f"[OK] VOID MEMBER (PPA-adjusted, line-level) rows inserted into Silver.TmpFactClaimPayment")


In [ ]:
# ============================================================
# Cell: Normalize PPA line-level rows to SQL-compatible PPA sources
# ------------------------------------------------------------
# SQL Server writes PPA balances as aggregate PPA rows with
# ClaimLineDetailId = -101 and adjusted columns = 0. Older line-level
# PPA insert cells can emit rows as PAID/VOID MEMBER/PROVIDER and
# pollute regular paid rows. This cell converts those rows in staging
# before exchange-rate updates and final consolidation.
# ============================================================
from pyspark.sql import functions as F

tmp_fact_table = 'Silver.TmpFactClaimPayment'
tmp_fact_df = spark.table(tmp_fact_table)
tmp_fact_schema = tmp_fact_df.schema
tmp_fact_cols = tmp_fact_df.columns

ppa_normalization_rules = [
    {
        'source': 'PAID MEMBER',
        'target': 'PPA MEMBER',
        'balance_base': 'ClaimMemberPPABalanceBaseCurrenty',
        'balance_pay': 'ClaimMemberPPABalancePaymentCurrency',
        'side': 'member',
        'void': False
    },
    {
        'source': 'VOID PROVIDER',
        'target': 'VOID PPA PROVIDER',
        'balance_base': 'ClaimProviderVoidPPABalanceBaseCurrenty',
        'balance_pay': 'ClaimProviderVoidPPABalancePaymentCurrency',
        'side': 'provider',
        'void': True
    },
    {
        'source': 'VOID MEMBER',
        'target': 'VOID PPA MEMBER',
        'balance_base': 'ClaimMemberVoidPPABalanceBaseCurrenty',
        'balance_pay': 'ClaimMemberVoidPPABalancePaymentCurrency',
        'side': 'member',
        'void': True
    }
]

group_keys = ['ClaimHeaderId', 'ClaimDetailId', 'ClaimRunId', 'ClaimDetailRunId', 'JournalEntryId', 'PaymentId']
decimal_zero = F.lit(0).cast('decimal(38,22)')

def normalize_ppa_rule(rule):
    source = rule['source']
    target = rule['target']
    balance_base = rule['balance_base']
    balance_pay = rule['balance_pay']
    side = rule['side']

    candidate_df = spark.table(tmp_fact_table).where(
        (F.col('PaidSource') == source) &
        (F.col('ClaimLineDetailId') != F.lit(-101)) &
        ((F.col(balance_base) != 0) | (F.col(balance_pay) != 0))
    )

    candidate_count = candidate_df.count()
    if candidate_count == 0:
        print(f'[OK] No line-level rows to normalize for {source} -> {target}')
        return

    sum_cols = {balance_base, balance_pay, 'BaseTotalPaidAmount', 'USD_TotalPaidAmount'}
    if side == 'member':
        sum_cols.update({'BaseMemberPaidAmount', 'PayMemberPaidAmount', 'USD_MemberPaidAmount'})
    else:
        sum_cols.update({'BaseProviderPaidAmount', 'PayProviderPaidAmount', 'USD_ProviderPaidAmount'})

    agg_exprs = []
    for col_name in tmp_fact_cols:
        if col_name in group_keys:
            continue
        if col_name in sum_cols:
            agg_exprs.append(F.sum(F.col(col_name)).alias(col_name))
        else:
            agg_exprs.append(F.first(F.col(col_name), ignorenulls=True).alias(col_name))

    delete_key_view = f"tmp_ppa_delete_keys_{source.replace(' ', '_').lower()}"
    candidate_df.select(*(group_keys + ['LoadDate'])).distinct().createOrReplaceTempView(delete_key_view)

    normalized_df = candidate_df.groupBy(*group_keys).agg(*agg_exprs)

    normalized_df = (normalized_df
        .withColumn('PaidSource', F.lit(target))
        .withColumn('ClaimLineDetailId', F.lit(-101))
        .withColumn('ClaimLineSequenceNumber', F.lit(1))
        .withColumn('BaseClaimProviderPaymentAdjustedAmount', decimal_zero)
        .withColumn('PayClaimProviderPaymentAdjustedAmount', decimal_zero)
        .withColumn('USD_ClaimProviderPaymentAdjustedAmount', decimal_zero)
        .withColumn('BaseClaimMemberPaymentAdjustedAmount', decimal_zero)
        .withColumn('PayClaimMemberPaymentAdjustedAmount', decimal_zero)
        .withColumn('USD_ClaimMemberPaymentAdjustedAmount', decimal_zero)
    )

    if side == 'member':
        normalized_df = (normalized_df
            .withColumn('BaseProviderPaidAmount', decimal_zero)
            .withColumn('PayProviderPaidAmount', decimal_zero)
            .withColumn('USD_ProviderPaidAmount', decimal_zero)
        )
    else:
        normalized_df = (normalized_df
            .withColumn('BaseMemberPaidAmount', decimal_zero)
            .withColumn('PayMemberPaidAmount', decimal_zero)
            .withColumn('USD_MemberPaidAmount', decimal_zero)
        )

    for field in tmp_fact_schema.fields:
        normalized_df = normalized_df.withColumn(field.name, F.col(field.name).cast(field.dataType))

    normalized_df = normalized_df.select(*tmp_fact_cols).cache()
    normalized_count = normalized_df.count()
    normalized_df.write.format('delta').mode('append').saveAsTable(tmp_fact_table)
    normalized_df.unpersist()

    spark.sql(f"""
        MERGE INTO {tmp_fact_table} AS tgt
        USING {delete_key_view} AS src
        ON tgt.PaidSource = '{source}'
       AND tgt.ClaimLineDetailId <> -101
       AND tgt.ClaimHeaderId = src.ClaimHeaderId
       AND tgt.ClaimDetailId = src.ClaimDetailId
       AND tgt.ClaimRunId = src.ClaimRunId
       AND tgt.ClaimDetailRunId = src.ClaimDetailRunId
       AND tgt.JournalEntryId = src.JournalEntryId
       AND tgt.PaymentId = src.PaymentId
       AND tgt.LoadDate = src.LoadDate
        WHEN MATCHED THEN DELETE
    """)

    print(f'[OK] Normalized {candidate_count} {source} line-level PPA rows into {normalized_count} {target} aggregate rows')

for ppa_rule in ppa_normalization_rules:
    normalize_ppa_rule(ppa_rule)

spark.sql("""
    UPDATE Silver.TmpFactClaimPayment
    SET
        BaseClaimProviderPaymentAdjustedAmount = CAST(0 AS DOUBLE),
        PayClaimProviderPaymentAdjustedAmount  = CAST(0 AS DOUBLE),
        USD_ClaimProviderPaymentAdjustedAmount  = CAST(0 AS DOUBLE)
    WHERE PaidSource = 'PAID PROVIDER'
      AND JournalEntryId IS NOT NULL
""")

spark.sql("""
    MERGE INTO Silver.TmpFactClaimPayment AS tgt
    USING (
        SELECT
            RunId,
            DetailRunId,
            ClaimDetailId,
            ClaimLineDetailId,
            SUM(CAST(adj_base AS DOUBLE)) AS AdjBaseAmount,
            SUM(CAST(adj_pay AS DOUBLE)) AS AdjPayAmount,
            SUM(CAST(paid_base AS DOUBLE)) AS PaidBaseAmount,
            SUM(CAST(paid_pay AS DOUBLE)) AS PaidPayAmount
        FROM vw_ProviderPPAAdjustment
        GROUP BY RunId, DetailRunId, ClaimDetailId, ClaimLineDetailId
    ) AS adj
    ON tgt.PaidSource = 'PAID PROVIDER'
   AND tgt.JournalEntryId IS NOT NULL
   AND tgt.ClaimRunId = adj.RunId
   AND tgt.ClaimDetailRunId = adj.DetailRunId
   AND tgt.ClaimDetailId = adj.ClaimDetailId
   AND tgt.ClaimLineDetailId = adj.ClaimLineDetailId
   AND ABS(adj.AdjPayAmount) <= ABS(adj.PaidPayAmount)
    WHEN MATCHED THEN UPDATE SET
        tgt.BaseClaimProviderPaymentAdjustedAmount = adj.AdjBaseAmount,
        tgt.PayClaimProviderPaymentAdjustedAmount  = adj.AdjPayAmount,
        tgt.USD_ClaimProviderPaymentAdjustedAmount  = CAST(COALESCE(adj.AdjPayAmount * tgt.PayCurrencyToUSDXChangeRate, adj.AdjBaseAmount) AS DOUBLE)
""")
print('[OK] Regular PAID PROVIDER adjusted amounts re-aligned after PPA normalization')

spark.sql("""
    UPDATE Silver.TmpFactClaimPayment
    SET
        BaseClaimMemberPaymentAdjustedAmount = CAST(0 AS DOUBLE),
        PayClaimMemberPaymentAdjustedAmount  = CAST(0 AS DOUBLE),
        USD_ClaimMemberPaymentAdjustedAmount  = CAST(0 AS DOUBLE)
    WHERE PaidSource = 'PAID MEMBER'
      AND JournalEntryId IS NOT NULL
""")

spark.sql("""
    MERGE INTO Silver.TmpFactClaimPayment AS tgt
    USING (
        SELECT
            RunId,
            DetailRunId,
            ClaimDetailId,
            ClaimLineDetailId,
            SUM(CAST(adj_base AS DOUBLE)) AS AdjBaseAmount,
            SUM(CAST(adj_pay AS DOUBLE)) AS AdjPayAmount,
            SUM(CAST(paid_base AS DOUBLE)) AS PaidBaseAmount,
            SUM(CAST(paid_pay AS DOUBLE)) AS PaidPayAmount
        FROM vw_MemberPPAAdjustment
        GROUP BY RunId, DetailRunId, ClaimDetailId, ClaimLineDetailId
    ) AS adj
    ON tgt.PaidSource = 'PAID MEMBER'
   AND tgt.JournalEntryId IS NOT NULL
   AND tgt.ClaimRunId = adj.RunId
   AND tgt.ClaimDetailRunId = adj.DetailRunId
   AND tgt.ClaimDetailId = adj.ClaimDetailId
   AND tgt.ClaimLineDetailId = adj.ClaimLineDetailId
   AND ABS(adj.AdjPayAmount) <= ABS(adj.PaidPayAmount)
    WHEN MATCHED THEN UPDATE SET
        tgt.BaseClaimMemberPaymentAdjustedAmount = adj.AdjBaseAmount,
        tgt.PayClaimMemberPaymentAdjustedAmount  = adj.AdjPayAmount,
        tgt.USD_ClaimMemberPaymentAdjustedAmount  = CAST(COALESCE(adj.AdjPayAmount * tgt.PayCurrencyToUSDXChangeRate, adj.AdjBaseAmount) AS DOUBLE)
""")
print('[OK] Regular PAID MEMBER adjusted amounts re-aligned after PPA normalization')


In [ ]:
# ============================================================
# Cell: PayCurrencyToUSDXChangeRate UPDATE ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â usp_DataPrepare_FactClaimCostData
# Corrige USD amounts cuando PayCurrencyToUSDXChangeRate es NULL
# Usa AMP_Generic_XChangeRate por CurrencyId + XChangeDate
# Dedup: ROW_NUMBER para evitar multiple source rows matching
# ============================================================
spark.sql("""
    MERGE INTO Silver.TmpFactClaimPayment AS t
    USING (
        SELECT ClaimRunId, ClaimDetailRunId, ClaimDetailId, ClaimLineDetailId, PaidSource, NewRate
        FROM (
            SELECT s.ClaimRunId, s.ClaimDetailRunId, s.ClaimDetailId, s.ClaimLineDetailId, s.PaidSource,
                   CAST(xr.XChangeRate AS DOUBLE) AS NewRate,
                   ROW_NUMBER() OVER (
                       PARTITION BY s.ClaimRunId, s.ClaimDetailRunId, s.ClaimDetailId, s.ClaimLineDetailId, s.PaidSource
                       ORDER BY xr.XChangeDate DESC
                   ) AS rn
            FROM Silver.TmpFactClaimPayment s
            INNER JOIN Bronze.AmigosPlus_AMP_Generic_XChangeRate xr
                ON xr.CurrencyId = CASE
                    WHEN s.PaidSource IN ('PAID PROVIDER','VOID PROVIDER','PPA PROVIDER','VOID PPA PROVIDER')
                    THEN s.ClaimPaymentCurrencyToProviderId
                    ELSE s.ClaimPaymentCurrencyToMemberId END
                AND xr.XChangeDate = s.ClaimServiceFromDate
            WHERE s.PayCurrencyToUSDXChangeRate IS NULL
        )
        WHERE rn = 1
    ) AS src
    ON  t.ClaimRunId = src.ClaimRunId
    AND t.ClaimDetailRunId = src.ClaimDetailRunId
    AND t.ClaimDetailId = src.ClaimDetailId
    AND t.ClaimLineDetailId = src.ClaimLineDetailId
    AND t.PaidSource = src.PaidSource
    WHEN MATCHED AND t.PayCurrencyToUSDXChangeRate IS NULL THEN UPDATE SET
        t.PayCurrencyToUSDXChangeRate = src.NewRate,
        t.USD_MemberPaidAmount = CASE
            WHEN t.PaidSource IN ('PAID MEMBER','VOID MEMBER','PPA MEMBER','VOID PPA MEMBER')
            THEN t.PayMemberPaidAmount * src.NewRate
            ELSE t.USD_MemberPaidAmount END,
        t.USD_ProviderPaidAmount = CASE
            WHEN t.PaidSource IN ('PAID PROVIDER','VOID PROVIDER','PPA PROVIDER','VOID PPA PROVIDER')
            THEN t.PayProviderPaidAmount * src.NewRate
            ELSE t.USD_ProviderPaidAmount END,
        t.USD_TotalPaidAmount = CASE
            WHEN t.PaidSource IN ('PAID MEMBER','VOID MEMBER','PPA MEMBER','VOID PPA MEMBER')
            THEN t.PayMemberPaidAmount * src.NewRate
            ELSE t.PayProviderPaidAmount * src.NewRate END,
        t.USD_ClaimMemberPaymentAdjustedAmount = CASE
            WHEN t.PaidSource IN ('PAID MEMBER','VOID MEMBER','PPA MEMBER','VOID PPA MEMBER')
            THEN t.PayClaimMemberPaymentAdjustedAmount * src.NewRate
            ELSE t.USD_ClaimMemberPaymentAdjustedAmount END,
        t.USD_ClaimProviderPaymentAdjustedAmount = CASE
            WHEN t.PaidSource IN ('PAID PROVIDER','VOID PROVIDER','PPA PROVIDER','VOID PPA PROVIDER')
            THEN t.PayClaimProviderPaymentAdjustedAmount * src.NewRate
            ELSE t.USD_ClaimProviderPaymentAdjustedAmount END
""")
print("[OK] PayCurrencyToUSDXChangeRate updated from AMP_Generic_XChangeRate")

In [ ]:
# ============================================================
# Pre-cache dimension tables for broadcast joins
# Las dimensiones son tablas pequeÃƒÆ’Ã‚Â±as ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â forzar broadcast para
# evitar shuffle en los JOINs de resoluciÃƒÆ’Ã‚Â³n de SKeys.
#
# Si una dim aÃƒÆ’Ã‚Âºn no existe en Gold, se crea un stub vacÃƒÆ’Ã‚Â­o con
# el schema esperado para que los LEFT JOIN resuelvan a NULL
# ÃƒÂ¢Ã¢â‚¬Â Ã¢â‚¬â„¢ COALESCE(-101) sin romper la query.
# ============================================================
from pyspark.sql.functions import broadcast

dim_tables = [
    # ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Dimensiones existentes ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
    ("Gold.bgla_DimStatus",              "vw_DimStatus_bc"),
    ("Gold.bgla_DimClaimReason",          "vw_DimClaimReason_bc"),
    ("Gold.bgla_DimDate",                 "vw_DimDate_bc"),
    ("Gold.bgla_DimPolicyMember",         "vw_DimPolicyMember_bc"),
    ("Gold.bgla_DimProvider",             "vw_DimProvider_bc"),
    ("Gold.bgla_DimSchedule",             "vw_DimSchedule_bc"),
    ("Gold.bgla_DimServiceIndicator",     "vw_DimServiceIndicator_bc"),
    ("Gold.bgla_DimCurrency",             "vw_DimCurrency_bc"),
    ("Gold.bgla_DimGeography",            "vw_DimGeography_bc"),
    ("Gold.bgla_DimPolicy",               "vw_DimPolicy_bc"),
    ("Gold.bgla_DimInsuranceBusiness",    "vw_DimInsuranceBusiness_bc"),
    ("Gold.bgla_DimServiceNetwork",       "vw_DimServiceNetwork_bc"),
    # ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Dimensiones agregadas (replican usp_DataPrepare_FactClaimCostData) ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
    ("Gold.bgla_DimEntity",               "vw_DimEntity_bc"),
    ("Gold.bgla_DimProduct",              "vw_DimProduct_bc"),
    ("Gold.bgla_DimGroup",                "vw_DimGroup_bc"),
    ("Gold.bgla_DimFamilyType",           "vw_DimFamilyType_bc"),
    ("Gold.bgla_DimPlan",                 "vw_DimPlan_bc"),
    ("Gold.bgla_DimBusinessMode",         "vw_DimBusinessMode_bc"),
    ("Gold.bgla_DimBusinessType",         "vw_DimBusinessType_bc"),
    ("Gold.bgla_DimCompany",              "vw_DimCompany_bc"),
    ("Gold.bgla_DimModeOfPayment",        "vw_DimModeOfPayment_bc"),
    ("Gold.bgla_DimPaymentMethod",        "vw_DimPaymentMethod_bc"),
    ("Gold.bgla_DimReceivedMethod",       "vw_DimReceivedMethod_bc"),
    ("Gold.bgla_DimAgent",                "vw_DimAgent_bc"),
    ("Gold.bgla_DimRegion",               "vw_DimRegion_bc"),
    ("Gold.bgla_DimAccumulatorScope",     "vw_DimAccumulatorScope_bc"),
    ("Gold.bgla_DimBillType",             "vw_DimBillType_bc"),
    ("Gold.bgla_DimPayTo",                "vw_DimPayTo_bc"),
    ("Gold.bgla_DimPayee",                "vw_DimPayee_bc"),
    ("Gold.bgla_DimRunType",              "vw_DimRunType_bc"),
    ("Gold.bgla_DimPaymentVoidReason",    "vw_DimPaymentVoidReason_bc"),
    ("Gold.bgla_DimThirdParty",           "vw_DimThirdParty_bc"),
    ("Gold.bgla_DimDiagnostic",           "vw_DimDiagnostic_bc"),
    ("Gold.bgla_DimProcedure",            "vw_DimProcedure_bc"),
    ("Gold.bgla_DimTypeOfService",        "vw_DimTypeOfService_bc"),
    ("Gold.bgla_DimPlaceOfService",       "vw_DimPlaceOfService_bc"),
]

# Schema esperado por dim ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â usado como stub si la tabla no existe en Gold.
# Tipos siguen el patrÃƒÆ’Ã‚Â³n Fabric: SKeys son BIGINT (mirror del Id natural).
dim_stub_schema = {
    "vw_DimStatus_bc":              "StatusId BIGINT, StatusSKey BIGINT",
    "vw_DimClaimReason_bc":         "ClaimReasonId BIGINT, ClaimReasonSKey BIGINT",
    "vw_DimDate_bc":                "Date DATE, Datekey BIGINT",
    "vw_DimPolicyMember_bc":        "PolicyMemberEligibilityId BIGINT, PolicyMemberSKey BIGINT",
    "vw_DimProvider_bc":            "ProviderId BIGINT, ProviderCategory STRING, ProviderSKey BIGINT",
    "vw_DimSchedule_bc":            "ScheduleId BIGINT, ScheduleSKey BIGINT",
    "vw_DimServiceIndicator_bc":    "ServiceIndicatorId BIGINT, ServiceIndicatorSKey BIGINT",
    "vw_DimCurrency_bc":            "CurrencyId BIGINT, CurrencySKey BIGINT",
    "vw_DimGeography_bc":           "GlobalLocationId BIGINT, CountryId BIGINT, StateId BIGINT, CityId BIGINT, GeographySKey BIGINT",
    "vw_DimPolicy_bc":              "PolicyId BIGINT, ToDate DATE, PolicySKey BIGINT",
    "vw_DimInsuranceBusiness_bc":   "InsuranceBusinessId BIGINT, ToDate DATE, InsuranceBusinessSKey BIGINT",
    "vw_DimServiceNetwork_bc":      "ServiceNetworkDescripcion STRING, ServiceNetworkSKey BIGINT",
    # Dimensiones agregadas
    "vw_DimEntity_bc":              "EntityId BIGINT, EntitySKey BIGINT",
    "vw_DimProduct_bc":             "ProductId BIGINT, ProductSKey BIGINT",
    "vw_DimGroup_bc":               "GroupId BIGINT, GroupSKey BIGINT",
    "vw_DimFamilyType_bc":          "FamilyTypeId BIGINT, FamilyTypeSKey BIGINT",
    "vw_DimPlan_bc":                "PlanId BIGINT, PlanSKey BIGINT",
    "vw_DimBusinessMode_bc":        "BusinessModeId BIGINT, BusinessModeSKey BIGINT",
    "vw_DimBusinessType_bc":        "BusinessTypeId BIGINT, BusinessTypeSKey BIGINT",
    "vw_DimCompany_bc":             "CompanyId BIGINT, CompanySKey BIGINT",
    "vw_DimModeOfPayment_bc":       "ModeOfPaymentId BIGINT, ModeOfPaymentSKey BIGINT",
    "vw_DimPaymentMethod_bc":       "PaymentMethodId BIGINT, PaymentMethodSKey BIGINT",
    "vw_DimReceivedMethod_bc":      "ReceivedMethodId BIGINT, ReceivedMethodSKey BIGINT",
    "vw_DimAgent_bc":               "AgentHierarchyId BIGINT, AgentSKey BIGINT",
    "vw_DimRegion_bc":              "RegionId BIGINT, RegionSKey BIGINT",
    "vw_DimAccumulatorScope_bc":    "AccumulatorScopeId BIGINT, AccumulatorScopeSKey BIGINT",
    "vw_DimBillType_bc":            "BillTypeId BIGINT, BillTypeSKey BIGINT",
    "vw_DimPayTo_bc":               "PayToId BIGINT, PayToSKey BIGINT",
    "vw_DimPayee_bc":               "PayeeId BIGINT, PayeeSKey BIGINT",
    "vw_DimRunType_bc":             "RunTypeId BIGINT, RunTypeSKey BIGINT",
    "vw_DimPaymentVoidReason_bc":   "PaymentVoidReasonId BIGINT, PaymentVoidReasonSKey BIGINT",
    "vw_DimThirdParty_bc":          "ThirdPartyId BIGINT, ThirdPartySKey BIGINT",
    "vw_DimDiagnostic_bc":          "DiagnosticId BIGINT, DiagnosticSKey BIGINT",
    "vw_DimProcedure_bc":           "ProcedureId BIGINT, ProcedureSKey BIGINT",
    "vw_DimTypeOfService_bc":       "TypeOfServiceId BIGINT, TypeOfServiceSKey BIGINT",
    "vw_DimPlaceOfService_bc":      "PlaceOfServiceId BIGINT, PlaceOfServiceSKey BIGINT",
}

missing_dims = []
for table_name, view_name in dim_tables:
    try:
        df = spark.table(table_name)
        broadcast(df).createOrReplaceTempView(view_name)
    except Exception:
        schema = dim_stub_schema.get(view_name)
        if schema:
            empty_df = spark.createDataFrame([], schema)
            broadcast(empty_df).createOrReplaceTempView(view_name)
            missing_dims.append(table_name)

if missing_dims:
    print(f"  ÃƒÂ¢Ã…Â¡Ã‚Â ÃƒÂ¯Ã‚Â¸Ã‚Â Dimensiones no encontradas (stubs vacÃƒÆ’Ã‚Â­os creados): {missing_dims}")
print(f"[OK] {len(dim_tables)} dimension tables pre-cached for broadcast joins")


In [ ]:
# ============================================================
# Schema migration ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â agregar columnas SKey faltantes en Gold.bgla_FactClaimPayment
# Replica las dimensiones de usp_DataPrepare_FactClaimCostData que NO estaban
# materializadas en el fact (Policy*SKey, ClaimRun*SKey, ClaimDiagnostic*SKey).
#
# ALTER TABLE ADD COLUMNS IF NOT EXISTS ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â idempotente.
# ============================================================
new_skey_columns = [
    # Policy dims (vienen de pp / vw_PoliciesAndProducts)
    ("PolicyEntitySKey",                 "BIGINT"),
    ("PolicyProductSKey",                "BIGINT"),
    ("PolicyGroupSKey",                  "BIGINT"),
    ("PolicyFamilyTypeSKey",             "BIGINT"),
    ("PolicyPlanSKey",                   "BIGINT"),
    ("PolicyBusinessModeSKey",           "BIGINT"),
    ("PolicyBusinessTypeSKey",           "BIGINT"),
    ("PolicyCompanySKey",                "BIGINT"),
    ("PolicyInsuranceBusinessSKey",      "BIGINT"),
    ("PolicyModeOfPaymentSKey",          "BIGINT"),
    ("PolicyPaymentMethodSKey",          "BIGINT"),
    ("PolicyReceivedMethodSKey",         "BIGINT"),
    ("PolicyRegionSKey",                 "BIGINT"),
    # Claim dims faltantes
    ("ClaimPrimaryDiagnosticSKey",       "BIGINT"),
    ("ClaimPrimaryProcedureSKey",        "BIGINT"),
    # Run dims faltantes
    ("ClaimRunTypeSKey",                 "BIGINT"),
    ("ClaimRunStatusSKey",               "BIGINT"),
    ("ClaimRunPaymentMethodSKey",        "BIGINT"),
]

existing_cols = {f.name.lower() for f in spark.table("Gold.bgla_FactClaimPayment").schema.fields}
to_add = [(c, t) for c, t in new_skey_columns if c.lower() not in existing_cols]

if to_add:
    cols_ddl = ", ".join([f"{c} {t}" for c, t in to_add])
    spark.sql(f"ALTER TABLE Gold.bgla_FactClaimPayment ADD COLUMNS ({cols_ddl})")
    print(f"[OK] {len(to_add)} SKey columns added to Gold.bgla_FactClaimPayment: {[c for c,_ in to_add]}")
else:
    print("[OK] All SKey columns already present in Gold.bgla_FactClaimPayment")


## Columnas Nuevas Agregadas (2026-04-29)

Total: **28 columnas nuevas** para alinear con `schema.md` (191 columnas totales).

### Grupo A ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â 11 columnas nuevas (PPA / VAT / Withholding / ServiceNetwork)

| # | Columna | Tipo | DescripciÃƒÆ’Ã‚Â³n |
|---|---------|------|-------------|
| 1 | `ServiceNetworkSKey` | INT | Surrogate key de DimServiceNetwork (JOIN por ServiceNetworkDescripcion) |
| 2 | `ClaimProviderPPABalanceBaseCurrenty` | DECIMAL(18,2) | Balance PPA Provider en moneda base |
| 3 | `ClaimProviderPPABalancePaymentCurrency` | DECIMAL(18,2) | Balance PPA Provider en moneda de pago |
| 4 | `ClaimMemberPPABalanceBaseCurrenty` | DECIMAL(18,2) | Balance PPA Member en moneda base |
| 5 | `ClaimMemberPPABalancePaymentCurrency` | DECIMAL(18,2) | Balance PPA Member en moneda de pago |
| 6 | `ClaimProviderVoidPPABalanceBaseCurrenty` | DECIMAL(18,2) | Void PPA Provider en moneda base |
| 7 | `ClaimProviderVoidPPABalancePaymentCurrency` | DECIMAL(18,2) | Void PPA Provider en moneda de pago |
| 8 | `ClaimMemberVoidPPABalanceBaseCurrenty` | DECIMAL(18,2) | Void PPA Member en moneda base |
| 9 | `ClaimMemberVoidPPABalancePaymentCurrency` | DECIMAL(18,2) | Void PPA Member en moneda de pago |
| 10 | `VATPaymentCurrency` | DECIMAL(18,2) | IVA convertido a moneda de pago |
| 11 | `WithholdingPaymentCurrency` | DECIMAL(18,2) | RetenciÃƒÆ’Ã‚Â³n convertida a moneda de pago |

### Grupo B ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â 17 columnas adicionales para alinear con schema.md

Estas se derivan de columnas `Id` ya existentes en staging (alias `SKey`) o se calculan segÃƒÆ’Ã‚Âºn la lÃƒÆ’Ã‚Â³gica del SP `usp_DataPrepare_FactClaimCostData`:

| # | Columna | Tipo | Origen / CÃƒÆ’Ã‚Â¡lculo |
|---|---------|------|------------------|
| 12 | `PolicyId` | INT | Pass-through directo desde staging |
| 13 | `Source` | VARCHAR | `'Amigos+'` (default segÃƒÆ’Ã‚Âºn SP `sourcesystem`) |
| 14 | `SavingAmount` | DECIMAL | `BaseAllowedAmount - BaseTotalPaidAmount` |
| 15 | `ClaimNetworkSkey` | INT | `-101` (sin DimNetwork por ahora) |
| 16 | `ClaimAccumulatorScopeSKey` | INT | Alias de `ClaimAccumulatorScopeId` |
| 17 | `ClaimBillTypeSKey` | INT | Alias de `ClaimBillTypeId` |
| 18 | `ClaimPayToSkey` | INT | Alias de `ClaimPayToId` |
| 19 | `ClaimPayeeSkey` | INT | Alias de `ClaimPayeeId` |
| 20 | `ClaimReceivedMethodSKey` | INT | Alias de `ClaimReceivedMethodId` |
| 21 | `PaymentStatusSKey` | INT | Alias de `PaymentStatusId` |
| 22 | `PaymentVoidReasonSkey` | INT | Alias de `PaymentVoidReasonId` |
| 23 | `PaymentThirdPartySKey` | INT | Alias de `ClaimPaymentThirdPartyId` |
| 24 | `PolicyAgentSKey` | INT | Alias de `PolicyAgentHierarchyId` |
| 25 | `PolicyMemberOwnerEligibilitySKey` | INT | Alias de `PolicyMemberOwnerId` |
| 26 | `PolicyGeographySKey` | INT | `-101` (composite ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â pendiente DimGeography lookup) |
| 27 | `ClaimTypeOfServiceSKey` | INT | Alias de `ClaimTOSId` |
| 28 | `ClaimPlaceOfServiceSKey` | INT | Alias de `ClaimPOSId` |

> **Nota**: Las columnas tipo "alias SKey" siguen el patrÃƒÆ’Ã‚Â³n del SP de SQL Server donde `[XxxSKey_YyyId] = isnull(cd.YyyId, -1)`, es decir, el SKey toma el mismo valor que el Id natural.

### LÃƒÆ’Ã‚Â³gica de PoblaciÃƒÆ’Ã‚Â³n por PaidSource (Grupo A):
- **PAID MEMBER/PROVIDER**: PPA columns en 0 (se calculan en PPA inserts)
- **PPA PROVIDER**: `ClaimProviderPPABalance*` = PPA.BaseAmount/PaidAmount; resto 0
- **PPA MEMBER**: `ClaimMemberPPABalance*` = PPA.BaseAmount/PaidAmount; resto 0
- **VOID PPA PROVIDER**: `ClaimProviderVoidPPABalance*` = PPA.BaseAmount/PaidAmount; resto 0
- **VOID PPA MEMBER**: `ClaimMemberVoidPPABalance*` = PPA.BaseAmount/PaidAmount; resto 0
- **VATPaymentCurrency / WithholdingPaymentCurrency**: Calculados con Factor ÃƒÆ’Ã¢â‚¬â€ Base ÃƒÆ’Ã¢â‚¬â€ XChangeRate

In [ ]:
# ============================================================
# Cell 24: Create final view with surrogate key resolution
# OPTIMIZACIÃƒÆ’Ã¢â‚¬Å“N: Usa broadcast views de dimensiones para evitar shuffle
# Replica TODAS las dimensiones de usp_DataPrepare_FactClaimCostData
# ============================================================
df_registros = spark.sql("""
    SELECT /*+ BROADCAST(DimStatus, DimClaimReason, DimDate_StatusReason, DimMember,
                          DimProvider_Billing, DimProvider_Service, DimSchedule,
                          DimServiceIndicator_Detail, DimServiceIndicator_Line,
                          DimDate_ServiceFrom, DimCurrency_Local, DimCurrency_Base,
                          DimDate_Received, DimGeography, DimPolicy,
                          DimInsuranceBusiness_Run, DimDate_RunCreated, DimDate_RunPrinted,
                          DimDate_PaymentPrinted, DimDate_PaymentPosted, DimDate_PaymentVoid,
                          DimCurrency_PayMember, DimCurrency_PayProvider, DimServiceNetwork,
                          DimEntity, DimProduct, DimGroup, DimFamilyType, DimPlan,
                          DimBusinessMode, DimBusinessType, DimCompany, DimInsuranceBusiness_Policy,
                          DimModeOfPayment, DimPaymentMethod, DimReceivedMethod_Policy,
                          DimAgent, DimRegion, DimMemberOwner, DimAccumulatorScope, DimBillType,
                          DimPayTo, DimPayee, DimReceivedMethod_Claim, DimRunType,
                          DimStatus_Run, DimPaymentMethod_Run, DimStatus_Payment,
                          DimPaymentVoidReason, DimThirdParty, DimDiagnostic, DimProcedure,
                          DimTOS, DimPOS) */
        UUID() AS FactClaimPaymentSKey,
        TMP.ClaimHeaderId,
        TMP.ClaimNumber,
        SUBSTRING(COALESCE(TMP.ClaimReferenceNumber, '_Undefined'), 1, 100) AS ClaimReferenceNumber,
        TMP.ClaimDetailId,
        TMP.ClaimVersion,
        TMP.ControlVersion,
        TMP.ClaimLineDetailId,
        TMP.ClaimLineSequenceNumber,
        TMP.ClamReferralHeaderId,
        SUBSTRING(COALESCE(TMP.ClaimUUID, '_Undefined'), 1, 100) AS ClaimUUID,
        -- Dimension SKey lookups (using broadcast views)
        COALESCE(DimStatus.StatusSKey, -101) AS ClaimStatusSKey,
        COALESCE(DimClaimReason.ClaimReasonSKey, -101) AS ClaimReasonSKey,
        COALESCE(DimDate_StatusReason.Datekey, -101) AS ClaimStatusReasonDatekey,
        COALESCE(DimMember.PolicyMemberSKey, -101) AS ClaimMemberEligibilitySKey,
        COALESCE(DimProvider_Billing.ProviderSKey, -101) AS ClaimBillingProviderSkey,
        TMP.ClaimBillingProviderCategory,
        COALESCE(DimProvider_Service.ProviderSKey, -101) AS ClaimServiceProviderSKey,
        TMP.ClaimServiceProviderCategory,
        COALESCE(DimSchedule.ScheduleSKey, -101) AS ClaimScheduleSKey,
        COALESCE(DimServiceIndicator_Detail.ServiceIndicatorSKey, -101) AS ClaimDetailServiceindicatorSkey,
        COALESCE(DimServiceIndicator_Line.ServiceIndicatorSKey, -101) AS ClaimLineDetailServiceindicatorSkey,
        COALESCE(DimDate_ServiceFrom.Datekey, -101) AS ClaimServiceFromDateKey,
        TMP.ClaimServiceFromDate,
        TMP.ClaimServiceToDate,
        COALESCE(DimCurrency_Local.CurrencySKey, -101) AS ClaimLocalCurrencySKey,
        TMP.ClaimLocalXChangeRate,
        TMP.ClaimLocalXChangeDate,
        COALESCE(DimCurrency_Base.CurrencySKey, -101) AS ClaimBaseCurrencySkey,
        COALESCE(DimDate_Received.Datekey, -101) AS ClaimReceivedDateKey,
        COALESCE(DimGeography.GeographySKey, -101) AS ClaimGeographySKey,
        TMP.ClaimAccumulatorScopeId,
        TMP.ClaimBillTypeId,
        TMP.ClaimPayToId,
        TMP.ClaimPayeeId,
        COALESCE(DimPolicy.PolicySKey, -101) AS PolicySKey,
        -- Policy Dimension business keys (Id) ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â SKeys agregadas mÃƒÆ’Ã‚Â¡s abajo
        TMP.PolicyModeOfPaymentId,
        TMP.PolicyPaymentMethodId,
        TMP.PolicyReceivedMethodId,
        TMP.PolicyGroupId,
        TMP.PolicyFamilyTypeId,
        TMP.PolicyProductId,
        TMP.PolicyEntityId,
        TMP.PolicyPlanId,
        TMP.PolicyBusinessModeId,
        TMP.PolicyBusinessTypeId,
        TMP.PolicyInsuranceBusinessId,
        TMP.PolicyCompanyId,
        COALESCE(DimDate_PolicyIssue.Datekey, -101) AS PolicyIssueDateKey,
        COALESCE(DimDate_PolicyRenewal.Datekey, -101) AS PolicyRenewalDateKey,
        COALESCE(DimDate_PolicyAnniversary.Datekey, -101) AS PolicyAnniversaryDateKey,
        COALESCE(DimDate_PolicyApplicationReceived.Datekey, -101) AS PolicyApplicationReceivedDateKey,
        COALESCE(DimDate_PolicyEffective.Datekey, -101) AS PolicyEffectiveDateKey,
        COALESCE(DimDate_PolicyCancel.Datekey, -101) AS PolicyCancelDateKey,
        COALESCE(DimDate_PolicyDeathBenefit.Datekey, -101) AS PolicyDeathBenefitperiodDateKey,
        TMP.PolicyAgentHierarchyId,
        TMP.PolicyRegionId,
        TMP.PolicyMemberOwnerId,
        TMP.PolicyGlobalLocationId,
        TMP.PolicyCountryId,
        TMP.PolicyStateId,
        TMP.PolicyCityId,
        -- Run SKeys
        TMP.ClaimRunId,
        TMP.ClaimDetailRunId,
        COALESCE(DimInsuranceBusiness_Run.InsuranceBusinessSKey, -101) AS ClaimRunInsuranceBusinessSKey,
        TMP.ClaimRunTypeId,
        COALESCE(DimDate_RunCreated.Datekey, -101) AS ClaimRunCreatedOnDateSkey,
        COALESCE(DimDate_RunPrinted.Datekey, -101) AS ClaimRunPrintedOnDateSkey,
        TMP.ClaimRunStatusId,
        TMP.ClaimRunPaymentMethodId,
        TMP.ClaimPrimaryDiagnosticId,
        TMP.ClaimPrimaryProcedureId,
        TMP.ClaimDiagnosticType,
        TMP.ClaimPOSId,
        TMP.ClaimTOSId,
        TMP.PaymentId,
        TMP.PaymentNumber,
        COALESCE(DimDate_PaymentPrinted.Datekey, -101) AS PaymentPrintedDateKey,
        TMP.PaymentStatusId,
        TMP.PaymentVoidReasonId,
        COALESCE(DimDate_PaymentPosted.Datekey, -101) AS PaymentPostedDateKey,
        COALESCE(DimDate_PaymentVoid.Datekey, -101) AS PaymentVoidDateSKey,
        TMP.PaymentBankId,
        TMP.PaymentGatewayTransactionId,
        TMP.PaymentAmount,
        TMP.PaymentForeignAmount,
        TMP.PaymentBankTransmissionId,
        TMP.JournalEntryId,
        TMP.JournalNotes,
        TMP.VoidJournalEntryId,
        TMP.VoidJournalNotes,
        COALESCE(DimCurrency_PayMember.CurrencySKey, -101) AS ClaimMemberPaymentCurrencySkey,
        TMP.ClaimPaymentToMemberXchangeRate,
        COALESCE(DimCurrency_PayProvider.CurrencySKey, -101) AS ClaimProviderPaymentCurrencySkey,
        TMP.ClaimPaymentToProviderXchangeRate,
        TMP.ClaimPaymentThirdPartyId,
        SUBSTRING(COALESCE(TMP.ClaimAccountNumber, '_Undefined'), 1, 100) AS ClaimAccountNumber,
        COALESCE(TMP.LocalBilledAmount, 0.0) AS LocalBilledAmount,
        COALESCE(TMP.BaseAllowedAmount, 0.0) AS BaseAllowedAmount,
        COALESCE(TMP.BaseBilledAmount, 0.0) AS BaseBilledAmount,
        COALESCE(TMP.BaseCoInsuranceAmount, 0.0) AS BaseCoInsuranceAmount,
        COALESCE(TMP.BaseAllowedCmp, 0.0) AS BaseAllowedCmp,
        COALESCE(TMP.BaseCopayAmount, 0.0) AS BaseCopayAmount,
        COALESCE(TMP.BaseDeductibleAmount, 0.0) AS BaseDeductibleAmount,
        COALESCE(TMP.BaseNetCoveredAmount, 0.0) AS BaseNetCoveredAmount,
        COALESCE(TMP.BaseMemberPaidAmount, 0.0) AS BaseMemberPaidAmount,
        COALESCE(TMP.BaseProviderPaidAmount, 0.0) AS BaseProviderPaidAmount,
        COALESCE(TMP.BaseDiscountAmount, 0.0) AS BaseDiscountAmount,
        COALESCE(TMP.BaseTotalIneligibleAmount, 0.0) AS BaseTotalIneligibleAmount,
        COALESCE(TMP.BaseOtherIneligibleAmount, 0.0) AS BaseOtherIneligibleAmount,
        COALESCE(TMP.BaseTotalPaidAmount, 0.0) AS BaseTotalPaidAmount,
        COALESCE(TMP.BaseWithHoldingAmount, 0.0) AS BaseWithHoldingAmount,
        COALESCE(TMP.BaseVATAmount, 0.0) AS BaseVATAmount,
        COALESCE(TMP.PayMemberPaidAmount, 0.0) AS PayMemberPaidAmount,
        COALESCE(TMP.PayProviderPaidAmount, 0.0) AS PayProviderPaidAmount,
        TMP.BaseXChangeRate,
        COALESCE(TMP.USD_AllowedAmount, 0.0) AS USD_AllowedAmount,
        COALESCE(TMP.USD_BilledAmount, 0.0) AS USD_BilledAmount,
        COALESCE(TMP.USD_CoInsuranceAmount, 0.0) AS USD_CoInsuranceAmount,
        COALESCE(TMP.USD_AllowedCmp, 0.0) AS USD_AllowedCmp,
        COALESCE(TMP.USD_CopayAmount, 0.0) AS USD_CopayAmount,
        COALESCE(TMP.USD_DeductibleAmount, 0.0) AS USD_DeductibleAmount,
        COALESCE(TMP.USD_NetCoveredAmount, 0.0) AS USD_NetCoveredAmount,
        COALESCE(TMP.USD_DiscountAmount, 0.0) AS USD_DiscountAmount,
        COALESCE(TMP.USD_TotalIneligibleAmount, 0.0) AS USD_TotalIneligibleAmount,
        COALESCE(TMP.USD_OtherIneligibleAmount, 0.0) AS USD_OtherIneligibleAmount,
        COALESCE(TMP.USD_WithHoldingAmount, 0.0) AS USD_WithHoldingAmount,
        COALESCE(TMP.USD_VATAmount, 0.0) AS USD_VATAmount,
        COALESCE(TMP.USD_MemberPaidAmount, 0.0) AS USD_MemberPaidAmount,
        COALESCE(TMP.USD_ProviderPaidAmount, 0.0) AS USD_ProviderPaidAmount,
        COALESCE(TMP.USD_TotalPaidAmount, 0.0) AS USD_TotalPaidAmount,
        TMP.Factor,
        COALESCE(DimDate_ClaimProcessed.Datekey, -101) AS ClaimProcessedDateSkey,
        TMP.ClaimProcessRowFlag,
        TMP.RunProcessed,
        CAST(NULL AS DATE) AS EffectiveDate,
        TMP.ActualRowFlag,
        TMP.LoadDate,
        SUBSTRING(COALESCE(TMP.PaidSource, '_Undefined'), 1, 100) AS PaidSource,
        COALESCE(TMP.BaseClaimProviderPaymentAdjustedAmount, 0.0) AS BaseClaimProviderPaymentAdjustedAmount,
        COALESCE(TMP.PayClaimProviderPaymentAdjustedAmount, 0.0) AS PayClaimProviderPaymentAdjustedAmount,
        COALESCE(TMP.USD_ClaimProviderPaymentAdjustedAmount, 0.0) AS USD_ClaimProviderPaymentAdjustedAmount,
        TMP.PayCurrencyToUSDXChangeRate,
        COALESCE(TMP.BaseClaimMemberPaymentAdjustedAmount, 0.0) AS BaseClaimMemberPaymentAdjustedAmount,
        COALESCE(TMP.PayClaimMemberPaymentAdjustedAmount, 0.0) AS PayClaimMemberPaymentAdjustedAmount,
        COALESCE(TMP.USD_ClaimMemberPaymentAdjustedAmount, 0.0) AS USD_ClaimMemberPaymentAdjustedAmount,
        SUBSTRING(COALESCE(TMP.ServiceNetwork, '_Undefined'), 1, 100) AS ServiceNetwork,
        SUBSTRING(COALESCE(TMP.Paid_To_Calculated, '_Undefined'), 1, 100) AS Paid_To_Calculated,
        SUBSTRING(COALESCE(TMP.INN_OON, '_Undefined'), 1, 10) AS INN_OON,
        TMP.MemberId,
        COALESCE(TMP.ClaimGap, 0) AS ClaimGap,
        TMP.ClaimReceivedMethodId,
        TMP.PolicyFirstYear,
        TMP.PolicyFirstMonth,
        TMP.FirstClaim,
        TMP.IsDigital,
        SUBSTRING(COALESCE(TMP.IsCOR, '_Non Defined IsCOR'), 1, 100) AS IsCOR,
        SUBSTRING(COALESCE(TMP.LocalTeam, '_Non Defined LocalTeam'), 1, 100) AS LocalTeam,
        TMP.IsFastTrack,
        TMP.HTHFeePaid,
        TMP.HTHFeePercent,
        TMP.IsDeleted,
        SUBSTRING(COALESCE(TMP.ProviderNetworkClassName, '_Undefined'), 1, 200) AS ProviderNetworkClassName,
        TMP.BrackedClaimKey,
        TMP.BrackedMemberKey,
        COALESCE(TMP.IncomeByInsured, 0.0) AS IncomeByInsured,
        COALESCE(TMP.SpendByInsured, 0.0) AS SpendByInsured,
        TMP.ClaimsOutliers,
        SUBSTRING(COALESCE(TMP.SubmissionSenderEmailAddress, '_Undefined'), 1, 250) AS SubmissionSenderEmailAddress,
        SUBSTRING(COALESCE(TMP.SubmissionSenderFullName, '_Undefined'), 1, 250) AS SubmissionSenderFullName,
        SUBSTRING(COALESCE(TMP.SubmissionInboundChannel, '_Undefined'), 1, 100) AS SubmissionInboundChannel,
        SUBSTRING(COALESCE(TMP.SubmissionTrackingNumber, '_Undefined'), 1, 100) AS SubmissionTrackingNumber,
        SUBSTRING(COALESCE(TMP.SubmissionSenderType, '_Undefined'), 1, 100) AS SubmissionSenderType,
        SUBSTRING(COALESCE(TMP.ClaimValidation, '_Undefined'), 1, 100) AS ClaimValidation,
        TMP.BatchLotName,
        TMP.BatchLotFromDate,
        TMP.TransactionLotReceivedDate,
        TMP.TransactionLotNumber,
        TMP.TransactionInvoiceNumber,
        TMP.LotNumber,
        COALESCE(TMP.IsAutomatic, 0) AS IsAutomatic,
        TMP.IsICU,
        TMP.IsEmergency,
        TMP.IsHospital_Emergency,
        TMP.IsChildbirths,
        TMP.IsCesareanSection,
        -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â 11 NEW COLUMNS (added 2026-04-29) ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
        COALESCE(DimServiceNetwork.ServiceNetworkSKey, -101) AS ServiceNetworkSKey,
        COALESCE(TMP.ClaimProviderPPABalanceBaseCurrenty, 0.0) AS ClaimProviderPPABalanceBaseCurrenty,
        COALESCE(TMP.ClaimProviderPPABalancePaymentCurrency, 0.0) AS ClaimProviderPPABalancePaymentCurrency,
        COALESCE(TMP.ClaimMemberPPABalanceBaseCurrenty, 0.0) AS ClaimMemberPPABalanceBaseCurrenty,
        COALESCE(TMP.ClaimMemberPPABalancePaymentCurrency, 0.0) AS ClaimMemberPPABalancePaymentCurrency,
        COALESCE(TMP.ClaimProviderVoidPPABalanceBaseCurrenty, 0.0) AS ClaimProviderVoidPPABalanceBaseCurrenty,
        COALESCE(TMP.ClaimProviderVoidPPABalancePaymentCurrency, 0.0) AS ClaimProviderVoidPPABalancePaymentCurrency,
        COALESCE(TMP.ClaimMemberVoidPPABalanceBaseCurrenty, 0.0) AS ClaimMemberVoidPPABalanceBaseCurrenty,
        COALESCE(TMP.ClaimMemberVoidPPABalancePaymentCurrency, 0.0) AS ClaimMemberVoidPPABalancePaymentCurrency,
        COALESCE(TMP.VATPaymentCurrency, 0.0) AS VATPaymentCurrency,
        COALESCE(TMP.WithholdingPaymentCurrency, 0.0) AS WithholdingPaymentCurrency,
        -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â 17 ADDITIONAL COLUMNS to align with schema.md (191 total) ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
        TMP.PolicyId,
        'Amigos+' AS Source,
        (COALESCE(TMP.BaseAllowedAmount, 0.0) - COALESCE(TMP.BaseTotalPaidAmount, 0.0)) AS SavingAmount,
        CAST(-101 AS INT) AS ClaimNetworkSkey,
        -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â Alias-SKey reemplazados por lookups reales contra Gold.bgla_dim* ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
        COALESCE(DimAccumulatorScope.AccumulatorScopeSKey, -1)        AS ClaimAccumulatorScopeSKey,
        COALESCE(DimBillType.BillTypeSKey, -1)                        AS ClaimBillTypeSKey,
        COALESCE(DimPayTo.PayToSKey, -101)                            AS ClaimPayToSkey,
        COALESCE(DimPayee.PayeeSKey, -101)                            AS ClaimPayeeSkey,
        COALESCE(DimReceivedMethod_Claim.ReceivedMethodSKey, -101)    AS ClaimReceivedMethodSKey,
        COALESCE(DimStatus_Payment.StatusSKey, -101)                  AS PaymentStatusSKey,
        COALESCE(DimPaymentVoidReason.PaymentVoidReasonSKey, -101)    AS PaymentVoidReasonSkey,
        COALESCE(DimThirdParty.ThirdPartySKey, -101)                  AS PaymentThirdPartySKey,
        COALESCE(DimAgent.AgentSKey, -101)                            AS PolicyAgentSKey,
        COALESCE(DimMemberOwner.PolicyMemberSKey, -101)               AS PolicyMemberOwnerEligibilitySKey,
        CAST(-101 AS INT)                                             AS PolicyGeographySKey,
        COALESCE(DimTOS.TypeOfServiceSKey, -101)                      AS ClaimTypeOfServiceSKey,
        COALESCE(DimPOS.PlaceOfServiceSKey, -101)                     AS ClaimPlaceOfServiceSKey,
        -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â Nuevas Policy*SKey (replican usp_DataPrepare_FactClaimCostData) ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
        COALESCE(DimEntity.EntitySKey, -101)                          AS PolicyEntitySKey,
        COALESCE(DimProduct.ProductSKey, -101)                        AS PolicyProductSKey,
        COALESCE(DimGroup.GroupSKey, -101)                            AS PolicyGroupSKey,
        COALESCE(DimFamilyType.FamilyTypeSKey, -101)                  AS PolicyFamilyTypeSKey,
        COALESCE(DimPlan.PlanSKey, -101)                              AS PolicyPlanSKey,
        COALESCE(DimBusinessMode.BusinessModeSKey, -101)              AS PolicyBusinessModeSKey,
        COALESCE(DimBusinessType.BusinessTypeSKey, -101)              AS PolicyBusinessTypeSKey,
        COALESCE(DimCompany.CompanySKey, -101)                        AS PolicyCompanySKey,
        COALESCE(DimInsuranceBusiness_Policy.InsuranceBusinessSKey, -101) AS PolicyInsuranceBusinessSKey,
        COALESCE(DimModeOfPayment.ModeOfPaymentSKey, -101)            AS PolicyModeOfPaymentSKey,
        COALESCE(DimPaymentMethod.PaymentMethodSKey, -101)            AS PolicyPaymentMethodSKey,
        COALESCE(DimReceivedMethod_Policy.ReceivedMethodSKey, -101)   AS PolicyReceivedMethodSKey,
        COALESCE(DimRegion.RegionSKey, -101)                          AS PolicyRegionSKey,
        -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â Nuevas Claim/Run SKey ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
        COALESCE(DimDiagnostic.DiagnosticSKey, -101)                  AS ClaimPrimaryDiagnosticSKey,
        COALESCE(DimProcedure.ProcedureSKey, -101)                    AS ClaimPrimaryProcedureSKey,
        COALESCE(DimRunType.RunTypeSKey, -101)                        AS ClaimRunTypeSKey,
        COALESCE(DimStatus_Run.StatusSKey, -101)                      AS ClaimRunStatusSKey,
        COALESCE(DimPaymentMethod_Run.PaymentMethodSKey, -101)        AS ClaimRunPaymentMethodSKey
    FROM Silver.TmpFactClaimPayment TMP
    LEFT JOIN vw_DimStatus_bc DimStatus ON TMP.ClaimStatusId = DimStatus.StatusId
    LEFT JOIN vw_DimClaimReason_bc DimClaimReason ON TMP.ClaimReasonId = DimClaimReason.ClaimReasonId
    LEFT JOIN vw_DimDate_bc DimDate_StatusReason ON DimDate_StatusReason.Date = TMP.ClaimStatusReasonDate
    LEFT JOIN vw_DimPolicyMember_bc DimMember ON TMP.ClaimMemberEligibilityId = DimMember.PolicyMemberEligibilityId
    LEFT JOIN vw_DimProvider_bc DimProvider_Billing ON TMP.ClaimBillingProviderId = DimProvider_Billing.ProviderId AND DimProvider_Billing.ProviderCategory = 'Billing'
    LEFT JOIN vw_DimProvider_bc DimProvider_Service ON TMP.ClaimServiceProviderId = DimProvider_Service.ProviderId AND DimProvider_Service.ProviderCategory = 'Service'
    LEFT JOIN vw_DimSchedule_bc DimSchedule ON TMP.ClaimScheduleId = DimSchedule.ScheduleId
    LEFT JOIN vw_DimServiceIndicator_bc DimServiceIndicator_Detail ON TMP.ClaimDetailServiceIndicatorId = DimServiceIndicator_Detail.ServiceIndicatorId
    LEFT JOIN vw_DimServiceIndicator_bc DimServiceIndicator_Line ON TMP.ClaimLineDetailServiceIndicatorId = DimServiceIndicator_Line.ServiceIndicatorId
    LEFT JOIN vw_DimDate_bc DimDate_ServiceFrom ON DimDate_ServiceFrom.Date = TMP.ClaimServiceFromDate
    LEFT JOIN vw_DimCurrency_bc DimCurrency_Local ON TMP.ClaimLocalCurrencyId = DimCurrency_Local.CurrencyId
    LEFT JOIN vw_DimCurrency_bc DimCurrency_Base ON TMP.ClaimBaseCurrencyId = DimCurrency_Base.CurrencyId
    LEFT JOIN vw_DimDate_bc DimDate_Received ON DimDate_Received.Date = TMP.ClaimReceivedDate
    LEFT JOIN vw_DimGeography_bc DimGeography ON TMP.ClaimGlobalLocationId = DimGeography.GlobalLocationId AND TMP.ClaimCountryId = DimGeography.CountryId AND TMP.ClaimStateId = DimGeography.StateId AND TMP.ClaimCityId = DimGeography.CityId
    LEFT JOIN vw_DimPolicy_bc DimPolicy ON TMP.PolicyId = DimPolicy.PolicyId AND DimPolicy.ToDate IS NULL
    LEFT JOIN vw_DimInsuranceBusiness_bc DimInsuranceBusiness_Run ON TMP.ClaimRunInsuranceBusinessId = DimInsuranceBusiness_Run.InsuranceBusinessId AND DimInsuranceBusiness_Run.ToDate IS NULL
    LEFT JOIN vw_DimDate_bc DimDate_RunCreated ON DimDate_RunCreated.Date = TMP.ClaimRunCreatedOnDate
    LEFT JOIN vw_DimDate_bc DimDate_RunPrinted ON DimDate_RunPrinted.Date = TMP.ClaimRunPrintedOnDate
    LEFT JOIN vw_DimDate_bc DimDate_PaymentPrinted ON DimDate_PaymentPrinted.Date = TMP.PaymentPrintedDate
    LEFT JOIN vw_DimDate_bc DimDate_PaymentPosted ON DimDate_PaymentPosted.Date = TMP.PaymentPostedDate
    LEFT JOIN vw_DimDate_bc DimDate_PaymentVoid ON DimDate_PaymentVoid.Date = TMP.PaymentVoidDate
    LEFT JOIN vw_DimCurrency_bc DimCurrency_PayMember ON TMP.ClaimPaymentCurrencyToMemberId = DimCurrency_PayMember.CurrencyId
    LEFT JOIN vw_DimCurrency_bc DimCurrency_PayProvider ON TMP.ClaimPaymentCurrencyToProviderId = DimCurrency_PayProvider.CurrencyId
    LEFT JOIN vw_DimServiceNetwork_bc DimServiceNetwork ON TMP.ServiceNetwork = DimServiceNetwork.ServiceNetworkDescripcion
    -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â JOINs para alias-SKey ahora resueltos contra dimensiones reales ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
    LEFT JOIN vw_DimAccumulatorScope_bc DimAccumulatorScope ON TMP.ClaimAccumulatorScopeId = DimAccumulatorScope.AccumulatorScopeId
    LEFT JOIN vw_DimBillType_bc DimBillType ON TMP.ClaimBillTypeId = DimBillType.BillTypeId
    LEFT JOIN vw_DimPayTo_bc DimPayTo ON TMP.ClaimPayToId = DimPayTo.PayToId
    LEFT JOIN vw_DimPayee_bc DimPayee ON TMP.ClaimPayeeId = DimPayee.PayeeId
    LEFT JOIN vw_DimReceivedMethod_bc DimReceivedMethod_Claim ON TMP.ClaimReceivedMethodId = DimReceivedMethod_Claim.ReceivedMethodId
    LEFT JOIN vw_DimStatus_bc DimStatus_Payment ON TMP.PaymentStatusId = DimStatus_Payment.StatusId
    LEFT JOIN vw_DimPaymentVoidReason_bc DimPaymentVoidReason ON TMP.PaymentVoidReasonId = DimPaymentVoidReason.PaymentVoidReasonId
    LEFT JOIN vw_DimThirdParty_bc DimThirdParty ON TMP.ClaimPaymentThirdPartyId = DimThirdParty.ThirdPartyId
    LEFT JOIN vw_DimAgent_bc DimAgent ON TMP.PolicyAgentHierarchyId = DimAgent.AgentHierarchyId
    LEFT JOIN vw_DimPolicyMember_bc DimMemberOwner ON TMP.PolicyMemberOwnerId = DimMemberOwner.PolicyMemberEligibilityId
    LEFT JOIN vw_DimTypeOfService_bc DimTOS ON TMP.ClaimTOSId = DimTOS.TypeOfServiceId
    LEFT JOIN vw_DimPlaceOfService_bc DimPOS ON TMP.ClaimPOSId = DimPOS.PlaceOfServiceId
    -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â JOINs para nuevas Policy*SKey (replican usp_DataPrepare_FactClaimCostData) ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
    LEFT JOIN vw_DimEntity_bc DimEntity ON TMP.PolicyEntityId = DimEntity.EntityId
    LEFT JOIN vw_DimProduct_bc DimProduct ON TMP.PolicyProductId = DimProduct.ProductId
    LEFT JOIN vw_DimGroup_bc DimGroup ON TMP.PolicyGroupId = DimGroup.GroupId
    LEFT JOIN vw_DimFamilyType_bc DimFamilyType ON TMP.PolicyFamilyTypeId = DimFamilyType.FamilyTypeId
    LEFT JOIN vw_DimPlan_bc DimPlan ON TMP.PolicyPlanId = DimPlan.PlanId
    LEFT JOIN vw_DimBusinessMode_bc DimBusinessMode ON TMP.PolicyBusinessModeId = DimBusinessMode.BusinessModeId
    LEFT JOIN vw_DimBusinessType_bc DimBusinessType ON TMP.PolicyBusinessTypeId = DimBusinessType.BusinessTypeId
    LEFT JOIN vw_DimCompany_bc DimCompany ON TMP.PolicyCompanyId = DimCompany.CompanyId
    LEFT JOIN vw_DimInsuranceBusiness_bc DimInsuranceBusiness_Policy ON TMP.PolicyInsuranceBusinessId = DimInsuranceBusiness_Policy.InsuranceBusinessId AND DimInsuranceBusiness_Policy.ToDate IS NULL
    LEFT JOIN vw_DimModeOfPayment_bc DimModeOfPayment ON TMP.PolicyModeOfPaymentId = DimModeOfPayment.ModeOfPaymentId
    LEFT JOIN vw_DimPaymentMethod_bc DimPaymentMethod ON TMP.PolicyPaymentMethodId = DimPaymentMethod.PaymentMethodId
    LEFT JOIN vw_DimReceivedMethod_bc DimReceivedMethod_Policy ON TMP.PolicyReceivedMethodId = DimReceivedMethod_Policy.ReceivedMethodId
    LEFT JOIN vw_DimRegion_bc DimRegion ON TMP.PolicyRegionId = DimRegion.RegionId
    -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â JOINs para Claim Diagnostic / Procedure ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
    LEFT JOIN vw_DimDiagnostic_bc DimDiagnostic ON TMP.ClaimPrimaryDiagnosticId = DimDiagnostic.DiagnosticId
    LEFT JOIN vw_DimProcedure_bc DimProcedure ON TMP.ClaimPrimaryProcedureId = DimProcedure.ProcedureId
    -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â JOINs para Claim Run dims ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
    LEFT JOIN vw_DimRunType_bc DimRunType ON TMP.ClaimRunTypeId = DimRunType.RunTypeId
    LEFT JOIN vw_DimStatus_bc DimStatus_Run ON TMP.ClaimRunStatusId = DimStatus_Run.StatusId
    LEFT JOIN vw_DimPaymentMethod_bc DimPaymentMethod_Run ON TMP.ClaimRunPaymentMethodId = DimPaymentMethod_Run.PaymentMethodId
    -- ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â JOINs para Policy*DateKey y ClaimProcessedDateSkey ÃƒÂ¢Ã¢â‚¬Â¢Ã‚ÂÃƒÂ¢Ã¢â‚¬Â¢Ã‚Â
    LEFT JOIN vw_DimDate_bc DimDate_PolicyIssue ON DimDate_PolicyIssue.Date = TMP.PolicyIssueDate
    LEFT JOIN vw_DimDate_bc DimDate_PolicyRenewal ON DimDate_PolicyRenewal.Date = TMP.PolicyRenewalDate
    LEFT JOIN vw_DimDate_bc DimDate_PolicyAnniversary ON DimDate_PolicyAnniversary.Date = TMP.PolicyAnniversaryDate
    LEFT JOIN vw_DimDate_bc DimDate_PolicyApplicationReceived ON DimDate_PolicyApplicationReceived.Date = TMP.PolicyApplicationReceivedDate
    LEFT JOIN vw_DimDate_bc DimDate_PolicyEffective ON DimDate_PolicyEffective.Date = TMP.PolicyEffectiveDate
    LEFT JOIN vw_DimDate_bc DimDate_PolicyCancel ON DimDate_PolicyCancel.Date = TMP.PolicyCancelDate
    LEFT JOIN vw_DimDate_bc DimDate_PolicyDeathBenefit ON DimDate_PolicyDeathBenefit.Date = TMP.PolicyDeathBenefitPeriodDate
    LEFT JOIN vw_DimDate_bc DimDate_ClaimProcessed ON DimDate_ClaimProcessed.Date = TMP.ClaimProcessedDate
""")

df_registros.createOrReplaceTempView("vwTmp_RegistrosFactClaimPayment")
print(f"[OK] vwTmp_RegistrosFactClaimPayment created with broadcast dimension joins")

In [ ]:
%run nb_Utils

In [ ]:
# ============================================================
# Registrar mapeo FactÃ¢â€ â€™Dimension en config_factdimensionmap
# Usado por InsertDataOrphan para insertar huÃƒÂ©rfanos.
#
# ValidaciÃƒÂ³n previa (evita errores en InsertDataOrphan):
#   1) La dimensiÃƒÂ³n exista en Gold.
#   2) FactKeyField exista como columna en vwTmp_RegistrosFactClaimPayment.
#   3) DimensionKeyField exista como columna en la dimensiÃƒÂ³n Gold.
# Cualquier mapeo que falle alguna validaciÃƒÂ³n se OMITE y se reporta.
# ============================================================

# Mapeos completos (FactName, DimName, FactKeyField, DimensionKeyField, TempTableName, IsActive)
all_mappings = [
    # Ã¢â€â‚¬Ã¢â€â‚¬ Dimensiones ya existentes Ã¢â€â‚¬Ã¢â€â‚¬
    ('bgla_FactClaimPayment','bgla_dimstatus','ClaimStatusSKey','StatusSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimclaimreason','ClaimReasonSKey','ClaimReasonSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpolicymember','ClaimMemberEligibilitySKey','PolicyMemberSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimprovider','ClaimBillingProviderSkey','ProviderSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimprovider','ClaimServiceProviderSKey','ProviderSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimschedule','ClaimScheduleSKey','ScheduleSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimserviceindicator','ClaimDetailServiceindicatorSkey','ServiceIndicatorSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimserviceindicator','ClaimLineDetailServiceindicatorSkey','ServiceIndicatorSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimcurrency','ClaimLocalCurrencySKey','CurrencySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimcurrency','ClaimBaseCurrencySkey','CurrencySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimcurrency','ClaimMemberPaymentCurrencySkey','CurrencySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimcurrency','ClaimProviderPaymentCurrencySkey','CurrencySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimgeography','ClaimGeographySKey','GeographySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpolicy','PolicySKey','PolicySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_diminsurancebusiness','ClaimRunInsuranceBusinessSKey','InsuranceBusinessSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_DimServiceNetwork','ServiceNetworkSKey','ServiceNetworkSKey','vwTmp_RegistrosFactClaimPayment',1),
    # Ã¢â€â‚¬Ã¢â€â‚¬ Alias-SKey ahora resueltos contra dimensiones reales Ã¢â€â‚¬Ã¢â€â‚¬
    ('bgla_FactClaimPayment','bgla_dimaccumulatorscope','ClaimAccumulatorScopeSKey','AccumulatorScopeSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimbilltype','ClaimBillTypeSKey','BillTypeSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpayto','ClaimPayToSkey','PayToSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpayee','ClaimPayeeSkey','PayeeSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimreceivedmethod','ClaimReceivedMethodSKey','ReceivedMethodSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimstatus','PaymentStatusSKey','StatusSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpaymentvoidreason','PaymentVoidReasonSkey','PaymentVoidReasonSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimthirdparty','PaymentThirdPartySKey','ThirdPartySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimagent','PolicyAgentSKey','AgentSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpolicymember','PolicyMemberOwnerEligibilitySKey','PolicyMemberSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimtypeofservice','ClaimTypeOfServiceSKey','TOSSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimplaceofservice','ClaimPlaceOfServiceSKey','POSSKey','vwTmp_RegistrosFactClaimPayment',1),
    # Ã¢â€â‚¬Ã¢â€â‚¬ Nuevas Policy*SKey (replican usp_DataPrepare_FactClaimCostData) Ã¢â€â‚¬Ã¢â€â‚¬
    ('bgla_FactClaimPayment','bgla_dimentity','PolicyEntitySKey','EntitySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimproduct','PolicyProductSKey','ProductSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimgroup','PolicyGroupSKey','GroupSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimfamilytype','PolicyFamilyTypeSKey','FamilyTypeSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimplan','PolicyPlanSKey','PlanSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimbusinessmode','PolicyBusinessModeSKey','BusinessModeSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimbusinesstype','PolicyBusinessTypeSKey','BusinessTypeSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimcompany','PolicyCompanySKey','CompanySKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_diminsurancebusiness','PolicyInsuranceBusinessSKey','InsuranceBusinessSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimmodeofpayment','PolicyModeOfPaymentSKey','ModeOfPaymentSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpaymentmethod','PolicyPaymentMethodSKey','PaymentMethodSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimreceivedmethod','PolicyReceivedMethodSKey','ReceivedMethodSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimregion','PolicyRegionSKey','RegionSKey','vwTmp_RegistrosFactClaimPayment',1),
    # Ã¢â€â‚¬Ã¢â€â‚¬ Nuevas Claim/Run SKey Ã¢â€â‚¬Ã¢â€â‚¬
    ('bgla_FactClaimPayment','bgla_dimdiagnostic','ClaimPrimaryDiagnosticSKey','DiagnosticSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimprocedure','ClaimPrimaryProcedureSKey','ProcedureSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimruntype','ClaimRunTypeSKey','RunTypeSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimstatus','ClaimRunStatusSKey','StatusSKey','vwTmp_RegistrosFactClaimPayment',1),
    ('bgla_FactClaimPayment','bgla_dimpaymentmethod','ClaimRunPaymentMethodSKey','PaymentMethodSKey','vwTmp_RegistrosFactClaimPayment',1),
]

# Ã¢â€â‚¬Ã¢â€â‚¬ ValidaciÃƒÂ³n 1: dimensiones existentes en Gold Ã¢â€â‚¬Ã¢â€â‚¬
existing_tables = {t.name.lower() for t in spark.catalog.listTables("Gold")}

# Ã¢â€â‚¬Ã¢â€â‚¬ ValidaciÃƒÂ³n 2: columnas de la vista temporal Ã¢â€â‚¬Ã¢â€â‚¬
try:
    tmp_cols = {f.name.lower() for f in spark.table("vwTmp_RegistrosFactClaimPayment").schema.fields}
except Exception as e:
    raise RuntimeError(f"vwTmp_RegistrosFactClaimPayment no encontrada. Ejecuta primero la celda que la crea. Detalle: {e}")

# Ã¢â€â‚¬Ã¢â€â‚¬ ValidaciÃƒÂ³n 3: cache de columnas por dimensiÃƒÂ³n Ã¢â€â‚¬Ã¢â€â‚¬
dim_cols_cache = {}
def _get_dim_cols(dim_name_lower):
    if dim_name_lower not in dim_cols_cache:
        dim_cols_cache[dim_name_lower] = {f.name.lower() for f in spark.table(f"Gold.{dim_name_lower}").schema.fields}
    return dim_cols_cache[dim_name_lower]

valid_mappings = []
skipped = []
for m in all_mappings:
    fact_name, dim_name, fact_key, dim_key, tmp_name, is_active = m
    dim_lower = dim_name.lower()
    if dim_lower not in existing_tables:
        skipped.append((dim_name, fact_key, "dimension no existe en Gold"))
        continue
    if fact_key.lower() not in tmp_cols:
        skipped.append((dim_name, fact_key, f"columna '{fact_key}' no existe en {tmp_name}"))
        continue
    dim_cols = _get_dim_cols(dim_lower)
    if dim_key.lower() not in dim_cols:
        skipped.append((dim_name, fact_key, f"columna '{dim_key}' no existe en Gold.{dim_name}"))
        continue
    valid_mappings.append(m)

if skipped:
    print(f"[WARN] {len(skipped)} mapeo(s) omitido(s):")
    for dim_name, fact_key, reason in skipped:
        print(f"  - {dim_name} / {fact_key} Ã¢â€ â€™ {reason}")

spark.sql("DELETE FROM Silver.config_factdimensionmap WHERE FactName = 'bgla_FactClaimPayment'")

if valid_mappings:
    values_sql = ",\n        ".join(
        f"('{m[0]}','{m[1]}','{m[2]}','{m[3]}','{m[4]}',{m[5]})" for m in valid_mappings
    )
    spark.sql(f"""
        INSERT INTO Silver.config_factdimensionmap
            (FactName, DimName, FactKeyField, DimensionKeyField, TempTableName, IsActive)
        VALUES
        {values_sql}
    """)

row_count = spark.sql("SELECT COUNT(*) AS cnt FROM Silver.config_factdimensionmap WHERE FactName = 'bgla_FactClaimPayment'").first()["cnt"]
print(f"[OK] Silver.config_factdimensionmap loaded Ã¢â‚¬â€ {row_count} dimension mappings for bgla_FactClaimPayment")


In [ ]:
InsertDataOrphan("bgla_FactClaimPayment")

In [ ]:
# ============================================================
# MERGE DELETE + INSERT
# El MERGE DELETE anterior comparaba business keys + muchas dimension SKeys
# con CAST/COALESCE, lo que hacia el match muy costoso. Se conserva
# comentado y se reemplaza por un delete por scope de negocio corto.
# ============================================================
if False:
    spark.sql("""
    MERGE INTO Gold.bgla_FactClaimPayment AS T
    USING vwTmp_RegistrosFactClaimPayment AS TMP
    ON  T.ClaimRunId = TMP.ClaimRunId
    AND T.ClaimDetailRunId = TMP.ClaimDetailRunId
    AND T.ClaimDetailId = TMP.ClaimDetailId
    AND T.ClaimLineDetailId = TMP.ClaimLineDetailId
    AND T.PaidSource = TMP.PaidSource
    -- Dimension SKeys
    AND COALESCE(CAST(T.ClaimStatusSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimStatusSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimReasonSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimReasonSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimStatusReasonDatekey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimStatusReasonDatekey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimMemberEligibilitySKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimMemberEligibilitySKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimBillingProviderSkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimBillingProviderSkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimServiceProviderSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimServiceProviderSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimScheduleSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimScheduleSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimDetailServiceindicatorSkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimDetailServiceindicatorSkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimLineDetailServiceindicatorSkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimLineDetailServiceindicatorSkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimServiceFromDateKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimServiceFromDateKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimLocalCurrencySKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimLocalCurrencySKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimBaseCurrencySkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimBaseCurrencySkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimReceivedDateKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimReceivedDateKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimGeographySKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimGeographySKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicySKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicySKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimRunInsuranceBusinessSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimRunInsuranceBusinessSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimRunCreatedOnDateSkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimRunCreatedOnDateSkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimRunPrintedOnDateSkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimRunPrintedOnDateSkey AS STRING), '-101')
    AND COALESCE(CAST(T.PaymentPrintedDateKey AS STRING), '-101') = COALESCE(CAST(TMP.PaymentPrintedDateKey AS STRING), '-101')
    AND COALESCE(CAST(T.PaymentPostedDateKey AS STRING), '-101') = COALESCE(CAST(TMP.PaymentPostedDateKey AS STRING), '-101')
    AND COALESCE(CAST(T.PaymentVoidDateSKey AS STRING), '-101') = COALESCE(CAST(TMP.PaymentVoidDateSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimMemberPaymentCurrencySkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimMemberPaymentCurrencySkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimProviderPaymentCurrencySkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimProviderPaymentCurrencySkey AS STRING), '-101')
    -- ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Alias-SKey ahora resueltos contra dimensiones reales ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
    AND COALESCE(CAST(T.ClaimAccumulatorScopeSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimAccumulatorScopeSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimBillTypeSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimBillTypeSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimPayToSkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimPayToSkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimPayeeSkey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimPayeeSkey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimReceivedMethodSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimReceivedMethodSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PaymentStatusSKey AS STRING), '-101') = COALESCE(CAST(TMP.PaymentStatusSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PaymentVoidReasonSkey AS STRING), '-101') = COALESCE(CAST(TMP.PaymentVoidReasonSkey AS STRING), '-101')
    AND COALESCE(CAST(T.PaymentThirdPartySKey AS STRING), '-101') = COALESCE(CAST(TMP.PaymentThirdPartySKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyAgentSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyAgentSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyMemberOwnerEligibilitySKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyMemberOwnerEligibilitySKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimTypeOfServiceSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimTypeOfServiceSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimPlaceOfServiceSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimPlaceOfServiceSKey AS STRING), '-101')
    -- ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Nuevas Policy*SKey (replican usp_DataPrepare_FactClaimCostData) ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
    AND COALESCE(CAST(T.PolicyEntitySKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyEntitySKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyProductSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyProductSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyGroupSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyGroupSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyFamilyTypeSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyFamilyTypeSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyPlanSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyPlanSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyBusinessModeSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyBusinessModeSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyBusinessTypeSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyBusinessTypeSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyCompanySKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyCompanySKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyInsuranceBusinessSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyInsuranceBusinessSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyModeOfPaymentSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyModeOfPaymentSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyPaymentMethodSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyPaymentMethodSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyReceivedMethodSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyReceivedMethodSKey AS STRING), '-101')
    AND COALESCE(CAST(T.PolicyRegionSKey AS STRING), '-101') = COALESCE(CAST(TMP.PolicyRegionSKey AS STRING), '-101')
    -- ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Nuevas Claim/Run SKey ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
    AND COALESCE(CAST(T.ClaimPrimaryDiagnosticSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimPrimaryDiagnosticSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimPrimaryProcedureSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimPrimaryProcedureSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimRunTypeSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimRunTypeSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimRunStatusSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimRunStatusSKey AS STRING), '-101')
    AND COALESCE(CAST(T.ClaimRunPaymentMethodSKey AS STRING), '-101') = COALESCE(CAST(TMP.ClaimRunPaymentMethodSKey AS STRING), '-101')
    -- ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ Otros SKey ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬ÃƒÂ¢Ã¢â‚¬ÂÃ¢â€šÂ¬
    AND COALESCE(CAST(T.ServiceNetworkSKey AS STRING), '-101') = COALESCE(CAST(TMP.ServiceNetworkSKey AS STRING), '-101')
    WHEN MATCHED THEN DELETE
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW vwTmp_FactClaimPayment_DeleteScope AS
    SELECT DISTINCT
        ClaimRunId,
        ClaimDetailRunId,
        ClaimDetailId,
        ClaimLineDetailId,
        PaidSource,
        JournalEntryId,
        PaymentId
    FROM vwTmp_RegistrosFactClaimPayment
    WHERE ClaimRunId IS NOT NULL
      AND ClaimDetailRunId IS NOT NULL
      AND ClaimDetailId IS NOT NULL
      AND ClaimLineDetailId IS NOT NULL
      AND PaidSource IS NOT NULL
""")

spark.sql("""
    MERGE INTO Gold.bgla_FactClaimPayment AS T
    USING vwTmp_FactClaimPayment_DeleteScope AS S
    ON  T.ClaimRunId = S.ClaimRunId
    AND T.ClaimDetailRunId = S.ClaimDetailRunId
    AND T.ClaimDetailId = S.ClaimDetailId
    AND T.ClaimLineDetailId = S.ClaimLineDetailId
    AND T.PaidSource = S.PaidSource
    AND T.JournalEntryId <=> S.JournalEntryId
    AND T.PaymentId <=> S.PaymentId
    WHEN MATCHED THEN DELETE
""")
print('[OK] Existing Gold rows deleted with optimized business-key scope')

spark.sql("""
    INSERT INTO Gold.bgla_FactClaimPayment (
        FactClaimPaymentSKey, ClaimHeaderId, ClaimNumber, ClaimAccountNumber, ClaimReferenceNumber,
        ClaimDetailId, ClaimVersion, ClaimLineDetailId, ClaimLineSequenceNumber, ClamReferralHeaderId,
        ClaimUUID, ClaimStatusSKey, ClaimReasonSKey, ClaimStatusReasonDateKey, ClaimMemberEligibilitySKey,
        ClaimBillingProviderSkey, ClaimServiceProviderSKey, ClaimScheduleSKey, ClaimNetworkSkey,
        ClaimDetailServiceIndicatorSkey, ClaimLineDetailServiceIndicatorSkey, ClaimServiceFromDateKey,
        ClaimServiceFromDate, ClaimServiceToDate, ClaimBaseCurrencySKey, ClaimLocalCurrencySKey,
        ClaimReceivedDateKey, ClaimGeographySKey, ClaimAccumulatorScopeSKey, ClaimBillTypeSKey,
        ClaimPayToSkey, ClaimPayeeSkey, PolicySKey, PolicyModeOfPaymentSKey, PolicyPaymentMethodSKey,
        PolicyReceivedMethodSKey, PolicyGroupSKey, PolicyFamilyTypeSKey, PolicyProductSKey,
        PolicyEntitySKey, PolicyPlanSKey, PolicyBusinessModeSKey, PolicyBusinessTypeSKey,
        PolicyInsuranceBusinessSKey, PolicyCompanySKey, PolicyIssueDateKey, PolicyRenewalDateKey,
        PolicyAnniversaryDateKey, PolicyApplicationReceivedDateKey, PolicyEffectiveDateKey,
        PolicyCancelDateKey, PolicyDeathBenefitperiodDateKey, PolicyAgentSKey, PolicyRegionSkey,
        PolicyMemberOwnerEligibilitySKey, PolicyGeographySKey, ClaimRunId, ClaimDetailRunId,
        ClaimRunTypeSkey, ClaimRunCreatedOnDateSkey, ClaimRunPrintedOnDateSkey, ClaimRunStatusSKey,
        ClaimRunPaymentMethodSkey, ClaimPrimaryDiagnosticSKey, ClaimPrimaryProcedureSKey,
        ClaimLineDetail_DiagnosticType, ClaimLineDetail_POSId, ClaimLineDetail_TOSId, ServiceNetwork,
        Paid_To_Calculated, InOnNetwork, PaymentId, PaymentNumber, PaymentPostedDateKey,
        PaymentPrintedDateKey, PaymentStatusSKey, PaymentVoidReasonSkey, PaymentVoidDateSKey,
        JournalEntryId, JournalEntryNotes, ClaimMemberPaymentCurrencySkey, ClaimProviderPaymentCurrencySkey,
        ClaimMemberPaymentXchangeRate, ClaimProviderPaymentXchangeRate, PaymentThirdPartySKey,
        ClaimProcessedDateSkey, LocalBilledAmount, ClaimAllowedAmountBaseCurrency,
        ClaimBilledAmountBaseCurrency, ClaimCoInsuranceAmountBaseCurrency, ClaimAllowedCmpBaseCurrency,
        ClaimCopayAmountBaseCurrency, ClaimDeductibleAmountBaseCurrency, ClaimNetCoveredAmountBaseCurrency,
        ClaimMemberPaidAmountBaseCurrency, ClaimProviderPaidAmountBaseCurrency,
        ClaimTotalIneligibleAmountBaseCurrency, ClaimTotalOtherIneligibleAmountBaseCurrency,
        ClaimTotalPaidAmountBaseCurrency, ClaimMemberPaidAmountPaymentCurrency,
        ClaimProviderPaidAmountPaymentCurrency, ClaimAllowedAmountUSDCurrency, ClaimBilledAmountUSDCurrency,
        ClaimCoInsuranceAmountUSDCurrency, ClaimAllowedCmpUSDCurrency, ClaimCopayAmountUSDCurrency,
        ClaimDeductibleAmountUSDCurrency, ClaimMemberPaidAmountUSDCurrency, ClaimNetCoveredAmountUSDCurrency,
        ClaimProviderPaidAmountUSDCurrency, ClaimTotalIneligibleAmountUSDCurrency,
        ClaimTotalOtherIneligibleAmountUSDCurrency, ClaimTotalPaidAmountUSDCurrency,
        ClaimProviderPaymentAdjustedAmountBaseCurrenty, ClaimProviderPaymentAdjustedAmountPaymentCurrency,
        ClaimProviderPaymentAdjustedAmountUSDCurrency, ClaimMemberPaymentAdjustedAmountBaseCurrenty,
        ClaimMemberPaymentAdjustedAmountPaymentCurrency, ClaimMemberPaymentAdjustedAmountUSDCurrency,
        ClaimProviderPPABalanceBaseCurrenty, ClaimProviderPPABalancePaymentCurrency,
        ClaimMemberPPABalanceBaseCurrenty, ClaimMemberPPABalancePaymentCurrency,
        ClaimProviderVoidPPABalanceBaseCurrenty, ClaimProviderVoidPPABalancePaymentCurrency,
        ClaimMemberVoidPPABalanceBaseCurrenty, ClaimMemberVoidPPABalancePaymentCurrency,
        ClaimProcessRowFlag, Source, EffectiveDate, LoadDate, ActualRowFlag,
        ClaimRunInsuranceBusinessSKey, PaymentGatewayTransactionId, PaymentAmount, PaymentForeignAmount,
        PaymentBankTransmissionId, ClaimLocalXChangeRate, ClaimLocalXChangeDate, ControlVersion, PaidSource,
        VATPaymentCurrency, VATBaseCurrenty, VATUSDCurrenty, WithholdingPaymentCurrency,
        WithholdingBaseCurrenty, WithholdingUSDCurrenty, MemberId, ClaimTotalDiscountAmountUSDCurrency,
        ClaimTotalDiscountAmount, ClaimGap, ClaimReceivedMethodSKey, PolicyFirstYear, PolicyFirstMonth,
        FistClaim, IsDigital, ServiceNetworkSKey, IsCOR, LocalTeam, ClaimsOutliers, IsFastTrack,
        HTHFeePaid, HTHFeePercent, IsDeleted, ProviderNetworkClassName, SavingAmount,
        IncomebyinsuredIA, SpendbyinsuredGA, BrackedClaimKey, BrackedMemberKey,
        SubmissionSenderEmailAddress, SubmissionSenderFullName, SubmissionInboundChannel,
        SubmissionTrackingNumber, SubmissionSenderType, ClaimValidation, BatchLotName, BatchLotFromDate,
        TransactionLotReceivedDate, TransactionLotNumber, TransactionInvoiceNumber, LotNumber, IsAutomatic,
        IsICU, IsEmergency, IsHospital_Emergency, IsChildbirths, IsCesareanSection,
        ClaimTypeOfServiceSKey, ClaimPlaceOfServiceSKey, PolicyId
    )
    SELECT
        FactClaimPaymentSKey, ClaimHeaderId, ClaimNumber, ClaimAccountNumber, ClaimReferenceNumber,
        ClaimDetailId, ClaimVersion, ClaimLineDetailId, ClaimLineSequenceNumber, ClamReferralHeaderId,
        ClaimUUID, ClaimStatusSKey, ClaimReasonSKey, ClaimStatusReasonDatekey, ClaimMemberEligibilitySKey,
        ClaimBillingProviderSkey, ClaimServiceProviderSKey, ClaimScheduleSKey, ClaimNetworkSkey,
        ClaimDetailServiceindicatorSkey, ClaimLineDetailServiceindicatorSkey, ClaimServiceFromDateKey,
        ClaimServiceFromDate, ClaimServiceToDate, ClaimBaseCurrencySkey, ClaimLocalCurrencySKey,
        ClaimReceivedDateKey, ClaimGeographySKey, ClaimAccumulatorScopeSKey, ClaimBillTypeSKey,
        ClaimPayToSkey, ClaimPayeeSkey, PolicySKey, PolicyModeOfPaymentSKey, PolicyPaymentMethodSKey,
        PolicyReceivedMethodSKey, PolicyGroupSKey, PolicyFamilyTypeSKey, PolicyProductSKey,
        PolicyEntitySKey, PolicyPlanSKey, PolicyBusinessModeSKey, PolicyBusinessTypeSKey,
        PolicyInsuranceBusinessSKey, PolicyCompanySKey,
        PolicyIssueDateKey, PolicyRenewalDateKey, PolicyAnniversaryDateKey,
        PolicyApplicationReceivedDateKey, PolicyEffectiveDateKey, PolicyCancelDateKey,
        PolicyDeathBenefitperiodDateKey,
        PolicyAgentSKey, PolicyRegionSKey, PolicyMemberOwnerEligibilitySKey, PolicyGeographySKey,
        ClaimRunId, ClaimDetailRunId, ClaimRunTypeSKey, ClaimRunCreatedOnDateSkey,
        ClaimRunPrintedOnDateSkey, ClaimRunStatusSKey, ClaimRunPaymentMethodSKey,
        ClaimPrimaryDiagnosticSKey, ClaimPrimaryProcedureSKey,
        ClaimDiagnosticType AS ClaimLineDetail_DiagnosticType,
        ClaimPOSId AS ClaimLineDetail_POSId,
        ClaimTOSId AS ClaimLineDetail_TOSId,
        ServiceNetwork, Paid_To_Calculated,
        INN_OON AS InOnNetwork,
        PaymentId, PaymentNumber, PaymentPostedDateKey, PaymentPrintedDateKey, PaymentStatusSKey,
        PaymentVoidReasonSkey, PaymentVoidDateSKey, JournalEntryId,
        JournalNotes AS JournalEntryNotes,
        ClaimMemberPaymentCurrencySkey, ClaimProviderPaymentCurrencySkey,
        ClaimPaymentToMemberXchangeRate AS ClaimMemberPaymentXchangeRate,
        ClaimPaymentToProviderXchangeRate AS ClaimProviderPaymentXchangeRate,
        PaymentThirdPartySKey, ClaimProcessedDateSkey,
        LocalBilledAmount,
        BaseAllowedAmount AS ClaimAllowedAmountBaseCurrency,
        BaseBilledAmount AS ClaimBilledAmountBaseCurrency,
        BaseCoInsuranceAmount AS ClaimCoInsuranceAmountBaseCurrency,
        BaseAllowedCmp AS ClaimAllowedCmpBaseCurrency,
        BaseCopayAmount AS ClaimCopayAmountBaseCurrency,
        BaseDeductibleAmount AS ClaimDeductibleAmountBaseCurrency,
        BaseNetCoveredAmount AS ClaimNetCoveredAmountBaseCurrency,
        BaseMemberPaidAmount AS ClaimMemberPaidAmountBaseCurrency,
        BaseProviderPaidAmount AS ClaimProviderPaidAmountBaseCurrency,
        BaseTotalIneligibleAmount AS ClaimTotalIneligibleAmountBaseCurrency,
        BaseOtherIneligibleAmount AS ClaimTotalOtherIneligibleAmountBaseCurrency,
        BaseTotalPaidAmount AS ClaimTotalPaidAmountBaseCurrency,
        PayMemberPaidAmount AS ClaimMemberPaidAmountPaymentCurrency,
        PayProviderPaidAmount AS ClaimProviderPaidAmountPaymentCurrency,
        USD_AllowedAmount AS ClaimAllowedAmountUSDCurrency,
        USD_BilledAmount AS ClaimBilledAmountUSDCurrency,
        USD_CoInsuranceAmount AS ClaimCoInsuranceAmountUSDCurrency,
        USD_AllowedCmp AS ClaimAllowedCmpUSDCurrency,
        USD_CopayAmount AS ClaimCopayAmountUSDCurrency,
        USD_DeductibleAmount AS ClaimDeductibleAmountUSDCurrency,
        USD_MemberPaidAmount AS ClaimMemberPaidAmountUSDCurrency,
        USD_NetCoveredAmount AS ClaimNetCoveredAmountUSDCurrency,
        USD_ProviderPaidAmount AS ClaimProviderPaidAmountUSDCurrency,
        USD_TotalIneligibleAmount AS ClaimTotalIneligibleAmountUSDCurrency,
        USD_OtherIneligibleAmount AS ClaimTotalOtherIneligibleAmountUSDCurrency,
        USD_TotalPaidAmount AS ClaimTotalPaidAmountUSDCurrency,
        BaseClaimProviderPaymentAdjustedAmount AS ClaimProviderPaymentAdjustedAmountBaseCurrenty,
        PayClaimProviderPaymentAdjustedAmount AS ClaimProviderPaymentAdjustedAmountPaymentCurrency,
        USD_ClaimProviderPaymentAdjustedAmount AS ClaimProviderPaymentAdjustedAmountUSDCurrency,
        BaseClaimMemberPaymentAdjustedAmount AS ClaimMemberPaymentAdjustedAmountBaseCurrenty,
        PayClaimMemberPaymentAdjustedAmount AS ClaimMemberPaymentAdjustedAmountPaymentCurrency,
        USD_ClaimMemberPaymentAdjustedAmount AS ClaimMemberPaymentAdjustedAmountUSDCurrency,
        ClaimProviderPPABalanceBaseCurrenty, ClaimProviderPPABalancePaymentCurrency,
        ClaimMemberPPABalanceBaseCurrenty, ClaimMemberPPABalancePaymentCurrency,
        ClaimProviderVoidPPABalanceBaseCurrenty, ClaimProviderVoidPPABalancePaymentCurrency,
        ClaimMemberVoidPPABalanceBaseCurrenty, ClaimMemberVoidPPABalancePaymentCurrency,
        ClaimProcessRowFlag, Source, EffectiveDate, LoadDate, ActualRowFlag,
        ClaimRunInsuranceBusinessSKey, PaymentGatewayTransactionId, PaymentAmount, PaymentForeignAmount,
        PaymentBankTransmissionId, ClaimLocalXChangeRate, ClaimLocalXChangeDate, ControlVersion, PaidSource,
        VATPaymentCurrency,
        BaseVATAmount AS VATBaseCurrenty,
        USD_VATAmount AS VATUSDCurrenty,
        WithholdingPaymentCurrency,
        BaseWithHoldingAmount AS WithholdingBaseCurrenty,
        USD_WithHoldingAmount AS WithholdingUSDCurrenty,
        MemberId,
        USD_DiscountAmount AS ClaimTotalDiscountAmountUSDCurrency,
        BaseDiscountAmount AS ClaimTotalDiscountAmount,
        ClaimGap, ClaimReceivedMethodSKey, PolicyFirstYear, PolicyFirstMonth,
        FirstClaim AS FistClaim,
        IsDigital, ServiceNetworkSKey, IsCOR, LocalTeam, ClaimsOutliers, IsFastTrack,
        HTHFeePaid, HTHFeePercent, IsDeleted, ProviderNetworkClassName, SavingAmount,
        IncomeByInsured AS IncomebyinsuredIA,
        SpendByInsured AS SpendbyinsuredGA,
        BrackedClaimKey, BrackedMemberKey,
        SubmissionSenderEmailAddress, SubmissionSenderFullName, SubmissionInboundChannel,
        SubmissionTrackingNumber, SubmissionSenderType, ClaimValidation, BatchLotName, BatchLotFromDate,
        TransactionLotReceivedDate, TransactionLotNumber, TransactionInvoiceNumber, LotNumber, IsAutomatic,
        IsICU, IsEmergency, IsHospital_Emergency, IsChildbirths, IsCesareanSection,
        ClaimTypeOfServiceSKey, ClaimPlaceOfServiceSKey, PolicyId
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                -- FIX dedup: incluir JournalEntryId + montos ajustados para preservar
                -- filas legitimamente distintas (ej. PAID raw 6936.52 vs PAID PPA-adj 6928.52
                -- bajo el mismo ClaimRunId/DetailRunId/JournalEntryId).
                PARTITION BY ClaimRunId, ClaimDetailRunId, ClaimDetailId, ClaimLineDetailId,
                             PaidSource, JournalEntryId,
                             BaseClaimProviderPaymentAdjustedAmount,
                             BaseClaimMemberPaymentAdjustedAmount
                ORDER BY LoadDate DESC
            ) AS rn
        FROM vwTmp_RegistrosFactClaimPayment
    )
    WHERE rn = 1 """)

In [ ]:
# ============================================================
# Cell 26: Dedup ActualRowFlag + EffectiveDate (consolidado)
# ============================================================
# Paso 1: Marcar duplicados como ActualRowFlag = 0
spark.sql("""
    MERGE INTO Gold.bgla_FactClaimPayment F
    USING (
        SELECT t.FactClaimPaymentSKey
        FROM (
            SELECT F.FactClaimPaymentSKey,
                DENSE_RANK() OVER (
                    PARTITION BY F.ClaimRunId, F.ClaimDetailId, F.ClaimLineDetailId, F.PaidSource
                    ORDER BY F.LoadDate DESC
                ) AS Fila,
                F.ActualRowFlag
            FROM Gold.bgla_FactClaimPayment F
        ) t
        WHERE t.Fila > 1 AND t.ActualRowFlag <> 0
    ) P
    ON F.FactClaimPaymentSKey = P.FactClaimPaymentSKey
    WHEN MATCHED THEN UPDATE SET F.ActualRowFlag = 0
""")

# Paso 2: Actualizar EffectiveDate para registros activos
spark.sql("""
    UPDATE Gold.bgla_FactClaimPayment
    SET EffectiveDate = current_date()
    WHERE ActualRowFlag = 1
""")

print("[OK] Dedup + EffectiveDate completed")

In [ ]:
# ============================================================
# Cell: ControlVersion UPDATE ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â usp_DataPrepare_BGLADW_ActualRowFlagClaims
# DENSE_RANK() OVER(PARTITION BY ClaimHeaderId ORDER BY ClaimDetailId DESC)
# Actualiza ControlVersion en Gold.bgla_FactClaimPayment
# ============================================================
spark.sql("""
    MERGE INTO Gold.bgla_FactClaimPayment AS t
    USING (
        SELECT FactClaimPaymentSKey,
               DENSE_RANK() OVER(PARTITION BY ClaimHeaderId ORDER BY ClaimDetailId DESC) AS ControlVersionNew
        FROM Gold.bgla_FactClaimPayment
    ) AS src
    ON t.FactClaimPaymentSKey = src.FactClaimPaymentSKey
    WHEN MATCHED AND t.ControlVersion <> src.ControlVersionNew
    THEN UPDATE SET t.ControlVersion = src.ControlVersionNew
""")
print("[OK] ControlVersion updated via DENSE_RANK (usp_DataPrepare_BGLADW_ActualRowFlagClaims)")

In [ ]:
# ============================================================
# Cell 27: Cleanup staging + delete non-actual + uncache
# ============================================================
# Paso 1: Drop staging table
spark.sql("DROP TABLE IF EXISTS Silver.TmpFactClaimPayment")

# Paso 2: Delete non-actual rows
spark.sql("DELETE FROM Gold.bgla_FactClaimPayment WHERE ActualRowFlag = 0")

# Paso 3: Liberar cache de vistas intermedias
for view_name in ["vw_ClaimRun", "vw_SubLineDetailValue", "vw_BaseClaimsLines"]:
    try:
        spark.sql(f"UNCACHE TABLE IF EXISTS {view_name}")
    except:
        pass

# Paso 4: OPTIMIZE Gold table (compactar archivos pequeÃƒÆ’Ã‚Â±os)
spark.sql("OPTIMIZE Gold.bgla_FactClaimPayment")

print("[OK] Pipeline nb_FactClaimPayment completed successfully")

In [ ]:
# ============================================================
# POST-LOAD MAINTENANCE Ã¢â‚¬â€ OPTIMIZE + ZORDER + VACUUM
# Mejora drÃƒÂ¡sticamente las lecturas downstream (data skipping + file pruning)
# ============================================================
try:
    spark.sql("""
        OPTIMIZE Silver.TmpFactClaimPayment
        ZORDER BY (ClaimHeaderId, ClaimDetailId, ClaimRunId)
    """)
    print("[OK] Silver.TmpFactClaimPayment OPTIMIZE + ZORDER applied")
except Exception as e:
    print(f"[WARN] OPTIMIZE Silver.TmpFactClaimPayment skipped: {e}")

try:
    spark.sql("""
        OPTIMIZE Gold.bgla_FactClaimPayment
        ZORDER BY (ClaimHeaderId, ClaimDetailId, ClaimRunId)
    """)
    print("[OK] Gold.bgla_FactClaimPayment OPTIMIZE + ZORDER applied")
except Exception as e:
    print(f"[WARN] OPTIMIZE Gold.bgla_FactClaimPayment skipped: {e}")

# Liberar caches Spark
try:
    spark.catalog.clearCache()
    print("[OK] Spark cache cleared")
except Exception as e:
    print(f"[WARN] Cache clear skipped: {e}")


In [ ]:
# ============================================================
# Mantenimiento Delta -- correr periodicamente (idealmente al final del notebook
# o como job semanal). Reduce dramaticamente el tiempo de MERGE en runs futuros.
#
# - OPTIMIZE: compacta files pequenos (Spark genera muchos files chicos tras
#   INSERTs frecuentes, lo que penaliza MERGE/SELECT).
# - ZORDER BY: re-ordena los files fisicamente para que el data-skipping de
#   Delta (min/max stats por file) sea efectivo en la clave de negocio del MERGE.
#   Con ZORDER, el MERGE solo lee/reescribe files que efectivamente contienen
#   las claves del batch nuevo, en vez de toda la tabla.
#
# Costo: 1-5 min la primera vez, decreciente despues. ROI: ~10x en MERGE futuro.
# ============================================================
try:
    spark.sql("""
        OPTIMIZE Gold.bgla_FactClaimPayment
        ZORDER BY (ClaimRunId, ClaimDetailRunId, ClaimDetailId, ClaimLineDetailId,
                   PaidSource, JournalEntryId)
    """)
    print("[OK] OPTIMIZE + ZORDER aplicado sobre Gold.bgla_FactClaimPayment")
except Exception as e:
    print(f"[WARN] OPTIMIZE fallo: {e}")

try:
    spark.sql("ANALYZE TABLE Gold.bgla_FactClaimPayment COMPUTE STATISTICS FOR ALL COLUMNS")
    print("[OK] ANALYZE STATISTICS aplicado")
except Exception as e:
    print(f"[WARN] ANALYZE fallo: {e}")
